# Rambla SA: perfiles de riesgo y carteras modelo

Proyecto Final de Programación Analítica II

Rambla SA le asigna el perfil de riesgo a sus 640 clientes preguntándoles si se consideran conservadores, moderados o agresivos, y con esa respuesta les arma la cartera. El gerente de asesoramiento sospecha que la gente se describe como quiere que la vean y no como se comporta. Quiere saber cuántos clientes tienen una cartera que no se parece a lo que dijeron ser, y quiénes son.

## Supuestos que afectan los resultados

- La fecha de corte de toda la información es el 30/06/2026, la que fija la consigna del caso. Las tasas, el tipo de cambio y la UI son de ese día, así que una encuesta u operación posterior sólo puede ser un error de carga.
- Todo se pasa a dólares con el tipo de cambio y la UI del 30/06/2026. Las posiciones en pesos quedan a costo y las de UI a valor de hoy.
- La cartera es lo que el cliente pagó: no hay ventas, así que supongo que nadie desinvirtió.
- Lo que rinde cada instrumento es su retorno histórico promedio del catálogo, no lo que ganó cada cliente. Con mucha volatilidad, lo que una cartera crece año a año es menor que ese promedio: aproximadamente el retorno menos la mitad de la volatilidad al cuadrado.
- El retorno real usa la inflación de los últimos 12 meses (5,4 % en Uruguay, 2,7 % en EE. UU.)
- La reserva de emergencia es de 3 meses del ingreso declarado, porque no hay dato de gastos. Es el mínimo de la regla habitual de 3 a 6 meses de gastos (según la guía de educación financiera de BBVA): deja más efectivo dentro de la cartera, así que los clientes parecen más conservadores. Eso puede achicar la cantidad de sobreexpuestos y agrandar la de subexpuestos.
- Del retorno que espera el cliente tomo el piso de su tramo, así que la brecha calculada es la mínima.
- Los cortes que separan conservador, moderado y agresivo en la escala de 1 a 5 son una convención, calibrada contra el SRRI, el indicador de riesgo de 1 a 7 que la regulación europea exigía en la ficha de cada fondo. Otros cortes darían otro reparto entre perfiles.
- La escala de riesgo de 1 a 5 es ordinal y la promedio como si fuera numérica.
- El patrimonio no declarado se estima con lo que el cliente operó en la firma, multiplicado por la relación entre declarado y operado de los clientes que sí contestan y operaron un monto parecido.
- La volatilidad de cada instrumento está en su propia moneda y no hay datos de cómo se mueven juntos. Su promedio ponderado es un techo en moneda propia, pero no incluye el riesgo cambiario: para quien mide en dólares, un plazo fijo en pesos es más riesgoso que su 0,5 %.
- Los límites de las carteras modelo son criterios míos: ninguna posición pasa de 30 % y cada una supera el mínimo del instrumento; sin fondo de emergencia, al menos 60 % líquido y volatilidad de hasta 10 %. A quien consintió el perfil conservador no se le da una cartera de más de 5 % de volatilidad sin re-perfilarlo.
- El cruce entre lo declarado y lo real sólo incluye clientes que tienen encuesta y cartera.

## Setup

In [1]:
from pathlib import Path
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from threadpoolctl import threadpool_limits

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


def formato_numero(valor: float) -> str:
    """Dos decimales para montos, cuatro para tantos por uno.

    El catálogo expresa retornos y volatilidades en tanto por uno (0,085 = 8,5 %):
    con dos decimales un 0,085 se vería como 0,09 y una volatilidad de 0,005 como 0,01.
    """
    return f"{valor:.4f}" if 0 < abs(valor) < 1 else f"{valor:,.2f}"


pd.set_option("display.float_format", formato_numero)


COLOR_PERFIL = {
    "Conservador": "#2a78d6",  # azul
    "Moderado": "#eb6834",  # naranja
    "Agresivo": "#1baf7a",  # aqua
}
TINTA_PRIMARIA = "#0b0b0b"
TINTA_SECUNDARIA = "#52514e"
SUPERFICIE = "#fcfcfb"
GRIS_GRILLA = "#d8d7d2"
RAMPA_AZUL = ["#f2f7fd", "#d6e6f8", "#a9cbf0", "#6ba6e3", "#2a78d6", "#1b5ba3"]

pio.templates.default = "plotly_white"


def formato_es(valor: float, decimales: int = 0) -> str:
    """Formatea con punto de miles y coma decimal, como se escribe en Uruguay."""
    texto = f"{valor:,.{decimales}f}"
    return texto.replace(",", "\x00").replace(".", ",").replace("\x00", ".")


# --- Gráficos para presentar ---------------------------------------------------------
# Los gráficos están pensados para mostrarse en una junta con directivos: el título dice
# la conclusión, las etiquetas van en palabras y no en nombres de columna, y el detalle
# aparece al pasar el mouse en lugar de estar escrito en el dibujo
FUENTE = "Segoe UI, Inter, Arial, sans-serif"
GRIS_NEUTRO = "#b5b4ae"

NOMBRE_LEGIBLE = {
    "meses_gastos_cubiertos": "Meses de gastos cubiertos",
    "patrimonio_invertible_usd": "Patrimonio invertible",
    "objetivo_principal": "Objetivo principal",
    "reaccion_caida_20": "Reacción ante una caída del 20 %",
    "ingreso_mensual_liquido": "Ingreso mensual",
    "edad": "Edad",
    "fuente_informacion": "Fuente de información",
    "retorno_esperado_anual": "Retorno esperado",
    "tiene_fondo_emergencia": "Tiene fondo de emergencia",
    "tolerancia_perdida_maxima": "Tolerancia a la pérdida máxima",
    "horizonte_inversion": "Horizonte de inversión",
    "frecuencia_monitoreo": "Frecuencia de monitoreo",
    "perfil_autodeclarado": "Perfil autodeclarado",
    "nivel_educativo": "Nivel educativo",
    "ocupacion": "Ocupación",
    "sexo": "Sexo",
    "tiene_deudas": "Tiene deudas",
    "departamento": "Departamento",
}


def nombre_legible(columna: str) -> str:
    """Nombre en palabras para mostrar en un gráfico."""
    return NOMBRE_LEGIBLE.get(columna, columna.replace("_", " ").capitalize())


def aclarar(color: str, cuanto: float = 0.55) -> str:
    """Mezcla el color con blanco: un tono suave y opaco del mismo color."""
    r, g, b = mcolors.to_rgb(color)
    return mcolors.to_hex(
        (r + (1 - r) * cuanto, g + (1 - g) * cuanto, b + (1 - b) * cuanto)
    )


def presentar(
    figura: go.Figure,
    titulo: str,
    subtitulo: str,
    alto: int = 460,
    ancho: int = 920,
    interactivo: bool = True,
) -> None:
    """Aplica el estilo común y muestra el gráfico.

    Sólo son interactivos los gráficos donde pasar el mouse agrega información que el
    dibujo no muestra; el resto se muestra fijo, sin cartel ni zoom. Todos llevan la
    fecha de corte de los datos.
    """
    if "30/06/2026" not in subtitulo:
        subtitulo += " · datos al 30/06/2026"
    figura.update_layout(
        title=dict(
            text=(
                f"<b>{titulo}</b><br><span style='font-size:12.5px;"
                f"color:{TINTA_SECUNDARIA}'>{subtitulo}</span>"
            ),
            x=0.01,
            xanchor="left",
            y=0.95,
            yanchor="top",
            yref="container",
            font=dict(size=17, color=TINTA_PRIMARIA),
        ),
        font=dict(family=FUENTE, size=12.5, color=TINTA_SECUNDARIA),
        paper_bgcolor=SUPERFICIE,
        plot_bgcolor=SUPERFICIE,
        separators=",.",
        hoverlabel=dict(
            bgcolor="white",
            bordercolor=GRIS_GRILLA,
            font=dict(family=FUENTE, size=12.5, color=TINTA_PRIMARIA),
            align="left",
        ),
        width=ancho,
        height=alto,
        margin=dict(t=110, l=20, r=30, b=60),
    )
    figura.update_xaxes(
        gridcolor=GRIS_GRILLA, zeroline=False, linecolor=GRIS_GRILLA, automargin=True
    )
    figura.update_yaxes(
        gridcolor=GRIS_GRILLA, zeroline=False, linecolor=GRIS_GRILLA, automargin=True
    )
    figura.show(config={"staticPlot": not interactivo})


print(f"pandas {pd.__version__} | numpy {np.__version__}")

pandas 3.0.3 | numpy 2.4.6


In [2]:
def encontrar_raiz_proyecto(ancla: str = "datos/encuesta_inversores.csv") -> Path:
    """Primera carpeta, desde el directorio actual hacia arriba, que tiene el ancla."""
    for carpeta in (Path.cwd(), *Path.cwd().parents):
        if (carpeta / ancla).exists():
            return carpeta
    raise FileNotFoundError(
        f"No encontré '{ancla}' desde {Path.cwd()}. "
        "Corré el notebook dentro del repositorio, respetando su estructura."
    )


RAIZ = encontrar_raiz_proyecto()
DATOS = RAIZ / "datos"
SALIDAS = RAIZ / "salidas"
SALIDAS.mkdir(exist_ok=True)


print(f"Raíz del proyecto : {RAIZ}")
print(f"Datos de entrada: {DATOS}")
print(f"Salidas: {SALIDAS}")

Raíz del proyecto : C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach
Datos de entrada: C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\datos
Salidas: C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\salidas


In [3]:
# Dato de la consigna: fecha de corte de toda la información
FECHA_CORTE = pd.Timestamp("2026-06-30")

# Exploración y calidad de datos

## Carga

In [4]:
COLUMNAS_SUCIAS = [
    "edad",
    "ingreso_mensual_liquido",
    "patrimonio_invertible_usd",
    "meses_gastos_cubiertos",
    "anios_experiencia_invirtiendo",
    "puntaje_test_conocimiento",
]

TIPOS_ENCUESTA = {
    "id_cliente": "string",
    **{c: "string" for c in COLUMNAS_SUCIAS},
    "cantidad_dependientes": "Int64",
    "conocimiento_autopercibido": "Int64",
    **{
        f"prioridad_{p}": "Int64"
        for p in [
            "plazo_fijo",
            "letras_bcu",
            "fondos_locales",
            "obligaciones",
            "etf_exterior",
            "inmuebles",
            "criptomonedas",
        ]
    },
}

TIPOS_MOVIMIENTOS = {
    "id_movimiento": "string",
    "id_cliente": "string",
    "id_instrumento": "Int64",
    "tipo_operacion": "string",
    "moneda": "string",
    "monto": "string",
}

TIPOS_CATALOGO = {
    "id_instrumento": "Int64",
    "nombre_instrumento": "string",
    "categoria": "string",
    "moneda": "string",
    "inversion_minima": "float64",
    "retorno_anual_historico": "float64",
    "volatilidad_anual": "float64",
    "nivel_riesgo": "Int64",
    "liquidez": "string",
}

encuesta_cruda = pd.read_csv(DATOS / "encuesta_inversores.csv", dtype=TIPOS_ENCUESTA)
movimientos_crudos = pd.read_csv(
    DATOS / "movimientos_cartera.csv", dtype=TIPOS_MOVIMIENTOS
)
catalogo = pd.read_csv(DATOS / "catalogo_instrumentos.csv", dtype=TIPOS_CATALOGO)
tasas = pd.read_csv(
    DATOS / "tasas_referencia.csv",
    dtype={"concepto": "string", "unidad": "string"},
    parse_dates=["fecha"],
)

for nombre, tabla in [
    ("encuesta", encuesta_cruda),
    ("movimientos", movimientos_crudos),
    ("catálogo", catalogo),
    ("tasas", tasas),
]:
    print(f"{nombre}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")

encuesta: 657 filas x 31 columnas
movimientos: 4481 filas x 7 columnas
catálogo: 14 filas x 9 columnas
tasas: 8 filas x 4 columnas


In [5]:
# Primeras filas de cada archivo, para verificar que cargó lo que se esperaba

print(
    "=" * 100,
    "\nENCUESTA — primeras filas (subconjunto de columnas, son 31)\n",
    "=" * 100,
)
display(
    encuesta_cruda[
        [
            "id_cliente",
            "fecha_respuesta",
            "edad",
            "departamento",
            "ingreso_mensual_liquido",
            "patrimonio_invertible_usd",
            "perfil_autodeclarado",
            "retorno_esperado_anual",
        ]
    ].head()
)

print("=" * 100, "\nMOVIMIENTOS — primeras filas\n", "=" * 100)
display(movimientos_crudos.head())

print(
    "=" * 100,
    "\nCATÁLOGO — las 14 filas, es la tabla de referencia del proyecto\n",
    "=" * 100,
)
display(catalogo)

print("=" * 100, "\nTASAS DE REFERENCIA — al 30/06/2026\n", "=" * 100)
display(tasas)

ENCUESTA — primeras filas (subconjunto de columnas, son 31)


,id_cliente,fecha_respuesta,edad,departamento,ingreso_mensual_liquido,patrimonio_invertible_usd,perfil_autodeclarado,retorno_esperado_anual
0,20504,2026-06-24,52.0,Montevideo,141700,4200.0,Conservador,10% a 20%
1,20466,2026-03-29,62.0,Colonia,$ 131.300,13900.0,Moderado,Menos de 5%
2,20432,2026-01-11,40.0,Maldonado,53700,4600.0,moderado,5% a 10%
3,20052,2026-02-21,48.0,Canelones,0,27700.0,Conservador,Menos de 5%
4,20084,2026-04-01,82.0,Canelones,85100,12100.0,CONSERVADOR,5% a 10%


MOVIMIENTOS — primeras filas


,id_movimiento,id_cliente,id_instrumento,fecha_operacion,tipo_operacion,moneda,monto
0,700954,20141,4,2023-08-06,Compra,UI,17803.22
1,700912,20135,5,2023-11-28,Suscripción,UYU,46255.68
2,700236,20033,3,26/02/2025,Suscripción,UYU,39224.29
3,700992,20148,8,2023-12-09,Suscripción,UYU,195561.19
4,701697,20255,12,17/05/2024,Compra,USD,4324.05


CATÁLOGO — las 14 filas, es la tabla de referencia del proyecto


,id_instrumento,nombre_instrumento,categoria,moneda,inversion_minima,retorno_anual_historico,volatilidad_anual,nivel_riesgo,liquidez
0,1,Plazo fijo en pesos (BROU),Renta fija local,UYU,"1,000.00",0.0850,0.0050,1,Baja
1,2,Plazo fijo en dólares,Renta fija local,USD,500.00,0.0330,0.0040,1,Baja
2,3,Letras de Regulación Monetaria (BCU),Renta fija local,UYU,"50,000.00",0.0920,0.0130,1,Media
3,4,Notas del Tesoro en Unidades Indexadas,Renta fija local,UI,"30,000.00",0.0310,0.0260,2,Media
4,5,Fondo de inversión local (AFISA),Fondos,UYU,"20,000.00",0.0710,0.0410,2,Alta
5,6,Obligaciones negociables (BVM),Renta fija corporativa,USD,"5,000.00",0.0650,0.0580,3,Baja
6,7,Fideicomiso financiero,Renta fija corporativa,UI,"40,000.00",0.0790,0.0720,3,Muy baja
7,8,Ahorro voluntario en AFAP,Previsional,UYU,"2,000.00",0.0490,0.0610,3,Muy baja
8,9,ETF de índice global (S&P 500 / MSCI World),Renta variable,USD,"1,000.00",0.0940,0.1520,4,Alta
9,10,Acciones individuales del exterior,Renta variable,USD,500.00,0.1010,0.2430,5,Alta


TASAS DE REFERENCIA — al 30/06/2026


,concepto,valor,unidad,fecha
0,"Inflación anual Uruguay (IPC, últimos 12 meses)",5.40,%,2026-06-30
1,Meta de inflación del BCU (centro del rango),4.50,%,2026-06-30
2,Tasa de política monetaria del BCU,8.75,%,2026-06-30
3,Tipo de cambio peso / dólar,42.80,UYU por USD,2026-06-30
4,Valor de la Unidad Indexada,6.55,UYU por UI,2026-06-30
5,Rendimiento de la Letra de Regulación Monetari...,9.20,%,2026-06-30
6,Inflación anual de Estados Unidos,2.70,%,2026-06-30
7,Rendimiento del bono del Tesoro de EE.UU. a 10...,4.10,%,2026-06-30


In [6]:
print("--- Tipos declarados ---")
print(encuesta_cruda.dtypes.value_counts().to_string())

print("\n--- Control de integridad referencial ---")
instrumentos_validos = set(catalogo["id_instrumento"])
fuera = ~movimientos_crudos["id_instrumento"].isin(instrumentos_validos)
print(f"Movimientos con id_instrumento fuera del catálogo: {int(fuera.sum())}")
print(
    f"Instrumentos del catálogo sin ninguna operación : "
    f"{len(instrumentos_validos - set(movimientos_crudos['id_instrumento'].dropna()))}"
)

--- Tipos declarados ---
str       15
Int64      9
string     7

--- Control de integridad referencial ---
Movimientos con id_instrumento fuera del catálogo: 0
Instrumentos del catálogo sin ninguna operación : 0


Los 14 instrumentos del catálogo tienen operaciones y no hay operaciones contra instrumentos que no existen, así que el cruce entre las dos tablas es completo.

## Panorama general

In [7]:
def describir_estructura(tabla: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Una fila por columna: tipo, no nulos, nulos, % de nulos y valores únicos."""
    resumen = pd.DataFrame(
        {
            "tipo": tabla.dtypes.astype("string"),
            "no_nulos": tabla.notna().sum(),
            "nulos": tabla.isna().sum(),
            "pct_nulos": (tabla.isna().mean() * 100).round(2),
            "unicos": tabla.nunique(dropna=True),
        }
    )
    resumen.index.name = f"{nombre} ({tabla.shape[0]}x{tabla.shape[1]})"
    return resumen


display(describir_estructura(encuesta_cruda, "encuesta"))
display(describir_estructura(movimientos_crudos, "movimientos"))

,tipo,no_nulos,nulos,pct_nulos,unicos
encuesta (657x31),,,,,
id_cliente,string,657,0,0.00,640
fecha_respuesta,str,657,0,0.00,258
edad,string,651,6,0.9100,59
sexo,str,657,0,0.00,2
departamento,str,657,0,0.00,25
cantidad_dependientes,Int64,657,0,0.00,6
nivel_educativo,str,657,0,0.00,4
ocupacion,str,657,0,0.00,6
ingreso_mensual_liquido,string,643,14,2.13,537


,tipo,no_nulos,nulos,pct_nulos,unicos
movimientos (4481x7),,,,,
id_movimiento,string,4481,0,0.00,4481
id_cliente,string,4481,0,0.00,706
id_instrumento,Int64,4481,0,0.00,14
fecha_operacion,str,4481,0,0.00,1842
tipo_operacion,string,4481,0,0.00,2
moneda,string,4481,0,0.00,3
monto,string,4429,52,1.16,1738


In [8]:
# Cardinalidad: dónde hay más categorías de las que deberían existir

COLUMNAS_CATEGORICAS = [
    "sexo",
    "departamento",
    "nivel_educativo",
    "ocupacion",
    "tiene_deudas",
    "tiene_fondo_emergencia",
    "perfil_autodeclarado",
    "tolerancia_perdida_maxima",
    "horizonte_inversion",
    "frecuencia_monitoreo",
    "retorno_esperado_anual",
    "reaccion_caida_20",
    "objetivo_principal",
    "fuente_informacion",
]

# Cuántas categorías DEBERÍA tener cada variable según el diccionario de datos
CATEGORIAS_ESPERADAS = {
    "sexo": 2,
    "departamento": 19,
    "nivel_educativo": 4,
    "ocupacion": 6,
    "tiene_deudas": 2,
    "tiene_fondo_emergencia": 2,
    "perfil_autodeclarado": 3,
    "tolerancia_perdida_maxima": 5,
    "horizonte_inversion": 4,
    "frecuencia_monitoreo": 5,
    "retorno_esperado_anual": 5,
    "reaccion_caida_20": 4,
    "objetivo_principal": 6,
    "fuente_informacion": 6,
}

cardinalidad = pd.DataFrame(
    {
        "observadas": encuesta_cruda[COLUMNAS_CATEGORICAS].nunique(),
        "esperadas": pd.Series(CATEGORIAS_ESPERADAS),
    }
)
cardinalidad["exceso"] = cardinalidad["observadas"] - cardinalidad["esperadas"]
display(cardinalidad.sort_values("exceso", ascending=False))

print(
    "\nVariables con categorías de más (= misma respuesta escrita de varias maneras):"
)
for columna in cardinalidad.index[cardinalidad["exceso"] > 0]:
    print(f"\n[{columna}]")
    print(encuesta_cruda[columna].value_counts(dropna=False).to_string())

,observadas,esperadas,exceso
departamento,25,19,6
perfil_autodeclarado,6,3,3
fuente_informacion,9,6,3
frecuencia_monitoreo,8,5,3
horizonte_inversion,7,4,3
tiene_fondo_emergencia,4,2,2
tiene_deudas,2,2,0
ocupacion,6,6,0
nivel_educativo,4,4,0
sexo,2,2,0



Variables con categorías de más (= misma respuesta escrita de varias maneras):

[departamento]
departamento
Montevideo        264
Canelones          80
Maldonado          61
MONTEVIDEO         46
Colonia            26
Paysandú           23
Salto              22
Canelones          17
Rivera             14
Rocha              13
San José           13
Tacuarembó         13
Soriano            13
Florida             8
Lavalleja           8
Cerro Largo         7
Río Negro           6
Treinta y Tres      4
San Jose            4
Rio Negro           3
Durazno             3
Flores              3
Tacuarembo          2
Artigas             2
Paysandu            2

[tiene_fondo_emergencia]
tiene_fondo_emergencia
No    317
Sí    271
NO     39
SI     30

[perfil_autodeclarado]
perfil_autodeclarado
Conservador    285
Moderado       157
Agresivo       127
CONSERVADOR     42
moderado        30
Agresivo        16

[horizonte_inversion]
horizonte_inversion
1 a 3 años        194
3 a 5 años        167
Más de

Seis variables tienen más categorías de las que existen, y es la misma respuesta escrita de otra forma.Las unifico ya.

In [9]:
# Normalización de las categóricas
encuesta = encuesta_cruda.copy()


def limpiar_texto(serie: pd.Series) -> pd.Series:
    """Quita espacios de los bordes y colapsa espacios internos repetidos."""
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )


MAPAS_CATEGORIAS = {
    "tiene_deudas": {"sí": "Sí", "si": "Sí", "no": "No"},
    "tiene_fondo_emergencia": {"sí": "Sí", "si": "Sí", "no": "No"},
    "perfil_autodeclarado": {
        "conservador": "Conservador",
        "moderado": "Moderado",
        "agresivo": "Agresivo",
    },
    "horizonte_inversion": {
        "menos de 1 año": "Menos de 1 año",
        "menos de 1 anio": "Menos de 1 año",
        "1 a 3 años": "1 a 3 años",
        "1-3 años": "1 a 3 años",
        "1-3 anios": "1 a 3 años",
        "3 a 5 años": "3 a 5 años",
        "3-5 años": "3 a 5 años",
        "3-5 anios": "3 a 5 años",
        "más de 5 años": "Más de 5 años",
        "mas de 5 años": "Más de 5 años",
        "mas de 5 anios": "Más de 5 años",
    },
    "frecuencia_monitoreo": {
        "diaria": "Diaria",
        "semanal": "Semanal",
        "mensual": "Mensual",
        "trimestral": "Trimestral",
        # "Anual" y "Anual o menos" son la misma categoría del formulario original
        "anual": "Anual o menos",
        "anual o menos": "Anual o menos",
    },
    "fuente_informacion": {
        "redes sociales": "Redes sociales",
        "amigos y familia": "Amigos y familia",
        "investigación propia": "Investigación propia",
        "prensa especializada": "Prensa especializada",
        # "Asesor" y "Banco" son abreviaturas de las dos categorías largas
        "asesor financiero": "Asesor financiero",
        "asesor": "Asesor financiero",
        "banco o corredor de bolsa": "Banco o corredor de bolsa",
        "banco": "Banco o corredor de bolsa",
    },
}


TILDES = str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU")
DEPARTAMENTOS = [
    "Artigas",
    "Canelones",
    "Cerro Largo",
    "Colonia",
    "Durazno",
    "Flores",
    "Florida",
    "Lavalleja",
    "Maldonado",
    "Montevideo",
    "Paysandú",
    "Río Negro",
    "Rivera",
    "Rocha",
    "Salto",
    "San José",
    "Soriano",
    "Tacuarembó",
    "Treinta y Tres",
]
MAPA_DEPARTAMENTOS = {d.translate(TILDES).lower(): d for d in DEPARTAMENTOS}

categorias_antes = {c: encuesta[c].nunique() for c in COLUMNAS_CATEGORICAS}

for columna, mapa in MAPAS_CATEGORIAS.items():
    encuesta[columna] = (
        limpiar_texto(encuesta[columna]).str.lower().map(mapa).astype("string")
    )

encuesta["departamento"] = (
    limpiar_texto(encuesta["departamento"])
    .str.translate(TILDES)
    .str.lower()
    .map(MAPA_DEPARTAMENTOS)
    .astype("string")
)

for columna in [
    "sexo",
    "nivel_educativo",
    "ocupacion",
    "tolerancia_perdida_maxima",
    "retorno_esperado_anual",
    "reaccion_caida_20",
    "objetivo_principal",
]:
    encuesta[columna] = limpiar_texto(encuesta[columna])

comparar = pd.DataFrame(
    {
        "categorias_antes": pd.Series(categorias_antes),
        "categorias_despues": pd.Series(
            {c: encuesta[c].nunique() for c in COLUMNAS_CATEGORICAS}
        ),
        "esperadas": pd.Series(CATEGORIAS_ESPERADAS),
    }
)
display(comparar)

sobrantes = comparar[comparar["categorias_despues"] > comparar["esperadas"]]
print(
    "Variables que todavía exceden lo esperado:",
    sobrantes.index.tolist() or "ninguna",
)

,categorias_antes,categorias_despues,esperadas
sexo,2,2,2
departamento,25,19,19
nivel_educativo,4,4,4
ocupacion,6,6,6
tiene_deudas,2,2,2
tiene_fondo_emergencia,4,2,2
perfil_autodeclarado,6,3,3
tolerancia_perdida_maxima,5,5,5
horizonte_inversion,7,4,4
frecuencia_monitoreo,8,5,5


Variables que todavía exceden lo esperado: ninguna


Después de unificar, cada variable tiene exactamente las categorías que debe tener.

### Números y fechas escritos a mano


In [10]:
# Convertir directo a número

textos_a_numero = {c: encuesta_cruda[c] for c in COLUMNAS_SUCIAS}
textos_a_numero["monto (movimientos)"] = movimientos_crudos["monto"]

perdidas = []
for columna, serie in textos_a_numero.items():
    texto = serie.str.strip()
    no_convierte = texto.notna() & pd.to_numeric(texto, errors="coerce").isna()
    perdidas.append(
        {
            "columna": columna,
            "con_dato": int(texto.notna().sum()),
            "no_convierte": int(no_convierte.sum()),
        }
    )
display(pd.DataFrame(perdidas).set_index("columna"))

# Qué tienen los valores que no convierten, tomando el ingreso como ejemplo
texto_ingreso = encuesta_cruda["ingreso_mensual_liquido"].str.strip()
fallan = texto_ingreso[
    texto_ingreso.notna() & pd.to_numeric(texto_ingreso, errors="coerce").isna()
]
forma = np.select(
    [
        fallan.str.contains(r"\$"),
        fallan.str.contains(r"\.\d{3},"),
        fallan.str.fullmatch(r"\d{1,3}(\.\d{3})+"),
        fallan.str.contains(r"\d,\d"),
    ],
    [
        "con símbolo $",
        "punto de miles y coma decimal",
        "punto de miles",
        "coma decimal",
    ],
    default="texto sin números",
)
print("Ingresos que la conversión directa no entiende, por forma:")
print(pd.Series(forma).value_counts().to_string())
print("\nEjemplos de cada forma:")
for nombre_forma in pd.unique(forma):
    print(f"  {nombre_forma}: {fallan[forma == nombre_forma].unique()[:3].tolist()}")

,con_dato,no_convierte
columna,,
edad,651,0
ingreso_mensual_liquido,643,283
patrimonio_invertible_usd,628,0
meses_gastos_cubiertos,624,0
anios_experiencia_invirtiendo,657,0
puntaje_test_conocimiento,657,0
monto (movimientos),4429,0


Ingresos que la conversión directa no entiende, por forma:
punto de miles y coma decimal    161
con símbolo $                     96
texto sin números                 26

Ejemplos de cada forma:
  con símbolo $: ['$ 131.300', '$ 99.700', '$ 201.700']
  punto de miles y coma decimal: ['22.200,00', '43.900,00', '78.500,00']
  texto sin números: ['Prefiero no contestar', 'S/D']


La conversión directa sólo falla en el ingreso, pero ahí pierde 283 de 643 valores 

- Por formato 161 ingresos tienen punto de miles y coma decimal y 96 llevan el símbolo $ Son números y se pueden convertir.
- 26 son textos ("Prefiero no contestar", "S/D") que dicen que el cliente no contestó. No son números.

In [11]:
def a_numero_uruguayo(serie: pd.Series) -> pd.Series:
    """Convierte a número texto escrito

    Maneja el símbolo de peso y los espacios (`"$ 131.300"`), el punto como separador
    de miles (`"131.300"`) y la coma decimal (`"22.000,00"`). Lo que no es un número
    queda vacío; esos casos se cuentan aparte.
    """
    s = serie.astype("string").str.strip()
    s = s.str.replace(r"[$\s]", "", regex=True)  # "$ 131.300" -> "131.300"

    # "22.000,00": el punto es miles y la coma es decimal
    tiene_ambos = s.str.contains(r"\.", na=False) & s.str.contains(",", na=False)
    s = s.mask(
        tiene_ambos,
        s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    )

    # "22000,50": sólo coma, es el decimal
    s = s.mask(s.str.contains(",", na=False), s.str.replace(",", ".", regex=False))

    # "131.300": puntos cada tres dígitos y sin cero adelante -> separador de miles
    # El `[1-9]` inicial evita destrozar un decimal legítimo como "0.085"
    es_miles = s.str.match(r"^-?[1-9]\d{0,2}(\.\d{3})+$", na=False)
    s = s.mask(es_miles, s.str.replace(".", "", regex=False))

    return pd.to_numeric(s, errors="coerce")


no_numericos = []
for columna, serie in textos_a_numero.items():
    texto = serie.str.strip()
    sigue = texto[texto.notna() & a_numero_uruguayo(serie).isna()]
    no_numericos.append(sigue.rename(columna))
    print(f"{columna}: quedan sin convertir {len(sigue)}")

textos_no_numericos = pd.concat(no_numericos)
print("\nLos valores que quedan sin convertir son estos textos:")
print(textos_no_numericos.value_counts().to_string())

# Decisión: las columnas se convierten; los textos que no son números quedan vacíos
for columna in COLUMNAS_SUCIAS:
    encuesta[columna] = a_numero_uruguayo(encuesta[columna])
movimientos = movimientos_crudos.copy()  # también trabajo sobre una copia
movimientos["monto"] = a_numero_uruguayo(movimientos["monto"])

edad: quedan sin convertir 0


ingreso_mensual_liquido: quedan sin convertir 26
patrimonio_invertible_usd: quedan sin convertir 0
meses_gastos_cubiertos: quedan sin convertir 0


anios_experiencia_invirtiendo: quedan sin convertir 0
puntaje_test_conocimiento: quedan sin convertir 0


monto (movimientos): quedan sin convertir 0

Los valores que quedan sin convertir son estos textos:
Prefiero no contestar    18
S/D                       8


Con la conversión propia todos los formatos pasan. Sólo quedan los 18 "Prefiero no contestar" y 8 "S/D"

In [12]:
# Formato de fechas

PATRON_FECHA_ISO = r"^\d{4}-\d{2}-\d{2}$"

for nombre, serie in [
    ("encuesta.fecha_respuesta", encuesta_cruda["fecha_respuesta"]),
    ("movimientos.fecha_operacion", movimientos_crudos["fecha_operacion"]),
]:
    texto = serie.str.strip()
    es_iso = texto.str.match(PATRON_FECHA_ISO)
    ejemplo = texto[~es_iso].iloc[0] if (~es_iso).any() else "-"
    print(
        f"{nombre}: {int(es_iso.sum())} como aaaa-mm-dd y {int((~es_iso).sum())} con "
        f"otro formato (por ejemplo {ejemplo})"
    )

fechas_crudas = encuesta_cruda["fecha_respuesta"].str.strip()
inferidas = pd.to_datetime(fechas_crudas, errors="coerce")
mixtas = pd.to_datetime(fechas_crudas, format="mixed", dayfirst=True, errors="coerce")
print(f"\nInferencia automática: {int(inferidas.isna().sum())} quedan sin convertir")
print(
    f'format="mixed" con dayfirst=True: {int((mixtas > FECHA_CORTE).sum())} quedan '
    "después de la fecha de corte"
)

encuesta.fecha_respuesta: 536 como aaaa-mm-dd y 121 con otro formato (por ejemplo 03/05/2026)
movimientos.fecha_operacion: 3796 como aaaa-mm-dd y 685 con otro formato (por ejemplo 26/02/2025)



Inferencia automática: 121 quedan sin convertir
format="mixed" con dayfirst=True: 101 quedan después de la fecha de corte


Las fechas vienen en dos formatos, la inferencia automática deja sin convertir las de la encuesta, y `format="mixed"` las convierte pero da vuelta día y mes en algunos casos que quedan después de la fecha de corte. Por eso separo por formato y convierto cada grupo con el suyo

In [13]:
# Convertir fechas separando por formato


def parsear_fechas_mixtas(serie: pd.Series) -> pd.Series:
    """Convierte una serie que mezcla 'aaaa-mm-dd' con 'dd/mm/aaaa'.

    Separa por patrón y convierte cada grupo con su formato explícito, para evitar que
    la inferencia automática dé vuelta día y mes.
    """
    texto = serie.astype("string").str.strip()
    es_iso = texto.str.match(PATRON_FECHA_ISO).fillna(False)

    fechas = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    fechas[es_iso] = pd.to_datetime(texto[es_iso], format="%Y-%m-%d", errors="coerce")
    fechas[~es_iso] = pd.to_datetime(texto[~es_iso], format="%d/%m/%Y", errors="coerce")
    return fechas


encuesta["fecha_respuesta"] = parsear_fechas_mixtas(encuesta["fecha_respuesta"])
movimientos["fecha_operacion"] = parsear_fechas_mixtas(movimientos["fecha_operacion"])

for nombre, serie in [
    ("encuesta.fecha_respuesta", encuesta["fecha_respuesta"]),
    ("movimientos.fecha_operacion", movimientos["fecha_operacion"]),
]:
    print(
        f"{nombre}: vacías={int(serie.isna().sum())} | "
        f"{serie.min().date()} -> {serie.max().date()} | "
        f"posteriores al corte: {int((serie > FECHA_CORTE).sum())}"
    )

encuesta.fecha_respuesta: vacías=0 | 2026-01-02 -> 2026-06-30 | posteriores al corte: 0
movimientos.fecha_operacion: vacías=0 | 2022-07-02 -> 2026-05-31 | posteriores al corte: 0


In [14]:
COLUMNAS_CONTINUAS = [
    "edad",
    "ingreso_mensual_liquido",
    "patrimonio_invertible_usd",
    "meses_gastos_cubiertos",
    "anios_experiencia_invirtiendo",
    "puntaje_test_conocimiento",
    "cantidad_dependientes",
    "conocimiento_autopercibido",
]

continuas = encuesta[COLUMNAS_CONTINUAS]
display(continuas.describe().T)

print("MOVIMIENTOS — monto por moneda (sin convertir todavía: no son comparables)")
montos = pd.DataFrame(
    {
        "moneda": movimientos["moneda"],
        "monto": movimientos["monto"],
    }
)
display(montos.groupby("moneda")["monto"].describe())

,count,mean,std,min,25%,50%,75%,max
edad,651.00,46.64,14.06,0.00,38.00,46.00,53.50,154.00
ingreso_mensual_liquido,617.00,"85,614.28","60,280.64",-999.00,"46,900.00","71,700.00","109,900.00","535,900.00"
patrimonio_invertible_usd,628.00,"24,964.99","29,705.02",-999.00,"7,400.00","15,900.00","31,650.00","279,100.00"
meses_gastos_cubiertos,624.00,2.93,3.93,0.00,0.00,0.00,6.00,12.00
anios_experiencia_invirtiendo,657.00,5.38,4.70,0.00,2.00,4.00,7.00,34.00
puntaje_test_conocimiento,657.00,4.30,2.06,-1.00,3.00,4.00,6.00,10.00
cantidad_dependientes,657.00,0.9072,0.9558,0.00,0.00,1.00,1.00,5.00
conocimiento_autopercibido,657.00,2.56,0.9982,1.00,2.00,3.00,3.00,5.00


MOVIMIENTOS — monto por moneda (sin convertir todavía: no son comparables)


,count,mean,std,min,25%,50%,75%,max
moneda,,,,,,,,
UI,831.00,"19,212.19","32,289.48","-21,478.63","4,357.52","9,375.69","19,409.11","419,665.76"
USD,"1,664.00","3,524.36","6,453.51","-10,651.22",639.19,"1,585.84","3,548.29","92,578.57"
UYU,"1,934.00","108,542.47","218,319.16","-718,329.50","19,142.84","42,716.68","115,281.62","5,317,319.60"


Ya se ven varios problemas: edades de 0 y de 154 años, un mínimo de −999 en ingreso y patrimonio (un "no contestó" que parece número), −1 en el test cuando la escala va de 0 a 10 y montos negativos en las tres monedas.

## Duplicados

Son dos problemas distintos. En la encuesta, el duplicado es una fila idéntica: el mismo cliente mandó el formulario dos veces. En las operaciones no hay filas idénticas, porque cada registro tiene un número nuevo, así que las busco por la combinación cliente, instrumento, fecha, tipo, moneda y monto.

En la encuesta también chequeo que ningún cliente repetido tenga respuestas distintas entre envíos.

In [15]:
# Duplicados en la encuesta

filas_antes = len(encuesta)
ids_repetidos = encuesta["id_cliente"].duplicated(keep=False)

print(f"Filas totalmente idénticas: {int(encuesta.duplicated().sum())}")
print(f"Filas con id_cliente repetido: {int(ids_repetidos.sum())}")
print(f"Clientes únicos: {encuesta['id_cliente'].nunique()}")

repetidos = encuesta[ids_repetidos]
contradictorios = len(repetidos) - int(repetidos.duplicated(keep=False).sum())
print(f"Ids repetidos con respuestas distintas: {contradictorios}")

print("\nEjemplo de un formulario enviado dos veces:")
id_ejemplo = encuesta.loc[ids_repetidos, "id_cliente"].iloc[0]
display(
    encuesta[encuesta["id_cliente"] == id_ejemplo][
        [
            "id_cliente",
            "fecha_respuesta",
            "edad",
            "perfil_autodeclarado",
            "ingreso_mensual_liquido",
            "patrimonio_invertible_usd",
        ]
    ]
)

encuesta = encuesta.drop_duplicates().reset_index(drop=True)
print(
    f"\nDuplicados eliminados: {filas_antes - len(encuesta)} de {filas_antes} "
    f"({(filas_antes - len(encuesta)) / filas_antes:.1%})"
)
assert encuesta["id_cliente"].is_unique, "Debe quedar una fila por cliente"

Filas totalmente idénticas: 17
Filas con id_cliente repetido: 34
Clientes únicos: 640
Ids repetidos con respuestas distintas: 0

Ejemplo de un formulario enviado dos veces:


,id_cliente,fecha_respuesta,edad,perfil_autodeclarado,ingreso_mensual_liquido,patrimonio_invertible_usd
8,20125,2026-05-03,53.00,Moderado,"62,600.00","10,200.00"
252,20125,2026-05-03,53.00,Moderado,"62,600.00","10,200.00"



Duplicados eliminados: 17 de 657 (2.6%)


In [16]:
# Duplicados en los movimientos

ops_antes = len(movimientos)
CLAVE_NEGOCIO = [
    "id_cliente",
    "id_instrumento",
    "fecha_operacion",
    "tipo_operacion",
    "moneda",
    "monto",
]

print(f"Filas totalmente idénticas: {int(movimientos.duplicated().sum())}")
print(
    f"id_movimiento repetidos: "
    f"{int(movimientos['id_movimiento'].duplicated().sum())}"
)

en_grupo_repetido = movimientos.duplicated(subset=CLAVE_NEGOCIO, keep=False)
print(
    f"Filas duplicadas por clave de negocio: {int(en_grupo_repetido.sum())} "
    f"({int(en_grupo_repetido.sum()) // 2} operaciones registradas dos veces)"
)

print("\nEjemplo: dos id_movimiento distintos para la misma operación")
display(movimientos[en_grupo_repetido].sort_values(CLAVE_NEGOCIO).head(4))

movimientos = (
    movimientos.sort_values("id_movimiento")
    .drop_duplicates(subset=CLAVE_NEGOCIO, keep="first")
    .reset_index(drop=True)
)

print(
    f"\nOperaciones duplicadas eliminadas: {ops_antes - len(movimientos)} de "
    f"{ops_antes} ({(ops_antes - len(movimientos)) / ops_antes:.1%})"
)

Filas totalmente idénticas: 0
id_movimiento repetidos: 0
Filas duplicadas por clave de negocio: 188 (94 operaciones registradas dos veces)

Ejemplo: dos id_movimiento distintos para la misma operación


,id_movimiento,id_cliente,id_instrumento,fecha_operacion,tipo_operacion,moneda,monto
1193,704397,20007,12,2025-09-02,Compra,USD,"1,733.81"
3241,700070,20007,12,2025-09-02,Compra,USD,"1,733.81"
269,700109,20013,14,2025-09-27,Compra,USD,173.64
3494,704429,20013,14,2025-09-27,Compra,USD,173.64



Operaciones duplicadas eliminadas: 94 de 4481 (2.1%)


El duplicado que importa es el de las operaciones. El riesgo de cada cliente se pondera por monto, así que una operación contada dos veces duplica el peso de ese instrumento y le cambia el perfil. Un duplicado en la encuesta sólo agrega una fila.

## Faltantes

In [17]:
def tabla_faltantes(tabla: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """Columnas con al menos un nulo, ordenadas por cantidad."""
    conteo = tabla.isna().sum()
    conteo = conteo[conteo > 0].sort_values(ascending=False)
    return pd.DataFrame(
        {
            "nulos": conteo,
            "pct": (conteo / len(tabla) * 100).round(2),
        }
    ).rename_axis(f"{nombre}: columnas con faltantes")


display(tabla_faltantes(encuesta, "encuesta"))
display(tabla_faltantes(movimientos, "movimientos"))

,nulos,pct
encuesta: columnas con faltantes,,
ingreso_mensual_liquido,39,6.09
meses_gastos_cubiertos,31,4.84
patrimonio_invertible_usd,29,4.53
objetivo_principal,26,4.06
reaccion_caida_20,19,2.97
edad,6,0.9400


,nulos,pct
movimientos: columnas con faltantes,,
monto,52,1.19


In [18]:
columnas_con_nulos = [c for c in encuesta.columns if encuesta[c].isna().any()]
filas_incompletas = encuesta[encuesta[columnas_con_nulos].isna().any(axis=1)]

print(
    f"Clientes con al menos un faltante declarado: {len(filas_incompletas)} "
    f"de {len(encuesta)} ({len(filas_incompletas) / len(encuesta):.1%})"
)
display(filas_incompletas[["id_cliente", *columnas_con_nulos]].head(10))

print("\nOperaciones sin monto (no se pueden usar para reconstruir la cartera):")
display(movimientos[movimientos["monto"].isna()].head(5))

Clientes con al menos un faltante declarado: 134 de 640 (20.9%)


,id_cliente,edad,ingreso_mensual_liquido,patrimonio_invertible_usd,meses_gastos_cubiertos,reaccion_caida_20,objetivo_principal
15,20550,58.00,0.00,<NA>,0,<NA>,Crecimiento del capital
26,20382,38.00,<NA>,"30,800.00",6,Compraría más,Compra de vivienda
33,20581,51.00,<NA>,"5,200.00",7,Compraría más,Crecimiento del capital
35,20284,<NA>,"124,600.00",0.00,0,No haría nada,Generar una renta mensual
40,20364,67.00,<NA>,"52,200.00",4,No haría nada,Compra de vivienda
52,20185,38.00,<NA>,"52,800.00",<NA>,Vendería todo,Crecimiento del capital
60,20283,38.00,"50,300.00","2,300.00",11,No haría nada,<NA>
62,20388,51.00,<NA>,"17,200.00",0,No haría nada,Compra de vivienda
64,20017,40.00,<NA>,"12,100.00",0,No haría nada,Complemento jubilatorio
76,20514,26.00,"386,400.00",<NA>,0,Compraría más,Crecimiento del capital



Operaciones sin monto (no se pueden usar para reconstruir la cartera):


,id_movimiento,id_cliente,id_instrumento,fecha_operacion,tipo_operacion,moneda,monto
121,700122,20015,4,2024-04-23,Compra,UI,<NA>
300,700301,20042,14,2023-08-14,Suscripción,USD,<NA>
306,700307,20045,6,2026-03-13,Compra,USD,<NA>
359,700360,20054,2,2026-05-06,Suscripción,USD,<NA>
596,700597,20088,3,2026-01-05,Compra,UYU,<NA>


In [19]:
conteo = encuesta.isna().sum()
conteo = conteo[conteo > 0].sort_values()
porcentaje = conteo / len(encuesta)

figura = go.Figure(
    go.Bar(
        x=porcentaje.values,
        y=[nombre_legible(columna) for columna in conteo.index],
        orientation="h",
        marker_color=COLOR_PERFIL["Conservador"],
        text=[f"{formato_es(v * 100, 1)} %" for v in porcentaje],
        textposition="outside",
        customdata=conteo.values,
        hovertemplate=(
            "<b>%{y}</b><br>%{customdata} de "
            + str(len(encuesta))
            + " clientes sin respuesta<extra></extra>"
        ),
    )
)
figura.update_xaxes(
    tickformat=".0%",
    range=[0, porcentaje.max() * 1.25],
    title="% de faltantes",
)
figura.update_yaxes(showgrid=False)
presentar(
    figura,
    f"{nombre_legible(porcentaje.index[-1])} es la variable con más faltantes: "
    f"{formato_es(porcentaje.iloc[-1] * 100, 1)} %",
    f"Encuesta de perfil · {len(encuesta)} clientes · al 30/06/2026",
    interactivo=False,
    alto=360,
)

El ingreso es la variable con más faltantes y ninguna otra pasa del 5 %, así que no descarto ninguna. Pero puede haber otras formas de decir "no contestó" que no quedan vacías.

### Montos de las operaciones

In [20]:
print("Tipos de operación:", movimientos["tipo_operacion"].dropna().unique().tolist())

negativos = movimientos["monto"] < 0
print(f"\nMontos negativos: {int(negativos.sum())} ({negativos.mean():.1%})")
print("Por moneda:", movimientos.loc[negativos, "moneda"].value_counts().to_dict())

sin_monto = movimientos["monto"].isna()
perdidas_por_cliente = movimientos.loc[sin_monto, "id_cliente"].value_counts()
quedarian_sin_nada = set(perdidas_por_cliente.index) - set(
    movimientos.loc[~sin_monto, "id_cliente"]
)
print(f"\nOperaciones sin monto: {int(sin_monto.sum())} ({sin_monto.mean():.1%})")
print(
    f"Clientes afectados: {len(perdidas_por_cliente)} · máximo por cliente: "
    f"{perdidas_por_cliente.max()} · se quedarían sin ninguna operación: "
    f"{len(quedarian_sin_nada)}"
)

Tipos de operación: ['Compra', 'Suscripción']

Montos negativos: 18 (0.4%)
Por moneda: {'UYU': 9, 'USD': 6, 'UI': 3}

Operaciones sin monto: 52 (1.2%)
Clientes afectados: 49 · máximo por cliente: 2 · se quedarían sin ninguna operación: 1


Sólo hay compras y suscripciones, así que un monto negativo no puede ser un retiro: los 18 negativos, repartidos en las tres monedas, son errores de signo y pasan a positivo.

Las 52 operaciones sin monto se sacan, no se puede ponderar un instrumento por un monto que no existe, y completarlo le inventaría al cliente una posición

In [21]:
movimientos["monto"] = movimientos["monto"].abs()
movimientos = movimientos[movimientos["monto"].notna()].reset_index(drop=True)
print(f"Operaciones que quedan: {len(movimientos)}")

Operaciones que quedan: 4335


## Centinelas y rangos imposibles

Un centinela es un valor que el sistema usa para indicar que no hay dato, puede venir como texto, código numérico, casi siempre negativo, etc. 

In [22]:
CENTINELA_NO_CONTESTA = -999  # ingreso y patrimonio
CENTINELA_TEST_NO_RENDIDO = -1  # puntaje del test, que va de 0 a 10

In [23]:
con_operaciones = encuesta["id_cliente"].isin(set(movimientos["id_cliente"]))
hallazgos = []
for columna in encuesta.columns:
    numerico = pd.to_numeric(encuesta[columna], errors="coerce")
    candidatos = [
        ("código -999", numerico == CENTINELA_NO_CONTESTA),
        ("código -1", numerico == CENTINELA_TEST_NO_RENDIDO),
    ]
    # El cero sólo es sospechoso en el ingreso y en el patrimonio: en los meses de
    # gastos cubiertos o en los dependientes es un valor normal. En el patrimonio es
    # imposible si el cliente tiene operaciones registradas
    if columna == "ingreso_mensual_liquido":
        candidatos.append(("cero (a revisar)", numerico == 0))
    if columna == "patrimonio_invertible_usd":
        candidatos.append(("cero con operaciones", (numerico == 0) & con_operaciones))
    candidatos.append(("vacío", encuesta[columna].isna()))
    for tipo, mascara in candidatos:
        if mascara.any():
            hallazgos.append(
                {"columna": columna, "tipo": tipo, "casos": int(mascara.sum())}
            )

display(pd.DataFrame(hallazgos).sort_values(["columna", "tipo"]).reset_index(drop=True))

,columna,tipo,casos
0,edad,vacío,6
1,ingreso_mensual_liquido,cero (a revisar),14
2,ingreso_mensual_liquido,código -999,9
3,ingreso_mensual_liquido,vacío,39
4,meses_gastos_cubiertos,vacío,31
5,objetivo_principal,vacío,26
6,patrimonio_invertible_usd,cero con operaciones,9
7,patrimonio_invertible_usd,código -999,13
8,patrimonio_invertible_usd,vacío,29
9,puntaje_test_conocimiento,código -1,18


Además del vacío y de los textos aparecen tres formas más: el −999 en ingreso y patrimonio, el −1 en el test y el 0 en ingreso y patrimonio.

El −999 y el −1 no son valores, son lo que grabó el formulario cuando el cliente no contestó, y dejarlos arrastraría promedios y correlaciones. En el patrimonio el cero se delata solo, esos clientes tienen operaciones con la firma, así que su patrimonio no puede ser cero. Los tres pasan a faltante en la celda de abajo.

En el ingreso un cero podría ser real para un estudiante, alguien sin trabajo

In [24]:
for columna, codigo in [
    ("ingreso_mensual_liquido", CENTINELA_NO_CONTESTA),
    ("patrimonio_invertible_usd", CENTINELA_NO_CONTESTA),
    ("puntaje_test_conocimiento", CENTINELA_TEST_NO_RENDIDO),
]:
    es_centinela = encuesta[columna] == codigo
    encuesta[columna] = encuesta[columna].mask(es_centinela)
    print(f"{columna}: {codigo} -> faltante en {int(es_centinela.sum())}")

patrimonio_cero = (encuesta["patrimonio_invertible_usd"] == 0) & con_operaciones
encuesta["patrimonio_invertible_usd"] = encuesta["patrimonio_invertible_usd"].mask(
    patrimonio_cero
)
print(
    f"patrimonio_invertible_usd: 0 con operaciones -> faltante en "
    f"{int(patrimonio_cero.sum())}"
)

ingreso_mensual_liquido: -999 -> faltante en 9
patrimonio_invertible_usd: -999 -> faltante en 13
puntaje_test_conocimiento: -1 -> faltante en 18
patrimonio_invertible_usd: 0 con operaciones -> faltante en 9


In [25]:
# Quiénes son los clientes con ingreso líquido declarado igual a cero?

ingreso_numerico = encuesta["ingreso_mensual_liquido"]
ingreso_cero = ingreso_numerico == 0

print(
    f"Clientes con ingreso mensual líquido declarado = 0: {int(ingreso_cero.sum())}\n"
)
print("Ocupación de esos clientes:")
print(encuesta.loc[ingreso_cero, "ocupacion"].value_counts().to_string())

print("\nPatrimonio invertible que declaran (USD):")
display(encuesta.loc[ingreso_cero, "patrimonio_invertible_usd"].describe().to_frame().T)

OCUPACIONES_SIN_INGRESO = ["Estudiante", "Jubilado"]
legitimos = int(
    encuesta.loc[ingreso_cero, "ocupacion"].isin(OCUPACIONES_SIN_INGRESO).sum()
)
print(
    f"De esos clientes, los que podrían legítimamente no tener ingreso "
    f"(Estudiante o Jubilado): {legitimos}"
)

Clientes con ingreso mensual líquido declarado = 0: 14

Ocupación de esos clientes:
ocupacion
Empresario / dueño de negocio          6
Profesional independiente              3
Empleado en relación de dependencia    3
Empleado público                       2

Patrimonio invertible que declaran (USD):


,count,mean,std,min,25%,50%,75%,max
patrimonio_invertible_usd,12.00,"26,741.67","22,524.07","3,300.00","7,300.00","25,700.00","38,800.00","73,300.00"


De esos clientes, los que podrían legítimamente no tener ingreso (Estudiante o Jubilado): 0


Los 14 tienen un trabajo activo y un patrimonio mediano de USD 25.700. Ninguno es estudiante ni jubilado. El 0 es un campo vacío que el formulario grabó como número, así que lo paso a faltante.

In [26]:
CENTINELA_CAMPO_VACIO = 0
ingreso_cero = encuesta["ingreso_mensual_liquido"] == CENTINELA_CAMPO_VACIO
encuesta["ingreso_mensual_liquido"] = encuesta["ingreso_mensual_liquido"].mask(
    ingreso_cero
)
print(f"ingreso_mensual_liquido: 0 -> faltante en {int(ingreso_cero.sum())}")

ingreso_mensual_liquido: 0 -> faltante en 14


In [27]:
# Rangos imposibles en las numéricas

# Rangos admisibles según el diccionario de datos y el sentido del negocio
# La cota inferior de edad es 18: en Uruguay un menor no puede contratar por sí mismo
# los instrumentos del catálogo, así que una edad menor es un error de carga
RANGOS_VALIDOS = {
    "edad": (18, 95),
    "ingreso_mensual_liquido": (0, None),
    "patrimonio_invertible_usd": (0, None),
    "meses_gastos_cubiertos": (0, 24),
    "anios_experiencia_invirtiendo": (0, 80),
    "puntaje_test_conocimiento": (0, 10),
    "conocimiento_autopercibido": (1, 5),
    "cantidad_dependientes": (0, 15),
}

filas_control = []
for columna, (minimo, maximo) in RANGOS_VALIDOS.items():
    valores = encuesta[columna]
    fuera = pd.Series(False, index=valores.index)
    if minimo is not None:
        fuera |= valores < minimo
    if maximo is not None:
        fuera |= valores > maximo
    filas_control.append(
        {
            "columna": columna,
            "min_admitido": minimo,
            "max_admitido": maximo,
            "min_observado": valores.min(),
            "max_observado": valores.max(),
            "fuera_de_rango": int(fuera.sum()),
        }
    )

display(pd.DataFrame(filas_control).set_index("columna"))

,min_admitido,max_admitido,min_observado,max_observado,fuera_de_rango
columna,,,,,
edad,18,95.00,0.00,154.00,16
ingreso_mensual_liquido,0,NaN,"22,000.00","535,900.00",0
patrimonio_invertible_usd,0,NaN,"1,500.00","279,100.00",0
meses_gastos_cubiertos,0,24.00,0.00,12.00,0
anios_experiencia_invirtiendo,0,80.00,0.00,34.00,0
puntaje_test_conocimiento,0,10.00,0.00,10.00,0
conocimiento_autopercibido,1,5.00,1.00,5.00,0
cantidad_dependientes,0,15.00,0.00,5.00,0


In [28]:
edad_numerica = encuesta["edad"]
edad_sospechosa = (edad_numerica < 18) | (edad_numerica > 95)

display(
    encuesta.loc[
        edad_sospechosa,
        [
            "id_cliente",
            "edad",
            "ocupacion",
            "anios_experiencia_invirtiendo",
            "patrimonio_invertible_usd",
            "perfil_autodeclarado",
        ],
    ].sort_values("edad")
)

# Nadie puede haber empezado a invertir antes de los 18
# Es la prueba de que estas edades son errores de carga y no clientes atípicos
experiencia = encuesta["anios_experiencia_invirtiendo"]
edad_al_empezar = edad_numerica - experiencia
incoherente = edad_al_empezar < 18

print(
    "\nClientes que, según sus respuestas, empezaron a invertir antes de los 18 años:"
)
print(f"  sobre la base cruda: {int(incoherente.sum())}")
print(
    f"  excluyendo las edades fuera de [18, 95] : "
    f"{int((incoherente & ~edad_sospechosa).sum())}"
)

print("\nLos casos de 16 y 17 años, con su experiencia declarada:")
display(
    encuesta.loc[
        edad_numerica.between(16, 17),
        ["id_cliente", "edad", "ocupacion", "anios_experiencia_invirtiendo"],
    ].assign(edad_al_empezar=edad_al_empezar)
)

,id_cliente,edad,ocupacion,anios_experiencia_invirtiendo,patrimonio_invertible_usd,perfil_autodeclarado
6,20254,0.00,Jubilado,8.00,"12,200.00",Conservador
84,20255,0.00,Empleado en relación de dependencia,1.00,"39,500.00",Conservador
118,20079,0.00,Jubilado,8.00,"28,100.00",Conservador
112,20109,0.00,Empleado en relación de dependencia,2.00,"38,400.00",Conservador
501,20519,0.00,Empleado en relación de dependencia,3.00,"9,500.00",Moderado
511,20068,0.00,Empleado en relación de dependencia,12.00,"35,800.00",Conservador
386,20053,0.00,Estudiante,3.00,"11,700.00",Agresivo
414,20586,0.00,Empresario / dueño de negocio,2.00,"6,900.00",Conservador
462,20336,16.00,Empleado público,7.00,"60,700.00",Conservador
290,20128,16.00,Profesional independiente,2.00,"57,100.00",Moderado



Clientes que, según sus respuestas, empezaron a invertir antes de los 18 años:
  sobre la base cruda: 14
  excluyendo las edades fuera de [18, 95] : 0

Los casos de 16 y 17 años, con su experiencia declarada:


,id_cliente,edad,ocupacion,anios_experiencia_invirtiendo,edad_al_empezar
45,20104,16.00,Empleado público,19.00,-3.00
240,20152,16.00,Jubilado,22.00,-6.00
290,20128,16.00,Profesional independiente,2.00,14.00
367,20339,17.00,Empleado en relación de dependencia,1.00,16.00
462,20336,16.00,Empleado público,7.00,9.00
571,20070,16.00,Empleado público,4.00,12.00


Son errores de carga, no clientes raros: hay un jubilado de 16 años con 22 años de experiencia invirtiendo y un empleado público de 16 con 19.

En la base hay 14 clientes que habrían empezado a invertir antes de los 18 años, y sacando las edades fuera de 18 a 95 no queda ninguno. El rango no está sacando clientes reales.

Esas edades pasan a faltante en vez de corregirse, porque no hay forma de saber el valor real

In [29]:
# los valores fuera de rango pasan a faltante

for columna, (minimo, maximo) in RANGOS_VALIDOS.items():
    valores = encuesta[columna]
    fuera = pd.Series(False, index=valores.index)
    if minimo is not None:
        fuera |= valores < minimo
    if maximo is not None:
        fuera |= valores > maximo
    if fuera.any():
        encuesta[columna] = valores.mask(fuera)
        print(
            f"{columna}: fuera de [{minimo}, {maximo}] -> faltante en "
            f"{int(fuera.sum())}"
        )

edad: fuera de [18, 95] -> faltante en 16


## Vacío por error vs. pregunta salteada

El vacío en meses de gastos cubiertos no significa lo mismo para todos. Si el cliente dijo que no tiene fondo de emergencia, la pregunta no le aplica y la respuesta es cero. Si dijo que sí tiene, el dato falta de verdad. Ponerles cero a todos convertiría a clientes con reserva en clientes sin reserva

In [30]:
tiene_fondo = encuesta["tiene_fondo_emergencia"]
meses = encuesta["meses_gastos_cubiertos"]

cruce = pd.crosstab(
    tiene_fondo,
    meses.isna(),
    rownames=["¿tiene fondo de emergencia?"],
    colnames=["meses_gastos_cubiertos está vacío"],
)
display(cruce)

print("Valores de meses_gastos_cubiertos cuando el cliente dice NO tener fondo:")
print(meses[tiene_fondo == "No"].value_counts(dropna=False).to_string())
print("\nValores cuando dice SÍ tener fondo (resumen):")
display(meses[tiene_fondo == "Sí"].describe().to_frame().T)

meses_gastos_cubiertos está vacío,False,True
¿tiene fondo de emergencia?,,
No,332,17
Sí,277,14


Valores de meses_gastos_cubiertos cuando el cliente dice NO tener fondo:
meses_gastos_cubiertos
0       332
<NA>     17

Valores cuando dice SÍ tener fondo (resumen):


,count,mean,std,min,25%,50%,75%,max
meses_gastos_cubiertos,277.00,6.34,3.36,1.00,4.00,6.00,9.00,12.00


Los 332 clientes sin fondo que contestaron pusieron 0, así que a los 17 de ese grupo que la dejaron vacía les pongo 0. Los 14 que dicen tener fondo quedan como faltantes.

In [31]:
salteada = meses.isna() & (tiene_fondo == "No")
error_carga = meses.isna() & (tiene_fondo == "Sí")

encuesta["meses_gastos_cubiertos"] = meses
encuesta.loc[salteada, "meses_gastos_cubiertos"] = 0

print(
    f"Vacíos imputados con 0 (no tiene fondo): {int(salteada.sum())} "
    f"({salteada.mean():.1%})"
)
print(
    f"Vacíos que quedan faltantes (sí tiene fondo): {int(error_carga.sum())} "
    f"({error_carga.mean():.1%})"
)

Vacíos imputados con 0 (no tiene fondo): 17 (2.7%)
Vacíos que quedan faltantes (sí tiene fondo): 14 (2.2%)


La reacción ante una caída del 20 % y el objetivo principal son las únicas preguntas de opción con faltantes, y son de opinión. Borrar la fila perdería las otras 30 respuestas del cliente, y completar con la respuesta más común le inventaría una opinión. Las dejo como una categoría más, "No contesta".

In [32]:
PREGUNTAS_SALTEABLES = ["reaccion_caida_20", "objetivo_principal"]

for columna in PREGUNTAS_SALTEABLES:
    faltan = encuesta[columna].isna()
    print(f"{columna}: {int(faltan.sum())} sin contestar ({faltan.mean():.1%})")
    encuesta[columna] = encuesta[columna].fillna("No contesta")

reaccion_caida_20: 19 sin contestar (3.0%)
objetivo_principal: 26 sin contestar (4.1%)


## ¿Quién es el que no contesta?

El patrimonio declarado se puede contrastar con el que la firma le conoce al cliente por sus operaciones. Son fuentes independientes: el cliente puede dejar el formulario en blanco, pero no puede esconder lo que operó. Calculo cuánto invirtió cada uno y comparo a los que contestan con los que no.

In [33]:
# Patrimonio observado: cuánto le conoce la firma a cada cliente por sus operaciones
# Es una primera aproximación en dólares, sólo para el análisis de faltantes


def leer_tasa(patron: str) -> float:
    """Devuelve el valor del único indicador cuyo concepto coincide con el patrón."""
    fila = tasas[tasas["concepto"].str.contains(patron, case=False, regex=True)]
    if len(fila) != 1:
        raise ValueError(
            f"El patrón '{patron}' coincide con {len(fila)} indicadores, debe ser 1"
        )
    return float(fila["valor"].iloc[0])


TIPO_CAMBIO = leer_tasa("Tipo de cambio")  # UYU por USD
VALOR_UI = leer_tasa("Unidad Indexada")  # UYU por UI
# Una UI vale VALOR_UI pesos, y un dólar vale TIPO_CAMBIO pesos
FACTOR_A_USD = {"USD": 1.0, "UYU": 1 / TIPO_CAMBIO, "UI": VALOR_UI / TIPO_CAMBIO}

print(f"Tipo de cambio: {TIPO_CAMBIO} UYU/USD | Unidad Indexada: {VALOR_UI} UYU/UI")

monto_usd = movimientos["monto"] * movimientos["moneda"].map(FACTOR_A_USD)
patrimonio_observado = monto_usd.groupby(movimientos["id_cliente"]).sum()

print(f"\nClientes con operaciones registradas: {len(patrimonio_observado)}")
print(f"Patrimonio observado total: USD {patrimonio_observado.sum():,.0f}")

Tipo de cambio: 42.8 UYU/USD | Unidad Indexada: 6.55 UYU/UI



Clientes con operaciones registradas: 705
Patrimonio observado total: USD 13,066,635


In [34]:
# El faltante es aleatorio? Se compara contra el patrimonio observado

# Los códigos y los ceros ya pasaron a faltante: falta lo que está vacío
falta_patrimonio = encuesta["patrimonio_invertible_usd"].isna()
falta_ingreso = encuesta["ingreso_mensual_liquido"].isna()

comparacion = pd.DataFrame(
    {
        "id_cliente": encuesta["id_cliente"],
        "observado_usd": encuesta["id_cliente"].map(patrimonio_observado),
        "falta_patrimonio": falta_patrimonio,
        "falta_ingreso": falta_ingreso,
    }
)

for columna, etiqueta in [
    ("falta_patrimonio", "patrimonio"),
    ("falta_ingreso", "ingreso"),
]:
    resumen = (
        comparacion.groupby(columna)["observado_usd"]
        .agg(
            clientes="count",
            media="mean",
            mediana="median",
            p75=lambda s: s.quantile(0.75),
        )
        .rename(
            index={False: f"declaró su {etiqueta}", True: f"NO declaró su {etiqueta}"}
        )
    )
    print(f"\n=== Patrimonio OBSERVADO (USD), según si declaró su {etiqueta} ===")
    display(resumen)
    brecha = (
        resumen.loc[f"NO declaró su {etiqueta}", "mediana"]
        / resumen.loc[f"declaró su {etiqueta}", "mediana"]
        - 1
    )
    print(
        f"El que no declara su {etiqueta} tiene una cartera real {brecha:+.0%} "
        f"más grande (en mediana) que el que sí declara."
    )
    # Mann-Whitney: compara las dos distribuciones sin suponer normalidad, que acá no
    # hay (los montos tienen cola larga a la derecha)
    con_dato = comparacion.dropna(subset=["observado_usd"])
    _, p_valor = stats.mannwhitneyu(
        con_dato.loc[con_dato[columna], "observado_usd"],
        con_dato.loc[~con_dato[columna], "observado_usd"],
    )
    print(f"p = {p_valor:.4f}")


=== Patrimonio OBSERVADO (USD), según si declaró su patrimonio ===


,clientes,media,mediana,p75
falta_patrimonio,,,,
declaró su patrimonio,582,"19,310.91","12,233.87","23,749.32"
NO declaró su patrimonio,51,"31,521.80","21,508.95","33,468.60"


El que no declara su patrimonio tiene una cartera real +76% más grande (en mediana) que el que sí declara.


p = 0.0010

=== Patrimonio OBSERVADO (USD), según si declaró su ingreso ===


,clientes,media,mediana,p75
falta_ingreso,,,,
declaró su ingreso,571,"19,679.09","12,486.01","24,313.59"
NO declaró su ingreso,62,"25,964.56","15,789.95","30,544.68"


El que no declara su ingreso tiene una cartera real +26% más grande (en mediana) que el que sí declara.
p = 0.1063


In [35]:
# cuánto tienen invertido los que contestan y los que no

largo = pd.concat(
    [
        comparacion.assign(
            dato="Patrimonio",
            respondio=np.where(
                comparacion["falta_patrimonio"], "No contestó", "Contestó"
            ),
        ),
        comparacion.assign(
            dato="Ingreso",
            respondio=np.where(comparacion["falta_ingreso"], "No contestó", "Contestó"),
        ),
    ]
).dropna(subset=["observado_usd"])

resumen_respuesta = largo.groupby(["respondio", "dato"])["observado_usd"].describe()

figura = go.Figure()
for respuesta, color in [
    ("Contestó", COLOR_PERFIL["Conservador"]),
    ("No contestó", COLOR_PERFIL["Moderado"]),
]:
    filas = resumen_respuesta.loc[respuesta].reindex(["Patrimonio", "Ingreso"])
    figura.add_trace(
        go.Bar(
            name=respuesta,
            x=filas.index,
            y=filas["50%"],
            marker_color=color,
            text=[f"USD {formato_es(v)}" for v in filas["50%"]],
            textposition="outside",
            customdata=np.column_stack([filas["count"], filas["25%"], filas["75%"]]),
            hovertemplate=(
                "<b>Pregunta: %{x} · " + respuesta.lower() + "</b><br>"
                "%{customdata[0]:.0f} clientes<br>"
                "Cartera mediana: USD %{y:,.0f}<br>"
                "Rango intercuartílico: USD %{customdata[1]:,.0f} "
                "y %{customdata[2]:,.0f}<extra></extra>"
            ),
        )
    )
figura.update_layout(
    barmode="group",
    bargap=0.35,
    legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"),
)
figura.update_yaxes(title="Cartera mediana (USD)", tickformat=",.0f")
figura.update_xaxes(title="Variable no declarada", showgrid=False)
presentar(
    figura,
    "Quienes no declaran patrimonio tienen carteras más grandes: el faltante no es "
    "aleatorio",
    "Cartera reconstruida desde el registro de operaciones · al 30/06/2026",
    interactivo=False,
    alto=440,
)

Los que no declaran su patrimonio tienen una cartera real 76 % más grande, en mediana, que los que sí lo declaran (Mann-Whitney, p = 0,001). Con el ingreso pasa lo mismo (+26 %), pero no es significativa (p = 0,11).

El que no contesta su patrimonio es el cliente con más plata. En la clasificación de Rubin (1976), si el faltante depende del propio valor que falta es MNAR (missing and not random). Pero acá hay una variable que lo anticipa: lo que el cliente operó en la firma. Si, a igual monto operado, el que no contesta tiene el mismo patrimonio que el que sí, el faltante es MAR (missing and random) y se puede corregir. 

Borrar esas filas saca justo a los clientes de mayor patrimonio, y completar con la mediana les pone el patrimonio del cliente típico. Estimo el patrimonio faltante a partir de lo operado, que existe para el 99 % de los clientes, y marco a esos clientes para poder repetir cualquier análisis sin ellos.

El patrimonio falta en los clientes con más plata, así que lo estimo a partir de lo que operaron. Los que sí declaran tienen más de lo que operaron en la firma, porque tienen plata en otros lados, y la diferencia es mayor en las carteras chicas. En el cuartil más chico de lo operado, lo declarado es 1,78 veces lo operado; en el más grande, 1,29. A cada cliente sin dato le aplico la relación de su cuartil y lo marco como imputado.

El ingreso queda como faltante. Las operaciones dicen cuánto invirtió el cliente, no cuánto gana, así que no hay otra fuente, y la mediana aplastaría la relación entre ingreso y riesgo 

In [36]:
# Imputar el patrimonio faltante con lo operado en la firma

encuesta["patrimonio_observado_usd"] = encuesta["id_cliente"].map(
    (movimientos["monto"] * movimientos["moneda"].map(FACTOR_A_USD))
    .groupby(movimientos["id_cliente"])
    .sum()
)

# Imputar lo operado tal cual subestimaría justo a los que no contestan, así que
# se escala por la relación mediana entre declarado y operado. Esa relación baja a
# medida que la cartera crece, así que se calcula por cuartil de lo operado
declaran = encuesta["patrimonio_invertible_usd"].notna() & encuesta[
    "patrimonio_observado_usd"
].gt(0)
cuartil_operado = pd.qcut(encuesta["patrimonio_observado_usd"], 4, labels=False)
relacion = encuesta["patrimonio_invertible_usd"] / encuesta["patrimonio_observado_usd"]
FACTOR_POR_CUARTIL = relacion[declaran].groupby(cuartil_operado[declaran]).median()
correlacion = encuesta.loc[declaran, "patrimonio_invertible_usd"].corr(
    encuesta.loc[declaran, "patrimonio_observado_usd"]
)
print(
    f"Entre los {int(declaran.sum())} que declaran: correlación declarado-operado "
    f"{correlacion:.2f}; relación mediana {relacion[declaran].median():.2f}"
)
print("Relación mediana declarado / operado, por cuartil de lo operado:")
print(FACTOR_POR_CUARTIL.rename(lambda q: f"cuartil {q + 1}").round(2).to_string())

falta = encuesta["patrimonio_invertible_usd"].isna()
puede_imputarse = falta & encuesta["patrimonio_observado_usd"].notna()
mediana_antes = encuesta["patrimonio_invertible_usd"].median()

encuesta["patrimonio_fue_imputado"] = puede_imputarse
encuesta["patrimonio_invertible_usd"] = encuesta["patrimonio_invertible_usd"].fillna(
    encuesta["patrimonio_observado_usd"] * cuartil_operado.map(FACTOR_POR_CUARTIL)
)

print(
    f"\nPatrimonio imputado: {int(puede_imputarse.sum())} "
    f"({puede_imputarse.mean():.1%})"
)
print(
    f"Patrimonio invertible mediano: USD {mediana_antes:,.0f} sin los faltantes · "
    f"USD {encuesta['patrimonio_invertible_usd'].median():,.0f} con la imputación"
)
print(
    f"Patrimonio todavía faltante: "
    f"{int(encuesta['patrimonio_invertible_usd'].isna().sum())} "
    "(clientes sin operaciones registradas)"
)

faltan_ingreso = int(encuesta["ingreso_mensual_liquido"].isna().sum())
encuesta["ingreso_fue_declarado"] = encuesta["ingreso_mensual_liquido"].notna()
print(
    f"Ingreso que queda faltante: {faltan_ingreso} "
    f"({faltan_ingreso / len(encuesta):.1%})"
)

Entre los 582 que declaran: correlación declarado-operado 0.97; relación mediana 1.43
Relación mediana declarado / operado, por cuartil de lo operado:
patrimonio_observado_usd
cuartil 1.0   1.78
cuartil 2.0   1.43
cuartil 3.0   1.33
cuartil 4.0   1.29

Patrimonio imputado: 51 (8.0%)
Patrimonio invertible mediano: USD 16,800 sin los faltantes · USD 17,650 con la imputación
Patrimonio todavía faltante: 0 (clientes sin operaciones registradas)
Ingreso que queda faltante: 62 (9.7%)


## Clientes que no cruzan entre tablas

In [37]:
ids_encuesta = set(encuesta["id_cliente"])
ids_movimientos = set(movimientos["id_cliente"])

sin_encuesta = sorted(ids_movimientos - ids_encuesta)
sin_operaciones = sorted(ids_encuesta - ids_movimientos)

print(f"Clientes con operaciones pero sin encuesta: {len(sin_encuesta)}")
print(f"Clientes con encuesta pero sin operaciones: {len(sin_operaciones)}")

ops_huerfanas = movimientos[movimientos["id_cliente"].isin(sin_encuesta)]
monto_huerfano = (
    ops_huerfanas["monto"] * ops_huerfanas["moneda"].map(FACTOR_A_USD)
).sum()

print(
    f"\nEsos {len(sin_encuesta)} clientes representan {len(ops_huerfanas)} operaciones "
    f"({len(ops_huerfanas) / len(movimientos):.1%} del registro) y "
    f"USD {monto_huerfano:,.0f} ({monto_huerfano / monto_usd.sum():.1%} del monto)."
)

# Los identificadores pertenecen a otro bloque de numeración?
print(
    f"\nRango de id_cliente en la encuesta: {min(ids_encuesta)} a {max(ids_encuesta)}"
)
print(f"Rango de id_cliente sin encuesta: {sin_encuesta[0]} a {sin_encuesta[-1]}")
print(
    f"Operaciones por cliente — con encuesta: "
    f"{len(movimientos) / len(ids_encuesta & ids_movimientos):.1f} | "
    f"sin encuesta: {len(ops_huerfanas) / len(sin_encuesta):.1f}"
)

Clientes con operaciones pero sin encuesta: 72
Clientes con encuesta pero sin operaciones: 7

Esos 72 clientes representan 73 operaciones (1.7% del registro) y USD 220,072 (1.7% del monto).

Rango de id_cliente en la encuesta: 20001 a 20640
Rango de id_cliente sin encuesta: 80015 a 80890
Operaciones por cliente — con encuesta: 6.8 | sin encuesta: 1.0


Los 72 clientes que operan sin encuesta quedan fuera del cruce: no hay perfil declarado con qué comparar. Sus números son de otro rango y tienen una operación en promedio contra siete del resto, así que parecen venir de otro sistema

Los 7 que tienen encuesta pero nunca operaron quedan en la base como clientes sin cartera, no entran al riesgo real.

In [38]:
# Los clientes de la encuesta que nunca operaron

display(
    encuesta[encuesta["id_cliente"].isin(sin_operaciones)][
        [
            "id_cliente",
            "edad",
            "ocupacion",
            "patrimonio_invertible_usd",
            "perfil_autodeclarado",
            "anios_experiencia_invirtiendo",
        ]
    ]
)

,id_cliente,edad,ocupacion,patrimonio_invertible_usd,perfil_autodeclarado,anios_experiencia_invirtiendo
58,20306,54.00,Empleado en relación de dependencia,"5,100.00",Moderado,6.00
113,20281,32.00,Empleado en relación de dependencia,"1,500.00",Moderado,5.00
358,20367,38.00,Estudiante,"1,800.00",Agresivo,8.00
379,20544,56.00,Profesional independiente,"3,100.00",Agresivo,11.00
448,20228,46.00,Empleado público,"4,100.00",Conservador,5.00
546,20183,48.00,Estudiante,"3,400.00",Agresivo,4.00
551,20096,43.00,Profesional independiente,"5,200.00",Agresivo,2.00


## Controles de la base limpia

In [39]:
resumen_etapa = pd.DataFrame(
    [
        {
            "tabla": "encuesta",
            "concepto": "filas",
            "antes": len(encuesta_cruda),
            "despues": len(encuesta),
        },
        {
            "tabla": "encuesta",
            "concepto": "clientes únicos",
            "antes": encuesta_cruda["id_cliente"].nunique(),
            "despues": encuesta["id_cliente"].nunique(),
        },
        {
            "tabla": "encuesta",
            "concepto": "faltantes totales en celdas",
            "antes": int(encuesta_cruda.isna().sum().sum()),
            "despues": int(encuesta.isna().sum().sum()),
        },
        {
            "tabla": "movimientos",
            "concepto": "operaciones",
            "antes": len(movimientos_crudos),
            "despues": len(movimientos),
        },
        {
            "tabla": "movimientos",
            "concepto": "clientes con operaciones",
            "antes": movimientos_crudos["id_cliente"].nunique(),
            "despues": movimientos["id_cliente"].nunique(),
        },
    ]
)
resumen_etapa["variacion"] = resumen_etapa["despues"] - resumen_etapa["antes"]
resumen_etapa["pct"] = (
    resumen_etapa["variacion"] / resumen_etapa["antes"] * 100
).round(2)
display(resumen_etapa.set_index(["tabla", "concepto"]))


assert encuesta["id_cliente"].is_unique, "Una fila por cliente"
assert (
    encuesta["fecha_respuesta"].le(FECHA_CORTE).all()
), "Sin encuestas posteriores al corte"
assert movimientos["monto"].gt(0).all(), "Todos los montos positivos y presentes"
assert (
    movimientos["fecha_operacion"].le(FECHA_CORTE).all()
), "Sin operaciones posteriores al corte"
assert encuesta["perfil_autodeclarado"].nunique() == 3, "Tres perfiles autodeclarados"
print("\nTodos los controles de la base limpia pasaron.")

encuesta.to_parquet(SALIDAS / "encuesta_limpia.parquet", index=False)
movimientos.to_parquet(SALIDAS / "movimientos_limpios.parquet", index=False)
print(f"Bases limpias guardadas en {SALIDAS}")

antes  despues  variacion     pct
tabla       concepto                                                      
encuesta    filas                          657      640        -17   -2.59
            clientes únicos                640      640          0    0.00
            faltantes totales en celdas    129      123         -6   -4.65
movimientos operaciones                   4481     4335       -146   -3.26
            clientes con operaciones       706      705         -1 -0.1400


Todos los controles de la base limpia pasaron.


Bases limpias guardadas en C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\salidas


## Asimetría y atípicos

In [40]:
# De variables monetarias


def contar_atipicos(serie: pd.Series) -> pd.Series:
    """Centro, forma y atípicos de una serie según los tres criterios habituales."""
    valores = serie.dropna()
    q1, q3 = valores.quantile([0.25, 0.75])
    rango = q3 - q1
    puntajes = (valores - valores.mean()) / valores.std()
    bajo, alto = valores.quantile([0.01, 0.99])
    return pd.Series(
        {
            "casos": len(valores),
            "media": valores.mean(),
            "mediana": valores.median(),
            "media_sobre_mediana": valores.mean() / valores.median(),
            "asimetria": valores.skew(),
            "por_iqr": int(
                ((valores < q1 - 1.5 * rango) | (valores > q3 + 1.5 * rango)).sum()
            ),
            "por_3_desvios": int((puntajes.abs() > 3).sum()),
            "por_percentil_1_99": int(((valores < bajo) | (valores > alto)).sum()),
        }
    )


monto_operacion_usd = movimientos["monto"] * movimientos["moneda"].map(FACTOR_A_USD)
atipicos = pd.DataFrame(
    {
        "ingreso mensual (UYU)": contar_atipicos(encuesta["ingreso_mensual_liquido"]),
        "patrimonio invertible (USD)": contar_atipicos(
            encuesta["patrimonio_invertible_usd"]
        ),
        "monto por operación (USD)": contar_atipicos(monto_operacion_usd),
    }
).T
display(
    atipicos.style.format(
        {
            "casos": "{:.0f}",
            "media": "{:,.0f}",
            "mediana": "{:,.0f}",
            "media_sobre_mediana": "{:.2f}",
            "asimetria": "{:.2f}",
            "por_iqr": "{:.0f}",
            "por_3_desvios": "{:.0f}",
            "por_percentil_1_99": "{:.0f}",
        }
    )
)

print("¿Son reales? Los cinco patrimonios más altos, con sus otras columnas:")
display(
    encuesta.nlargest(5, "patrimonio_invertible_usd")[
        [
            "id_cliente",
            "patrimonio_invertible_usd",
            "patrimonio_observado_usd",
            "ingreso_mensual_liquido",
            "ocupacion",
            "edad",
        ]
    ]
)

,casos,media,mediana,media_sobre_mediana,asimetria,por_iqr,por_3_desvios,por_percentil_1_99
ingreso mensual (UYU),578,"89,407","75,900",1.18,2.39,23,10,6
patrimonio invertible (USD),640,"27,359","17,650",1.55,3.07,44,21,13
monto por operación (USD),4335,"3,014","1,344",2.24,7.15,438,76,86


¿Son reales? Los cinco patrimonios más altos, con sus otras columnas:


,id_cliente,patrimonio_invertible_usd,patrimonio_observado_usd,ingreso_mensual_liquido,ocupacion,edad
161,20009,"279,100.00","260,735.33","90,800.00",Empleado público,41.00
626,20136,"208,100.00","174,726.26",<NA>,Empresario / dueño de negocio,38.00
287,20393,"197,100.00","183,581.15","239,900.00",Empresario / dueño de negocio,66.00
144,20397,"171,900.00","164,470.45","158,200.00",Profesional independiente,59.00
520,20570,"171,100.00","149,366.09","151,200.00",Profesional independiente,54.00


Los patrimonios más altos no son errores: lo que esos clientes operaron en la firma está cerca de lo que declaran. Se quedan. La distribución tiene cola larga, la media del patrimonio es 1,6 veces la mediana, así que para describir al cliente típico uso la mediana.

## Resumen de decisiones de limpieza

- Saqué 17 formularios enviados dos veces (2,6 %). Ningún cliente repetido tenía respuestas distintas.
- Saqué 94 operaciones registradas dos veces (2,1 %). Si no, el riesgo de esos clientes se calculaba con un instrumento contado doble.
- Convertí aparte las fechas escritas con barras. La conversión automática daba vuelta día y mes y ponía 101 encuestas después del corte.
- Unifiqué las respuestas escritas de varias formas en seis variables, para no partir la misma categoría en dos.
- Pasé a faltante los centinelas, casos de −999 en ingreso y patrimonio, −1 en el test, "no contesto" y ceros en ingreso y patrimonio.
- Pasé a faltante edades fuera de rango, porque no hay forma de saber el valor real.
- A los 17 clientes sin fondo que dejaron vacíos los meses cubiertos les puse 0; los 14 que tienen fondo quedaron como faltantes.
- Las respuestas salteadas en reacción ante una caída y objetivo quedaron como "No contesta".
- Di vuelta el signo de montos negativos (0,4 %) y saqué operaciones sin monto (1,2 %). Ningún cliente perdió más de dos operaciones.
- Estimé el patrimonio de 51 clientes que no lo declararon (8 %) con lo operado por la relación de su cuartil. La mediana pasa de USD 16.800 a 17.650.
- El ingreso del 9,7% de los cliente queda faltante. Los clientes sin encuesta (1,7 % del monto) quedan fuera del cruce. Los patrimonios muy altos se quedan, porque las operaciones los confirman.

# Creación de variables

## Monedas

Sumar montos en monedas distintas no sirve. Un plazo fijo de $ 400.000 son unos USD 9.300, menos que un ETF de USD 10.000, pero sin convertir parece cuarenta veces más grande. Como el riesgo se pondera por monto, los instrumentos en pesos, que son los más conservadores, dominarían todo.

Convierto a dólares porque el patrimonio declarado ya viene en dólares, porque es la moneda de referencia del asesoramiento patrimonial en Uruguay y porque 8/14 instrumentos están en dólares. Uso el tipo de cambio y la UI del 30/06/2026, no los de cada operación, porque quiero comparar carteras entre clientes y no reconstruir el valor de cada compra.

In [41]:
INFLACION_UY = leer_tasa(r"Inflación anual Uruguay") / 100
INFLACION_US = leer_tasa(r"Inflación anual de Estados Unidos") / 100
TASA_POLITICA_UY = leer_tasa("política monetaria") / 100
TASA_TESORO_US = leer_tasa("Tesoro de EE") / 100

print(f"Tipo de cambio: {TIPO_CAMBIO:.2f} UYU/USD")
print(f"Unidad Indexada: {VALOR_UI:.2f} UYU/UI  (= USD {FACTOR_A_USD['UI']:.4f})")
print(f"Inflación Uruguay: {INFLACION_UY:.1%}")
print(f"Inflación EE.UU.: {INFLACION_US:.1%}")
print(f"Tasa política BCU: {TASA_POLITICA_UY:.2%}")
print(f"Bono Tesoro EE.UU. 10a: {TASA_TESORO_US:.2%}")

Tipo de cambio: 42.80 UYU/USD
Unidad Indexada: 6.55 UYU/UI  (= USD 0.1530)
Inflación Uruguay: 5.4%
Inflación EE.UU.: 2.7%
Tasa política BCU: 8.75%
Bono Tesoro EE.UU. 10a: 4.10%


In [42]:
movimientos_utiles = movimientos[
    movimientos["id_cliente"].isin(set(encuesta["id_cliente"]))
].copy()
movimientos_utiles["monto_usd"] = movimientos_utiles["monto"] * movimientos_utiles[
    "moneda"
].map(FACTOR_A_USD)

comparacion_monedas = movimientos_utiles.groupby("moneda").agg(
    operaciones=("monto", "size"),
    monto_sin_convertir=("monto", "sum"),
    monto_en_usd=("monto_usd", "sum"),
)
comparacion_monedas["peso_sin_convertir"] = (
    comparacion_monedas["monto_sin_convertir"]
    / comparacion_monedas["monto_sin_convertir"].sum()
)
comparacion_monedas["peso_real_usd"] = (
    comparacion_monedas["monto_en_usd"] / comparacion_monedas["monto_en_usd"].sum()
)
display(
    comparacion_monedas.style.format(
        {
            "monto_sin_convertir": "{:,.0f}",
            "monto_en_usd": "{:,.0f}",
            "peso_sin_convertir": "{:.1%}",
            "peso_real_usd": "{:.1%}",
        }
    )
)

print(
    f"Total de la cartera de la firma: USD {movimientos_utiles['monto_usd'].sum():,.0f}"
)

,operaciones,monto_sin_convertir,monto_en_usd,peso_sin_convertir,peso_real_usd
moneda,,,,,
UI,796,"14,866,403","2,275,115",6.5%,17.7%
USD,1605,"5,722,325","5,722,325",2.5%,44.5%
UYU,1861,"207,542,429","4,849,122",91.0%,37.7%


Total de la cartera de la firma: USD 12,846,563


Sin convertir, los pesos parecen el 91 % de la plata de la firma; en dólares son el 37,7 %

## ¿Los dólares en la caja de seguridad son cartera?

Los dólares en caja son riesgo 1 y rinden 0 %. Pesan mucho y bajan el riesgo medido de quien los tiene, así que importa si los cuento.

En planificación financiera se separan dos tipos de efectivo. La reserva de emergencia queda fuera de la cartera de inversión porque cumple otra función; la práctica habitual es tener entre tres y seis meses de gastos básicos (según la guía de educación financiera de BBVA). El efectivo que excede esa reserva sí es una decisión de inversión, con un costo: lo que deja de rendir (cash drag). Contar todo el efectivo haría ver conservador a quien simplemente no invierte; no contar nada borraría una decisión real.

Criterio: hasta tres meses del ingreso declarado es reserva y queda afuera; lo que sobra entra a la cartera con riesgo 1. Uso tres y no seis porque con más efectivo adentro los clientes parecen más conservadores: eso puede achicar la cantidad de sobreexpuestos, así que ese número es un piso. 
Para los 18 que no declararon ingreso uso su propia respuesta: si dicen tener fondo, el efectivo es ese fondo y queda afuera; si no, entra entero.

In [43]:
# Parámetro: meses de ingreso que se consideran reserva de emergencia
MESES_DE_RESERVA = 3

In [44]:
# Descomposición del efectivo en reserva de emergencia y excedente asignado

ID_EFECTIVO = 14  # Dólares en caja de seguridad

efectivo_por_cliente = (
    movimientos_utiles[movimientos_utiles["id_instrumento"] == ID_EFECTIVO]
    .groupby("id_cliente")["monto_usd"]
    .sum()
)

datos_cliente = encuesta.set_index("id_cliente")
reserva_requerida = (
    datos_cliente["ingreso_mensual_liquido"] * MESES_DE_RESERVA / TIPO_CAMBIO
)

efectivo = pd.DataFrame({"efectivo_usd": efectivo_por_cliente}).join(
    pd.DataFrame(
        {
            "reserva_requerida": reserva_requerida,
            "tiene_fondo": datos_cliente["tiene_fondo_emergencia"],
        }
    )
)

# Regla principal: la reserva es el menor entre lo que tiene y lo que necesita
efectivo["reserva"] = np.minimum(
    efectivo["efectivo_usd"], efectivo["reserva_requerida"]
)

# Respaldo para quien no declaró ingreso: se usa su propia respuesta sobre el fondo
sin_ingreso = efectivo["reserva_requerida"].isna()
efectivo.loc[sin_ingreso, "reserva"] = np.where(
    efectivo.loc[sin_ingreso, "tiene_fondo"] == "Sí",
    efectivo.loc[sin_ingreso, "efectivo_usd"],  # el efectivo ES el fondo declarado
    0.0,  # declaró no tener fondo: es asignación
)
efectivo["excedente"] = efectivo["efectivo_usd"] - efectivo["reserva"]

print(f"Clientes con dólares en caja de seguridad : {len(efectivo)}")
print(f"Efectivo total: USD {efectivo['efectivo_usd'].sum():,.0f}")
print(
    f"  reserva de emergencia (queda FUERA): USD {efectivo['reserva'].sum():,.0f} "
    f"({efectivo['reserva'].sum() / efectivo['efectivo_usd'].sum():.0%})"
)
print(
    f"  excedente (entra a la cartera): USD {efectivo['excedente'].sum():,.0f} "
    f"({efectivo['excedente'].sum() / efectivo['efectivo_usd'].sum():.0%})"
)
print(
    "\nClientes cuyo efectivo es todo reserva: "
    f"{int((efectivo['excedente'] <= 0.01).sum())}"
)
print(f"Clientes sin ingreso declarado (respaldo): {int(sin_ingreso.sum())}")

Clientes con dólares en caja de seguridad : 171
Efectivo total: USD 1,056,554
  reserva de emergencia (queda FUERA): USD 631,936 (60%)
  excedente (entra a la cartera): USD 424,618 (40%)

Clientes cuyo efectivo es todo reserva: 131
Clientes sin ingreso declarado (respaldo): 18


In [45]:
# quienes guardan efectivo y dicen no tener fondo de emergencia

contradicen = efectivo[efectivo["tiene_fondo"] == "No"]
print(
    f"Clientes que declaran NO tener fondo de emergencia pero guardan dólares en caja: "
    f"{len(contradicen)}"
)
print(f"  monto que acumulan : USD {contradicen['efectivo_usd'].sum():,.0f}")
print(f"  mediana por cliente: USD {contradicen['efectivo_usd'].median():,.0f}")
print(
    "\nTienen un fondo de emergencia; no lo reconocen como tal. Se retoma en la Etapa "
    "3 "
    "junto con el resto de las inconsistencias internas."
)

display(
    efectivo.groupby("tiene_fondo")
    .agg(
        clientes=("efectivo_usd", "size"),
        efectivo_total=("efectivo_usd", "sum"),
        mediana=("efectivo_usd", "median"),
        reserva=("reserva", "sum"),
        excedente=("excedente", "sum"),
    )
    .style.format(
        {
            "efectivo_total": "{:,.0f}",
            "mediana": "{:,.0f}",
            "reserva": "{:,.0f}",
            "excedente": "{:,.0f}",
        }
    )
)

Clientes que declaran NO tener fondo de emergencia pero guardan dólares en caja: 73
  monto que acumulan : USD 468,630
  mediana por cliente: USD 2,769

Tienen un fondo de emergencia; no lo reconocen como tal. Se retoma en la Etapa 3 junto con el resto de las inconsistencias internas.


,clientes,efectivo_total,mediana,reserva,excedente
tiene_fondo,,,,,
No,73,"468,630","2,769","195,977","272,654"
Sí,98,"587,924","3,233","435,959","151,964"


## Cartera efectiva

Lo que importa es dónde puso la plata el cliente. Con valor de mercado, los instrumentos más riesgosos, que rindieron más, se verían inflados sólo por el paso del tiempo. Uso todas las operaciones de 2022 a 2026: una compra de 2023 sigue siendo parte de la cartera.

In [46]:
# Reconstrucción de la cartera: cuánto tiene cada cliente y en qué instrumentos

# El efectivo entra sólo por su excedente. Se prorratea entre los movimientos de
# efectivo del cliente para no tener que inventar a cuál de ellos corresponde la
# reserva
proporcion_que_entra = (efectivo["excedente"] / efectivo["efectivo_usd"]).clip(0, 1)

movimientos_utiles["monto_cartera"] = movimientos_utiles["monto_usd"]
es_efectivo = movimientos_utiles["id_instrumento"] == ID_EFECTIVO
movimientos_utiles.loc[es_efectivo, "monto_cartera"] = movimientos_utiles.loc[
    es_efectivo, "monto_usd"
] * movimientos_utiles.loc[es_efectivo, "id_cliente"].map(proporcion_que_entra).fillna(
    1.0
)

posiciones = movimientos_utiles[movimientos_utiles["monto_cartera"] > 0].merge(
    catalogo, on="id_instrumento", how="left"
)

cartera = posiciones.groupby("id_cliente").agg(
    cartera_usd=("monto_cartera", "sum"),
    operaciones=("monto_cartera", "size"),
    instrumentos=("id_instrumento", "nunique"),
    primera_operacion=("fecha_operacion", "min"),
    ultima_operacion=("fecha_operacion", "max"),
)
cartera["meses_desde_ultima"] = (
    (FECHA_CORTE - cartera["ultima_operacion"]).dt.days / 30.44
).round(1)

print(f"Clientes con cartera de inversión reconstruida: {len(cartera)}")
display(
    cartera[["cartera_usd", "operaciones", "instrumentos", "meses_desde_ultima"]]
    .describe()
    .T
)

Clientes con cartera de inversión reconstruida: 628


,count,mean,std,min,25%,50%,75%,max
cartera_usd,628.00,"19,450.04","25,167.79",39.23,"5,266.13","11,663.60","23,429.70","254,370.84"
operaciones,628.00,6.27,3.47,1.00,4.00,6.00,8.00,22.00
instrumentos,628.00,2.53,1.18,1.00,2.00,2.00,3.00,7.00
meses_desde_ultima,628.00,8.76,8.06,1.00,3.10,6.00,12.00,45.60


Los clientes sin cartera, porque nunca operaron o porque todo su efectivo es reserva, quedan fuera del cruce entre declarado y real: no hay comportamiento que comparar. Quedan en la base como un grupo aparte.

In [47]:
# Los clientes que quedan sin cartera de inversión

sin_cartera = sorted(set(encuesta["id_cliente"]) - set(cartera.index))
todo_en_reserva = [c for c in sin_cartera if c in efectivo.index]

print(
    f"Clientes sin cartera de inversión: {len(sin_cartera)} de {len(encuesta)} "
    f"({len(sin_cartera) / len(encuesta):.1%})"
)
print(f"  nunca operaron: " f"{len(sin_cartera) - len(todo_en_reserva)}")
print(
    f"  sólo tienen efectivo, y todo es reserva de emergencia: {len(todo_en_reserva)}"
)

display(
    encuesta[encuesta["id_cliente"].isin(todo_en_reserva)][
        [
            "id_cliente",
            "edad",
            "ocupacion",
            "ingreso_mensual_liquido",
            "patrimonio_invertible_usd",
            "perfil_autodeclarado",
            "tiene_fondo_emergencia",
        ]
    ]
)

Clientes sin cartera de inversión: 12 de 640 (1.9%)
  nunca operaron: 7
  sólo tienen efectivo, y todo es reserva de emergencia: 5


,id_cliente,edad,ocupacion,ingreso_mensual_liquido,patrimonio_invertible_usd,perfil_autodeclarado,tiene_fondo_emergencia
60,20283,38.00,Empleado en relación de dependencia,"50,300.00","2,300.00",Moderado,Sí
78,20355,44.00,Estudiante,"24,400.00","2,800.00",Conservador,Sí
111,20262,39.00,Jubilado,"22,000.00","2,000.00",Moderado,Sí
338,20129,58.00,Profesional independiente,<NA>,"14,430.82",Conservador,Sí
360,20082,28.00,Empleado público,"22,000.00","1,029.42",Moderado,No


In [48]:
# Composición de la cartera de la firma por instrumento y por categoría de activo

composicion = posiciones.groupby(["categoria", "nombre_instrumento"]).agg(
    monto_usd=("monto_cartera", "sum"), clientes=("id_cliente", "nunique")
)
composicion["peso"] = composicion["monto_usd"] / composicion["monto_usd"].sum()
composicion = composicion.sort_values("monto_usd", ascending=False)
display(composicion.style.format({"monto_usd": "{:,.0f}", "peso": "{:.1%}"}))

por_categoria = (
    posiciones.groupby("categoria")["monto_cartera"].sum().sort_values(ascending=False)
)
print("Peso por familia de activo:")
print((por_categoria / por_categoria.sum()).map("{:.1%}".format).to_string())

Peso por familia de activo:
categoria
Renta fija local          33.8%
Renta fija corporativa    18.0%
Previsional               14.1%
Fondos                     9.5%
Materias primas            8.1%
Renta variable             7.6%
Efectivo                   3.5%
Inmobiliario               3.2%
Cripto                     2.3%


## Riesgo efectivo

El riesgo efectivo de cada cliente es el nivel de riesgo de sus instrumentos ponderado por cuánto tiene en cada uno:

$$\text{riesgo efectivo}_c = \frac{\sum_i \text{monto}_{c,i} \times \text{nivel de riesgo}_i}{\sum_i \text{monto}_{c,i}}$$

Es la forma estándar de pasar una característica de los activos a la cartera: el retorno esperado de una cartera es el promedio de los retornos ponderado por el peso de cada activo (Markowitz). Acá hago lo mismo con el nivel de riesgo. La escala de 1 a 5 es ordinal: promediarla supone que la distancia entre 1 y 2 es la misma que entre 4 y 5.

Uso la escala del catálogo porque es la que la firma usa con sus clientes; más adelante la comparo con una medida por volatilidad.

La concentración la mido con el índice de Herfindahl-Hirschman (HHI): la suma de los pesos de cada instrumento al cuadrado. Vale 1 si todo está en un solo instrumento y baja cuanto más repartida está la cartera.

In [49]:
# Riesgo efectivo y medidas complementarias de la cartera


def promedio_ponderado(grupo: pd.DataFrame, columna: str) -> float:
    """Promedio de `columna` ponderado por el monto invertido en cada posición."""
    return round(float(np.average(grupo[columna], weights=grupo["monto_cartera"])), 10)


def concentracion_herfindahl(grupo: pd.DataFrame) -> float:
    """Índice de Herfindahl sobre los pesos por instrumento.

    Vale 1 si toda la cartera está en un solo instrumento y tiende a 1/n si está
    repartida en partes iguales entre n instrumentos. Dos clientes con el mismo riesgo
    promedio no corren el mismo riesgo si uno está concentrado y el otro diversificado.
    """
    pesos = grupo.groupby("id_instrumento")["monto_cartera"].sum()
    return float(((pesos / pesos.sum()) ** 2).sum())


perfil_cartera = posiciones.groupby("id_cliente").apply(
    lambda g: pd.Series(
        {
            "riesgo_efectivo": promedio_ponderado(g, "nivel_riesgo"),
            "riesgo_simple": float(g["nivel_riesgo"].mean()),
            "volatilidad_cartera": promedio_ponderado(g, "volatilidad_anual"),
            "retorno_nominal_cartera": promedio_ponderado(g, "retorno_anual_historico"),
            "concentracion": concentracion_herfindahl(g),
        }
    ),
    include_groups=False,
)

display(perfil_cartera.describe().T)

,count,mean,std,min,25%,50%,75%,max
riesgo_efectivo,628.00,2.46,1.03,1.00,1.68,2.33,3.00,5.00
riesgo_simple,628.00,2.44,0.9936,1.00,1.67,2.33,3.00,5.00
volatilidad_cartera,628.00,0.0740,0.0972,0.00,0.0246,0.0466,0.0800,0.6530
retorno_nominal_cartera,628.00,0.0687,0.0314,0.00,0.0531,0.0660,0.0790,0.2480
concentracion,628.00,0.6095,0.2527,0.1803,0.3964,0.5457,0.8561,1.00


In [50]:
# Cuánto cambia ponderar por monto en lugar de promediar instrumentos?

diferencia = (perfil_cartera["riesgo_efectivo"] - perfil_cartera["riesgo_simple"]).abs()
print(
    f"Correlación entre ambas medidas: "
    f"{perfil_cartera['riesgo_efectivo'].corr(perfil_cartera['riesgo_simple']):.3f}"
)
print(
    f"Clientes que difieren en más de 0,5 puntos: {int((diferencia > 0.5).sum())} "
    f"({(diferencia > 0.5).mean():.1%})"
)
print(f"Diferencia máxima: {diferencia.max():.2f} puntos")

print("\nLos cinco casos donde más importa la ponderación:")
casos = diferencia.nlargest(5).index
detalle = perfil_cartera.loc[casos, ["riesgo_efectivo", "riesgo_simple"]].join(
    cartera[["cartera_usd", "instrumentos"]]
)
display(detalle)

print("\nCartera del cliente donde la diferencia es mayor:")
peor = diferencia.idxmax()
display(
    posiciones[posiciones["id_cliente"] == peor]
    .groupby("nombre_instrumento")
    .agg(monto_usd=("monto_cartera", "sum"), nivel_riesgo=("nivel_riesgo", "first"))
    .assign(peso=lambda d: d["monto_usd"] / d["monto_usd"].sum())
    .sort_values("monto_usd", ascending=False)
    .style.format({"monto_usd": "{:,.0f}", "peso": "{:.1%}"})
)

Correlación entre ambas medidas: 0.929
Clientes que difieren en más de 0,5 puntos: 100 (15.9%)
Diferencia máxima: 1.87 puntos

Los cinco casos donde más importa la ponderación:


,riesgo_efectivo,riesgo_simple,cartera_usd,instrumentos
id_cliente,,,,
20145,1.13,3.00,"22,991.89",2
20302,4.73,2.88,"34,426.38",3
20611,1.06,2.71,"112,105.88",2
20446,1.02,2.60,"16,932.71",2
20104,1.45,2.83,"39,010.00",3



Cartera del cliente donde la diferencia es mayor:


,monto_usd,nivel_riesgo,peso
nombre_instrumento,,,
Dólares en caja de seguridad,"22,271",1,96.9%
Criptomonedas,721,5,3.1%


## Cortes para las tres categorías

Para partir la medida de 1 a 5 en conservador, moderado y agresivo calibro los cortes contra el SRRI.

SRRI: Synthetic Risk and Reward Indicator (indicador sintético de riesgo y remuneración) es la escala de riesgo de 1 a 7 que la Unión Europea exigía mostrar en el documento de datos fundamentales para el inversor (KIID) de cada fondo de inversión UCITS. 

Cómo se calcula? con la volatilidad de los retornos semanales del fondo en los últimos cinco años, llevada a base anual. Según esa volatilidad, el fondo cae en una de siete clases: la 1 va hasta 0,5 %, la 2 hasta 2 %, la 3 hasta 5 %, la 4 hasta 10 %, la 5 hasta 15 %, la 6 hasta 25 % y la 7 es todo lo que está por encima. 

La guía no dice cómo pasar de siete clases a tres perfiles, así que por convención propia, agrupo las clases 1 a 3 como conservador, la 4 como moderado y las 5 a 7 como agresivo. Equivale a menos de 5 %, de 5 % a 10 % y más de 10 % de volatilidad. Con la volatilidad de cada instrumento calculo la de cada cartera y busco qué cortes de la escala de la firma dan la misma clasificación.

In [51]:
# Calibración de los cortes contra la partición SRRI

# CESR/10-673: cortes de volatilidad anualizada que separan las siete clases SRRI
CORTES_SRRI = [0, 0.005, 0.02, 0.05, 0.10, 0.15, 0.25, np.inf]

catalogo_srri = catalogo.assign(
    srri=pd.cut(
        catalogo["volatilidad_anual"],
        bins=CORTES_SRRI,
        labels=[1, 2, 3, 4, 5, 6, 7],
        right=False,
    ).astype(int)
)

print("Escala interna de Rambla contra la clase SRRI de cada instrumento:")
display(
    catalogo_srri[
        ["nombre_instrumento", "moneda", "volatilidad_anual", "nivel_riesgo", "srri"]
    ]
    .assign(brecha=lambda d: d["srri"] - d["nivel_riesgo"])
    .sort_values("volatilidad_anual")
    .style.format({"volatilidad_anual": "{:.1%}"})
)

print(
    f"\nCorrelación entre la escala de la firma y SRRI: "
    f"{catalogo_srri['nivel_riesgo'].corr(catalogo_srri['srri']):.3f}"
)

Escala interna de Rambla contra la clase SRRI de cada instrumento:


,nombre_instrumento,moneda,volatilidad_anual,nivel_riesgo,srri,brecha
13,Dólares en caja de seguridad,USD,0.0%,1,1,0
1,Plazo fijo en dólares,USD,0.4%,1,1,0
0,Plazo fijo en pesos (BROU),UYU,0.5%,1,2,1
2,Letras de Regulación Monetaria (BCU),UYU,1.3%,1,2,1
3,Notas del Tesoro en Unidades Indexadas,UI,2.6%,2,3,1
4,Fondo de inversión local (AFISA),UYU,4.1%,2,3,1
5,Obligaciones negociables (BVM),USD,5.8%,3,4,1
7,Ahorro voluntario en AFAP,UYU,6.1%,3,4,1
6,Fideicomiso financiero,UI,7.2%,3,4,1
10,Inmueble para renta,USD,8.9%,3,4,1



Correlación entre la escala de la firma y SRRI: 0.975


La escala de la firma correlaciona 0,975 con el SRRI, así que usarla está bien. Pero le da el mismo 5 a las acciones (24,3 % de volatilidad) y a la cripto (65,3 %), que el SRRI pone en las clases 6 y 7: una cartera toda en cripto y otra toda en acciones dan 5,0 las dos. 

In [52]:
# Parámetro: cortes de volatilidad del SRRI agrupado en tres perfiles
CORTES_VOLATILIDAD_PERFIL = [0, 0.05, 0.10, np.inf]  # hasta 5 %, de 5 % a 10 %, más

In [53]:
ETIQUETAS_PERFIL = ["Conservador", "Moderado", "Agresivo"]

# Partición de referencia, por volatilidad de la cartera (SRRI 1-3 / 4 / 5-7)
perfil_por_volatilidad = pd.cut(
    perfil_cartera["volatilidad_cartera"],
    bins=CORTES_VOLATILIDAD_PERFIL,
    labels=ETIQUETAS_PERFIL,
    right=False,
)

print("Reparto según la partición SRRI por volatilidad de cartera:")
print(perfil_por_volatilidad.value_counts().reindex(ETIQUETAS_PERFIL).to_string())

# A qué riesgo de la escala interna equivalen los límites de 5 % y 10 % de volatilidad?
print("\nEquivalencia empírica de los límites:")
for limite in CORTES_VOLATILIDAD_PERFIL[1:3]:
    cerca = perfil_cartera[
        (perfil_cartera["volatilidad_cartera"] - limite).abs() < 0.004
    ]
    print(
        f"  clientes con volatilidad proxima a {limite:.0%}: n={len(cerca)} | "
        f"riesgo mediano en la escala de la firma = "
        f"{cerca['riesgo_efectivo'].median():.2f}"
    )

# Búsqueda del par de cortes que mejor reproduce la partición de referencia
busqueda = []
for corte_bajo in np.arange(1.8, 3.01, 0.1):
    for corte_alto in np.arange(3.0, 4.21, 0.1):
        if corte_alto <= corte_bajo:
            continue
        propuesto = pd.cut(
            perfil_cartera["riesgo_efectivo"],
            bins=[0, corte_bajo, corte_alto, 5.01],
            labels=ETIQUETAS_PERFIL,
            right=False,
        )
        acuerdo = (propuesto.astype(str) == perfil_por_volatilidad.astype(str)).mean()
        busqueda.append(
            {
                "corte_bajo": round(corte_bajo, 1),
                "corte_alto": round(corte_alto, 1),
                "acuerdo": acuerdo,
            }
        )

busqueda = pd.DataFrame(busqueda).sort_values("acuerdo", ascending=False)
print("\nLos cinco pares de cortes con mayor acuerdo contra la partición SRRI:")
display(busqueda.head(5).style.format({"acuerdo": "{:.1%}"}))

Reparto según la partición SRRI por volatilidad de cartera:


volatilidad_cartera
Conservador    336
Moderado       172
Agresivo       120

Equivalencia empírica de los límites:
  clientes con volatilidad proxima a 5%: n=42 | riesgo mediano en la escala de la firma = 2.51
  clientes con volatilidad proxima a 10%: n=11 | riesgo mediano en la escala de la firma = 3.33



Los cinco pares de cortes con mayor acuerdo contra la partición SRRI:


,corte_bajo,corte_alto,acuerdo
108,2.600000,3.400000,93.6%
95,2.500000,3.400000,93.6%
96,2.500000,3.500000,93.3%
109,2.600000,3.500000,93.3%
107,2.600000,3.300000,93.0%


Me quedo con 2,5 y 3,5: dan casi la misma clasificación que el SRRI (93,3 %, a 0,3 puntos del mejor par de la búsqueda).

In [54]:
# Parámetros: cortes elegidos para la escala de riesgo de 1 a 5
CORTE_CONSERVADOR_MODERADO = 2.5
CORTE_MODERADO_AGRESIVO = 3.5

In [55]:
CORTES_PERFIL = [0, CORTE_CONSERVADOR_MODERADO, CORTE_MODERADO_AGRESIVO, 5.01]

perfil_cartera["perfil_real"] = pd.cut(
    perfil_cartera["riesgo_efectivo"],
    bins=CORTES_PERFIL,
    labels=ETIQUETAS_PERFIL,
    right=False,
)

acuerdo_adoptado = (
    perfil_cartera["perfil_real"].astype(str) == perfil_por_volatilidad.astype(str)
).mean()
mejor = busqueda.iloc[0]
print(
    f"Cortes adoptados {CORTE_CONSERVADOR_MODERADO} / {CORTE_MODERADO_AGRESIVO}: "
    f"acuerdo con la partición SRRI = {acuerdo_adoptado:.1%}"
)
print(
    f"Óptimo de la búsqueda {mejor['corte_bajo']} / {mejor['corte_alto']}: "
    f"{mejor['acuerdo']:.1%}  (diferencia: {mejor['acuerdo'] - acuerdo_adoptado:+.1%})"
)

print("\nReparto de clientes con los cortes adoptados:")
reparto = perfil_cartera["perfil_real"].value_counts().reindex(ETIQUETAS_PERFIL)
display(
    pd.DataFrame(
        {"clientes": reparto, "porcentaje": reparto / reparto.sum()}
    ).style.format({"porcentaje": "{:.1%}"})
)

Cortes adoptados 2.5 / 3.5: acuerdo con la partición SRRI = 93.3%
Óptimo de la búsqueda 2.6 / 3.4: 93.6%  (diferencia: +0.3%)

Reparto de clientes con los cortes adoptados:


,clientes,porcentaje
perfil_real,,
Conservador,336,53.5%
Moderado,191,30.4%
Agresivo,101,16.1%


In [56]:
# qué pasa si se mueven los cortes?

sensibilidad = []
for corte_bajo in (2.0, 2.2, 2.5, 2.8):
    for corte_alto in (3.0, 3.2, 3.5, 3.8):
        if corte_alto <= corte_bajo:
            continue
        propuesto = pd.cut(
            perfil_cartera["riesgo_efectivo"],
            bins=[0, corte_bajo, corte_alto, 5.01],
            labels=ETIQUETAS_PERFIL,
            right=False,
        )
        conteo = propuesto.value_counts().reindex(ETIQUETAS_PERFIL)
        sensibilidad.append(
            {
                "corte_bajo": corte_bajo,
                "corte_alto": corte_alto,
                "conservadores": int(conteo.iloc[0]),
                "moderados": int(conteo.iloc[1]),
                "agresivos": int(conteo.iloc[2]),
                "acuerdo_srri": (
                    propuesto.astype(str) == perfil_por_volatilidad.astype(str)
                ).mean(),
            }
        )

sensibilidad = pd.DataFrame(sensibilidad)
matriz_acuerdo = sensibilidad.pivot(
    index="corte_bajo", columns="corte_alto", values="acuerdo_srri"
)

display(
    matriz_acuerdo.rename_axis(index="corte bajo", columns="corte alto")
    .style.format("{:.1%}")
    .highlight_max(axis=None, color="#d6e6f8")
)

# El corte bajo sólo mueve a los conservadores y el alto sólo a los agresivos
print("Conservadores según el corte bajo:")
print(sensibilidad.groupby("corte_bajo")["conservadores"].first().to_string())
print("\nAgresivos según el corte alto:")
print(sensibilidad.groupby("corte_alto")["agresivos"].first().to_string())

corte alto,3.000000,3.200000,3.500000,3.800000
corte bajo,,,,
2.000000,65.4%,78.0%,79.1%,74.7%
2.200000,74.2%,86.8%,87.9%,83.4%
2.500000,79.6%,92.2%,93.3%,88.9%
2.800000,76.4%,89.0%,90.1%,85.7%


Conservadores según el corte bajo:
corte_bajo
2.00    226
2.20    283
2.50    336
2.80    381

Agresivos según el corte alto:
corte_alto
3.00    221
3.20    138
3.50    101
3.80     73


En la grilla, 2,5 y 3,5 es la combinación que más coincide con el SRRI (93,3 %). Cada corte mueve una sola punta: el bajo cambia cuántos conservadores hay (de 226 a 381) y el alto cuántos agresivos (de 221 a 73).

## Medida alternativa por volatilidad

Clasifico cada cartera por su volatilidad con los cortes del SRRI y la comparo con la medida principal. Los clientes que cambian de categoría son los que la escala de la firma mide mal.

In [57]:
# Comparación de las dos medidas de riesgo

perfil_cartera["perfil_por_volatilidad"] = perfil_por_volatilidad

cruce_medidas = pd.crosstab(
    perfil_cartera["perfil_real"],
    perfil_cartera["perfil_por_volatilidad"],
    rownames=["escala de la firma"],
    colnames=["volatilidad (SRRI)"],
)
display(cruce_medidas.reindex(index=ETIQUETAS_PERFIL, columns=ETIQUETAS_PERFIL))

cambian = perfil_cartera["perfil_real"].astype(str) != perfil_cartera[
    "perfil_por_volatilidad"
].astype(str)
print(
    f"Clientes clasificados distinto por las dos medidas: {int(cambian.sum())} "
    f"({cambian.mean():.1%})"
)

# Quiénes son? Se mira cuánta cripto tienen
peso_cripto = (
    posiciones[posiciones["categoria"] == "Cripto"]
    .groupby("id_cliente")["monto_cartera"]
    .sum()
) / cartera["cartera_usd"]
perfil_cartera["peso_cripto"] = peso_cripto.reindex(perfil_cartera.index).fillna(0)

print("\nPeso medio de cripto en la cartera:")
print(
    f"  clientes que las dos medidas clasifican igual: "
    f"{perfil_cartera.loc[~cambian, 'peso_cripto'].mean():.1%}"
)
print(
    f"  clientes que cambian de categoría: "
    f"{perfil_cartera.loc[cambian, 'peso_cripto'].mean():.1%}"
)

subestimados = perfil_cartera[
    (perfil_cartera["perfil_por_volatilidad"] == "Agresivo")
    & (perfil_cartera["perfil_real"] != "Agresivo")
]
print(
    f"\nClientes que el estándar llama agresivos y la escala interna NO: "
    f"{len(subestimados)}"
)
print(
    f"  peso medio de cripto en sus carteras: {subestimados['peso_cripto'].mean():.1%}"
)

volatilidad (SRRI),Conservador,Moderado,Agresivo
escala de la firma,,,
Conservador,324,11,1
Moderado,12,161,18
Agresivo,0,0,101


Clientes clasificados distinto por las dos medidas: 42 (6.7%)

Peso medio de cripto en la cartera:
  clientes que las dos medidas clasifican igual: 2.8%
  clientes que cambian de categoría: 4.9%

Clientes que el estándar llama agresivos y la escala interna NO: 19
  peso medio de cripto en sus carteras: 10.8%


Cambian 42 clientes, que tienen más cripto que el resto (4,9 % contra 2,8 % en promedio). La mayoría cambia hacia arriba: el estándar los ve más riesgosos que la escala de la empresa. La escala de 1 a 5 no tiene lugar para un activo con 65 % de volatilidad, así que subestima el riesgo de las carteras con cripto

## Retorno real

Un retorno nominal en pesos no se compara con uno en dólares: cada uno tiene que descontar la inflación de su moneda.

$$r_{real} = \frac{1 + r_{nominal}}{1 + \pi_{moneda}} - 1$$

Es la ecuación de Fisher, en lugar de la aproximación de restar la inflación al retorno. Uso 5,4 % de inflación para los pesos y 2,7 % para los dólares, que son las del archivo de tasas. Los instrumentos en UI no los ajusto porque ya están indexados. El resultado es un retorno real en la moneda de cada instrumento, así que el promedio de una cartera mezcla poder de compra en pesos y en dólares.

Por eso también lo paso todo a poder de compra en pesos, para alguien que gasta en Uruguay, un activo en dólares depende además del tipo de cambio. Como no tengo una expectativa de devaluación, uso la diferencia de tasas de interés entre monedas, anticipa cuánto se deprecia la de tasa más alta. 

Comparo la tasa de política monetaria del BCU con el bono del Tesoro de EE. UU. a diez años, las dos tasas que trae el caso que da 4,5 % de devaluación por año. Eso es más que la diferencia de inflaciones (2,6 %), así que supone que el peso se abarata en términos reales y favorece a los activos en dólares. Como los plazos tampoco coinciden, la uso sólo como sensibilidad.

In [58]:
# Retorno real de cada instrumento, en las dos versiones

# Devaluación implícita en la paridad de tasas de interés
DEVALUACION_IMPLICITA = (1 + TASA_POLITICA_UY) / (1 + TASA_TESORO_US) - 1


def retorno_real_moneda_propia(fila: pd.Series) -> float:
    """Deflacta el retorno por la inflación de la moneda del instrumento.

    La UI ya está indexada a la inflación uruguaya, así que su retorno es real por
    construcción y no se deflacta de nuevo.
    """
    nominal = fila["retorno_anual_historico"]
    if fila["moneda"] == "UI":
        return nominal
    inflacion = INFLACION_UY if fila["moneda"] == "UYU" else INFLACION_US
    return (1 + nominal) / (1 + inflacion) - 1


def retorno_real_en_pesos(fila: pd.Series) -> float:
    """Lleva todo a poder de compra uruguayo, con la devaluación esperada del peso."""
    nominal = fila["retorno_anual_historico"]
    if fila["moneda"] == "USD":
        nominal = (1 + nominal) * (1 + DEVALUACION_IMPLICITA) - 1
    elif fila["moneda"] == "UI":
        nominal = (1 + nominal) * (1 + INFLACION_UY) - 1  # la UI se re-expresa en pesos
    return (1 + nominal) / (1 + INFLACION_UY) - 1


catalogo_real = catalogo.copy()
catalogo_real["retorno_real"] = catalogo_real.apply(retorno_real_moneda_propia, axis=1)
catalogo_real["retorno_real_pesos"] = catalogo_real.apply(retorno_real_en_pesos, axis=1)
catalogo_real["retorno_por_unidad_riesgo"] = catalogo_real[
    "retorno_real"
] / catalogo_real["volatilidad_anual"].replace(0, np.nan)

print(
    f"Devaluación implícita en la paridad de tasas: {DEVALUACION_IMPLICITA:.2%} anual\n"
)
display(
    catalogo_real[
        [
            "nombre_instrumento",
            "moneda",
            "retorno_anual_historico",
            "retorno_real",
            "retorno_real_pesos",
            "volatilidad_anual",
            "nivel_riesgo",
            "liquidez",
            "retorno_por_unidad_riesgo",
        ]
    ]
    .sort_values("retorno_real", ascending=False)
    .style.format(
        {
            "retorno_anual_historico": "{:.1%}",
            "retorno_real": "{:.1%}",
            "retorno_real_pesos": "{:.1%}",
            "volatilidad_anual": "{:.1%}",
            "retorno_por_unidad_riesgo": "{:.2f}",
        }
    )
)

Devaluación implícita en la paridad de tasas: 4.47% anual



,nombre_instrumento,moneda,retorno_anual_historico,retorno_real,retorno_real_pesos,volatilidad_anual,nivel_riesgo,liquidez,retorno_por_unidad_riesgo
12,Criptomonedas,USD,24.8%,21.5%,23.7%,65.3%,5,Alta,0.33
6,Fideicomiso financiero,UI,7.9%,7.9%,7.9%,7.2%,3,Muy baja,1.10
9,Acciones individuales del exterior,USD,10.1%,7.2%,9.1%,24.3%,5,Alta,0.30
8,ETF de índice global (S&P 500 / MSCI World),USD,9.4%,6.5%,8.4%,15.2%,4,Alta,0.43
5,Obligaciones negociables (BVM),USD,6.5%,3.7%,5.6%,5.8%,3,Baja,0.64
2,Letras de Regulación Monetaria (BCU),UYU,9.2%,3.6%,3.6%,1.3%,1,Media,2.77
11,Oro,USD,6.1%,3.3%,5.2%,14.1%,4,Media,0.23
3,Notas del Tesoro en Unidades Indexadas,UI,3.1%,3.1%,3.1%,2.6%,2,Media,1.19
10,Inmueble para renta,USD,5.8%,3.0%,4.9%,8.9%,3,Muy baja,0.34
0,Plazo fijo en pesos (BROU),UYU,8.5%,2.9%,2.9%,0.5%,1,Baja,5.88


In [59]:
orden_nominal = catalogo_real.sort_values("retorno_anual_historico", ascending=False)
orden_real = catalogo_real.sort_values("retorno_real", ascending=False)
orden_real_pesos = catalogo_real.sort_values("retorno_real_pesos", ascending=False)

comparacion_ranking = pd.DataFrame(
    {
        "por retorno nominal": orden_nominal["nombre_instrumento"].str[:38].values,
        "por retorno real": orden_real["nombre_instrumento"].str[:38].values,
        "por real en pesos": orden_real_pesos["nombre_instrumento"].str[:38].values,
    },
    index=range(1, len(catalogo_real) + 1),
)
display(comparacion_ranking)

negativos = catalogo_real[catalogo_real["retorno_real"] < 0]
print("Instrumentos que PIERDEN poder de compra en términos reales:")
display(
    negativos[
        [
            "nombre_instrumento",
            "moneda",
            "retorno_anual_historico",
            "retorno_real",
            "liquidez",
        ]
    ].style.format({"retorno_anual_historico": "{:.1%}", "retorno_real": "{:.1%}"})
)

coincide_top5 = set(orden_real["nombre_instrumento"].head(5)) == set(
    orden_real_pesos["nombre_instrumento"].head(5)
)
print(f"\n¿Las dos versiones de retorno real dan el mismo top 5? {coincide_top5}")

,por retorno nominal,por retorno real,por real en pesos
1,Criptomonedas,Criptomonedas,Criptomonedas
2,Acciones individuales del exterior,Fideicomiso financiero,Acciones individuales del exterior
3,ETF de índice global (S&P 500 / MSCI W,Acciones individuales del exterior,ETF de índice global (S&P 500 / MSCI W
4,Letras de Regulación Monetaria (BCU),ETF de índice global (S&P 500 / MSCI W,Fideicomiso financiero
5,Plazo fijo en pesos (BROU),Obligaciones negociables (BVM),Obligaciones negociables (BVM)
6,Fideicomiso financiero,Letras de Regulación Monetaria (BCU),Oro
7,Fondo de inversión local (AFISA),Oro,Inmueble para renta
8,Obligaciones negociables (BVM),Notas del Tesoro en Unidades Indexadas,Letras de Regulación Monetaria (BCU)
9,Oro,Inmueble para renta,Notas del Tesoro en Unidades Indexadas
10,Inmueble para renta,Plazo fijo en pesos (BROU),Plazo fijo en pesos (BROU)


Instrumentos que PIERDEN poder de compra en términos reales:


,nombre_instrumento,moneda,retorno_anual_historico,retorno_real,liquidez
7,Ahorro voluntario en AFAP,UYU,4.9%,-0.5%,Muy baja
13,Dólares en caja de seguridad,USD,0.0%,-2.6%,Alta



¿Las dos versiones de retorno real dan el mismo top 5? True


In [60]:
# Retorno real de la cartera de cada cliente

posiciones = posiciones.merge(
    catalogo_real[["id_instrumento", "retorno_real", "retorno_real_pesos"]],
    on="id_instrumento",
    how="left",
)

retornos_cliente = posiciones.groupby("id_cliente").apply(
    lambda g: pd.Series(
        {
            "retorno_real_cartera": promedio_ponderado(g, "retorno_real"),
            "retorno_real_pesos_cartera": promedio_ponderado(g, "retorno_real_pesos"),
        }
    ),
    include_groups=False,
)

perfil_cartera = perfil_cartera.join(retornos_cliente)
display(
    perfil_cartera[
        [
            "retorno_nominal_cartera",
            "retorno_real_cartera",
            "retorno_real_pesos_cartera",
        ]
    ]
    .describe()
    .T.style.format("{:.2%}", subset=["mean", "std", "min", "25%", "50%", "75%", "max"])
)

print("Retorno real medio de la cartera, por perfil de riesgo real:")
display(
    perfil_cartera.groupby("perfil_real", observed=True)
    .agg(
        clientes=("retorno_real_cartera", "size"),
        retorno_nominal=("retorno_nominal_cartera", "mean"),
        retorno_real=("retorno_real_cartera", "mean"),
        volatilidad=("volatilidad_cartera", "mean"),
    )
    .style.format(
        {"retorno_nominal": "{:.2%}", "retorno_real": "{:.2%}", "volatilidad": "{:.2%}"}
    )
)

,count,mean,std,min,25%,50%,75%,max
retorno_nominal_cartera,628.000000,6.87%,3.14%,0.00%,5.31%,6.60%,7.90%,24.80%
retorno_real_cartera,628.000000,3.43%,3.33%,-2.63%,1.61%,2.94%,4.32%,21.52%
retorno_real_pesos_cartera,628.000000,4.09%,3.63%,-0.89%,2.17%,3.09%,5.23%,23.70%


Retorno real medio de la cartera, por perfil de riesgo real:


,clientes,retorno_nominal,retorno_real,volatilidad
perfil_real,,,,
Conservador,336,5.97%,2.11%,2.56%
Moderado,191,6.50%,3.58%,7.56%
Agresivo,101,10.56%,7.54%,23.21%


In [61]:
# Tabla de clientes: una fila por cliente con todo lo construido en esta etapa

clientes = (
    encuesta.set_index("id_cliente")
    .join(cartera)
    .join(perfil_cartera)
    .join(efectivo[["efectivo_usd", "reserva", "excedente"]])
)

clientes["tiene_cartera"] = clientes["cartera_usd"].notna()
for columna in ["efectivo_usd", "reserva", "excedente"]:
    clientes[columna] = clientes[columna].fillna(0)
clientes["peso_efectivo"] = (
    clientes["efectivo_usd"] / (clientes["cartera_usd"] + clientes["reserva"])
).fillna(0)

print(f"Tabla de clientes: {clientes.shape[0]} filas x {clientes.shape[1]} columnas")
print(f"  con cartera de inversión : {int(clientes['tiene_cartera'].sum())}")
print(f"  sin cartera: {int((~clientes['tiene_cartera']).sum())}")

con_cartera = clientes[clientes["tiene_cartera"]]
assert len(clientes) == len(encuesta), "Una fila por cliente"
assert (
    con_cartera["riesgo_efectivo"].between(1, 5).all()
), "El riesgo efectivo tiene que caer dentro de la escala 1 a 5"
assert (
    con_cartera["perfil_real"].notna().all()
), "Todo cliente con cartera tiene perfil real"
assert np.isclose(
    clientes["cartera_usd"].sum() + clientes["reserva"].sum(),
    movimientos_utiles["monto_usd"].sum(),
), "Cartera + reserva tiene que reconstruir el total operado"
print("\nControles de las variables creadas: todos pasaron.")

clientes.to_parquet(SALIDAS / "clientes_con_perfil.parquet")
posiciones.to_parquet(SALIDAS / "posiciones.parquet", index=False)
print(f"\nGuardado en {SALIDAS}")

Tabla de clientes: 640 filas x 54 columnas
  con cartera de inversión : 628
  sin cartera: 12

Controles de las variables creadas: todos pasaron.



Guardado en C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\salidas


## Resumen de la etapa

La escala de la firma está comprimida arriba (acciones y cripto comparten el 5), y en términos reales el ahorro en AFAP (−0,5 %) y los dólares en la caja (−2,6 %) pierden poder de compra, aunque el cliente los vea como lo más seguro.

# Análisis de negocio

## Dicen vs. hacen

El porcentaje de coincidencia solo no alcanza: no descuenta el azar (asignando perfiles de forma random algo coincidiría igual) y trata igual a un conservador con cartera moderada que a uno con cartera agresiva. Para comparar dos clasificaciones ordenadas uso el kappa ponderado de Cohen, que compara el acuerdo observado con el esperable por azar. Uso pesos lineales, como se recomienda cuando se informa un solo índice.

Para leerlo uso la escala de Landis y Koch: hasta 0,20 es insignificante, de 0,21 a 0,40 débil, de 0,41 a 0,60 moderado, de 0,61 a 0,80 sustancial y más de 0,80 casi perfecto.

In [62]:
# Medidas de acuerdo entre dos clasificaciones ordinales


def matriz_pesos(k: int, tipo: str = "lineal") -> np.ndarray:
    """Pesos de desacuerdo para kappa ponderado: 0 en la diagonal, 1 en el peor caso."""
    indices = np.arange(k)
    distancia = np.abs(indices[:, None] - indices[None, :]) / (k - 1)
    return distancia if tipo == "lineal" else distancia**2


def kappa_cohen(
    observado: np.ndarray,
    predicho: np.ndarray,
    k: int = 3,
    pesos: str | None = "lineal",
) -> float:
    """Kappa de Cohen, con o sin pesos ordinales.

    kappa = 1 - (desacuerdo observado) / (desacuerdo esperado por azar)

    Con `pesos=None` todos los desacuerdos valen igual. Con pesos lineales, un
    desacuerdo
    de una categoría pesa la mitad que uno de dos.
    """
    conjunta = np.zeros((k, k))
    for fila, columna in zip(observado, predicho):
        conjunta[fila, columna] += 1
    conjunta /= conjunta.sum()
    esperada = np.outer(conjunta.sum(axis=1), conjunta.sum(axis=0))

    if pesos is None:
        acuerdo_observado = np.trace(conjunta)
        acuerdo_esperado = np.trace(esperada)
        return (acuerdo_observado - acuerdo_esperado) / (1 - acuerdo_esperado)

    W = matriz_pesos(k, pesos)
    return 1 - (W * conjunta).sum() / (W * esperada).sum()


def interpretar_kappa(valor: float) -> str:
    """Etiqueta de la escala de interpretación habitual."""
    for limite, etiqueta in [
        (0.20, "insignificante"),
        (0.40, "débil"),
        (0.60, "moderado"),
        (0.80, "sustancial"),
    ]:
        if valor <= limite:
            return etiqueta
    return "casi perfecto"


ORDEN_PERFIL = {"Conservador": 0, "Moderado": 1, "Agresivo": 2}
print("Funciones de acuerdo ordinal definidas.")

Funciones de acuerdo ordinal definidas.


In [63]:
# El cruce: perfil autodeclarado contra perfil de la cartera real

comparables = clientes[clientes["tiene_cartera"]].copy()
comparables["rango_declarado"] = (
    comparables["perfil_autodeclarado"].map(ORDEN_PERFIL).astype(int)
)
comparables["rango_real"] = (
    comparables["perfil_real"].astype(str).map(ORDEN_PERFIL).astype(int)
)
comparables["distancia"] = comparables["rango_real"] - comparables["rango_declarado"]

matriz = pd.crosstab(
    comparables["perfil_autodeclarado"],
    comparables["perfil_real"],
    rownames=["lo que DIJO ser"],
    colnames=["lo que su cartera HACE"],
)
matriz = matriz.reindex(index=ETIQUETAS_PERFIL, columns=ETIQUETAS_PERFIL)
display(matriz)

coinciden = int(np.diag(matriz).sum())
total = len(comparables)
print(
    f"Clientes comparables: {total} (los {int((~clientes['tiene_cartera']).sum())} sin "
    "cartera no entran, no hay comportamiento que contrastar)"
)
print(f"Coinciden : {coinciden} ({coinciden / total:.1%})")
print(f"NO COINCIDEN: {total - coinciden} ({1 - coinciden / total:.1%})")

lo que su cartera HACE,Conservador,Moderado,Agresivo
lo que DIJO ser,,,
Conservador,206,84,22
Moderado,87,60,32
Agresivo,43,47,47


Clientes comparables: 628 (los 12 sin cartera no entran, no hay comportamiento que contrastar)
Coinciden : 313 (49.8%)
NO COINCIDEN: 315 (50.2%)


In [64]:
# Parámetros de los remuestreos
REPETICIONES_BOOTSTRAP = 1000  # remuestreos para cada intervalo de confianza
SEMILLA = 42  # fija todo lo aleatorio del notebook

`REPETICIONES_BOOTSTRAP` (1.000) es una cantidad habitual para un intervalo de confianza por remuestreo. 
`SEMILLA` (42) fija todo lo aleatorio 

In [65]:
# Las tres medidas de acuerdo

declarado = comparables["rango_declarado"].values
real = comparables["rango_real"].values

medidas = pd.DataFrame(
    [
        {
            "medida": "Desajuste simple (% que no coincide)",
            "valor": 1 - coinciden / total,
            "interpretacion": "—",
        },
        {
            "medida": "Kappa de Cohen sin ponderar",
            "valor": kappa_cohen(declarado, real, pesos=None),
            "interpretacion": interpretar_kappa(
                kappa_cohen(declarado, real, pesos=None)
            ),
        },
        {
            "medida": "Kappa ponderado lineal",
            "valor": kappa_cohen(declarado, real, pesos="lineal"),
            "interpretacion": interpretar_kappa(
                kappa_cohen(declarado, real, pesos="lineal")
            ),
        },
        {
            "medida": "Kappa ponderado cuadrático",
            "valor": kappa_cohen(declarado, real, pesos="cuadratico"),
            "interpretacion": interpretar_kappa(
                kappa_cohen(declarado, real, pesos="cuadratico")
            ),
        },
    ]
).set_index("medida")
display(medidas.style.format({"valor": "{:.3f}"}))

kappa_lineal = kappa_cohen(declarado, real, pesos="lineal")
kappa_simple = kappa_cohen(declarado, real, pesos=None)
print(
    f"Kappa ponderado {kappa_lineal:.2f}: acuerdo {interpretar_kappa(kappa_lineal)} "
    f"(Landis y Koch). Sin ponderar, {kappa_simple:.2f}: "
    f"{interpretar_kappa(kappa_simple)}."
)
# Intervalo por remuestreo: se repite el cálculo sobre muestras con reposición
generador = np.random.default_rng(SEMILLA)
kappas = [
    kappa_cohen(declarado[muestra], real[muestra], pesos="lineal")
    for muestra in (
        generador.integers(0, len(declarado), len(declarado))
        for _ in range(REPETICIONES_BOOTSTRAP)
    )
]
bajo_k, alto_k = np.percentile(kappas, [2.5, 97.5])
print(f"Intervalo del 95 % para el kappa ponderado: {bajo_k:.2f} a {alto_k:.2f}")
print("La autodeclaración agrega poca información por encima de adivinar.")

,valor,interpretacion
medida,,
Desajuste simple (% que no coincide),0.502,—
Kappa de Cohen sin ponderar,0.181,insignificante
Kappa ponderado lineal,0.252,débil
Kappa ponderado cuadrático,0.325,débil


Kappa ponderado 0.25: acuerdo débil (Landis y Koch). Sin ponderar, 0.18: insignificante.


Intervalo del 95 % para el kappa ponderado: 0.19 a 0.31
La autodeclaración agrega poca información por encima de adivinar.


Las dos direcciones del desajuste no son igual de graves.

El sobreexpuesto (se dijo conservador y tiene una cartera más riesgosa) es un problema legal. La normativa del BCU obliga a corredores y asesores a clasificar a cada cliente y recomendarle productos acordes a su perfil; apartarse del perfil consentido genera responsabilidad y puede terminar en sanciones. En Europa rige el mismo principio con MiFID II. Ante una caída, ese cliente tiene un formulario firmado que dice que era conservador. Puede haber clientes que dejaron constancia de querer operar fuera de su perfil, pero eso no está en estos datos.

El subexpuesto (se dijo agresivo y tiene una cartera más conservadora) no incumple nada: es retorno que no consigue y comisiones que la firma no cobra. Es un problema comercial.

In [66]:
# Severidad del desajuste, separada por dirección, con el patrimonio en juego

ETIQUETAS_DISTANCIA = {
    -2: "−2 · se dijo Agresivo y tiene cartera Conservadora",
    -1: "−1 · un escalón más conservador de lo declarado",
    0: " 0 · coincide",
    1: "+1 · un escalón más riesgoso de lo declarado",
    2: "+2 · se dijo Conservador y tiene cartera Agresiva",
}

severidad = (
    comparables.groupby("distancia")
    .agg(
        clientes=("cartera_usd", "size"),
        patrimonio_usd=("cartera_usd", "sum"),
        cartera_mediana=("cartera_usd", "median"),
        riesgo_medio=("riesgo_efectivo", "mean"),
    )
    .rename(index=ETIQUETAS_DISTANCIA)
)
severidad["pct_clientes"] = severidad["clientes"] / total
display(
    severidad[
        [
            "clientes",
            "pct_clientes",
            "patrimonio_usd",
            "cartera_mediana",
            "riesgo_medio",
        ]
    ].style.format(
        {
            "pct_clientes": "{:.1%}",
            "patrimonio_usd": "{:,.0f}",
            "cartera_mediana": "{:,.0f}",
            "riesgo_medio": "{:.2f}",
        }
    )
)

sobreexpuestos = comparables[comparables["distancia"] > 0]
subexpuestos = comparables[comparables["distancia"] < 0]
criticos = comparables[comparables["distancia"] == 2]

print(
    f"SOBREEXPUESTOS (problema legal): {len(sobreexpuestos):d} clientes "
    f"({len(sobreexpuestos) / total:.1%}) | USD "
    f"{sobreexpuestos['cartera_usd'].sum():>10,.0f}"
)
print(
    f"  de ellos, el caso crítico (+2): {len(criticos):d} clientes "
    f"({len(criticos) / total:.1%}) | USD {criticos['cartera_usd'].sum():>10,.0f}"
)
print(
    f"SUBEXPUESTOS (problema comercial) : {len(subexpuestos):d} clientes "
    f"({len(subexpuestos) / total:.1%}) | USD "
    f"{subexpuestos['cartera_usd'].sum():>10,.0f}"
)

,clientes,pct_clientes,patrimonio_usd,cartera_mediana,riesgo_medio
distancia,,,,,
−2 · se dijo Agresivo y tiene cartera Conservadora,43,6.8%,"681,553","11,806",1.67
−1 · un escalón más conservador de lo declarado,134,21.3%,"2,972,371","11,855",2.16
0 · coincide,313,49.8%,"5,495,785","10,559",2.29
+1 · un escalón más riesgoso de lo declarado,116,18.5%,"2,629,340","14,648",3.25
+2 · se dijo Conservador y tiene cartera Agresiva,22,3.5%,"435,579","10,109",3.92


SOBREEXPUESTOS (problema legal): 138 clientes (22.0%) | USD  3,064,919
  de ellos, el caso crítico (+2): 22 clientes (3.5%) | USD    435,579
SUBEXPUESTOS (problema comercial) : 177 clientes (28.2%) | USD  3,653,923


In [67]:
# Quiénes son los 22 clientes del caso crítico

display(
    criticos[
        [
            "edad",
            "ocupacion",
            "patrimonio_invertible_usd",
            "cartera_usd",
            "riesgo_efectivo",
            "volatilidad_cartera",
            "peso_cripto",
            "tolerancia_perdida_maxima",
            "fuente_informacion",
        ]
    ]
    .sort_values("cartera_usd", ascending=False)
    .style.format(
        {
            "patrimonio_invertible_usd": "{:,.0f}",
            "cartera_usd": "{:,.0f}",
            "riesgo_efectivo": "{:.2f}",
            "volatilidad_cartera": "{:.1%}",
            "peso_cripto": "{:.1%}",
        }
    )
)

print("Rasgos de los sobreexpuestos críticos frente al resto de los comparables:")
resumen_criticos = pd.DataFrame(
    {
        "críticos (+2)": criticos[
            [
                "edad",
                "riesgo_efectivo",
                "volatilidad_cartera",
                "peso_cripto",
                "puntaje_test_conocimiento",
            ]
        ].mean(),
        "resto": comparables[comparables["distancia"] != 2][
            [
                "edad",
                "riesgo_efectivo",
                "volatilidad_cartera",
                "peso_cripto",
                "puntaje_test_conocimiento",
            ]
        ].mean(),
    }
)
display(resumen_criticos.style.format("{:.3f}"))

print(
    f"\nUsan redes sociales como fuente: "
    f"{(criticos['fuente_informacion'] == 'Redes sociales').mean():.0%} de los "
    f"críticos "
    f"contra {(comparables['fuente_informacion'] == 'Redes sociales').mean():.0%} del "
    f"total."
)

criticos.to_csv(
    SALIDAS / "3_clientes_criticos_sobreexpuestos.csv", encoding="utf-8-sig"
)
print(f"Listado guardado en {SALIDAS / '3_clientes_criticos_sobreexpuestos.csv'}")

,edad,ocupacion,patrimonio_invertible_usd,cartera_usd,riesgo_efectivo,volatilidad_cartera,peso_cripto,tolerancia_perdida_maxima,fuente_informacion
id_cliente,,,,,,,,,
20570,54.000000,Profesional independiente,"171,100","149,366",3.52,12.1%,0.0%,Hasta 5%,Prensa especializada
20193,50.000000,Empleado en relación de dependencia,"60,600","54,676",3.77,12.5%,0.0%,Hasta 10%,Redes sociales
20235,35.000000,Empleado en relación de dependencia,"43,000","39,164",4.04,14.7%,0.0%,Más de 20%,Redes sociales
20488,68.000000,Jubilado,"48,000","25,618",3.78,13.0%,0.0%,No aceptaría ninguna pérdida,Investigación propia
20255,,Empleado en relación de dependencia,"39,500","25,039",3.69,11.7%,0.0%,Hasta 10%,Amigos y familia
20537,52.000000,Empleado en relación de dependencia,"21,500","15,872",5.00,24.3%,0.0%,Hasta 5%,Amigos y familia
20442,82.000000,Jubilado,"17,100","12,559",4.00,14.1%,0.0%,Hasta 5%,Asesor financiero
20176,41.000000,Profesional independiente,"21,700","12,381",3.68,11.5%,0.0%,Más de 20%,Redes sociales
20091,40.000000,Empresario / dueño de negocio,"15,000","12,240",3.68,11.9%,0.0%,Hasta 10%,Redes sociales


Rasgos de los sobreexpuestos críticos frente al resto de los comparables:


,críticos (+2),resto
edad,46.524,47.221
riesgo_efectivo,3.916,2.403
volatilidad_cartera,0.181,0.070
peso_cripto,0.086,0.028
puntaje_test_conocimiento,4.000,4.495



Usan redes sociales como fuente: 36% de los críticos contra 28% del total.
Listado guardado en C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\salidas\3_clientes_criticos_sobreexpuestos.csv


In [68]:
# Quiénes son los desajustados: sobreexpuestos y subexpuestos contra los que coinciden

DIRECCION = {-1: "subexpuesto", 0: "coincide", 1: "sobreexpuesto"}
quienes = (
    comparables.assign(
        direccion=np.sign(comparables["distancia"]).map(DIRECCION),
        usa_redes=comparables["fuente_informacion"] == "Redes sociales",
        sin_fondo=comparables["tiene_fondo_emergencia"] == "No",
        tolera_poco=comparables["tolerancia_perdida_maxima"].isin(
            ["Hasta 5%", "No aceptaría ninguna pérdida"]
        ),
        espera_mas_de_30=comparables["retorno_esperado_anual"] == "Más de 30%",
    )
    .groupby("direccion")
    .agg(
        clientes=("cartera_usd", "size"),
        cartera_mediana=("cartera_usd", "median"),
        edad=("edad", "mean"),
        test=("puntaje_test_conocimiento", "mean"),
        usa_redes=("usa_redes", "mean"),
        sin_fondo=("sin_fondo", "mean"),
        tolera_poco=("tolera_poco", "mean"),
        espera_mas_de_30=("espera_mas_de_30", "mean"),
    )
    .reindex(["sobreexpuesto", "coincide", "subexpuesto"])
)
display(
    quienes.style.format(
        {
            "cartera_mediana": "{:,.0f}",
            "edad": "{:.1f}",
            "test": "{:.2f}",
            "usa_redes": "{:.0%}",
            "sin_fondo": "{:.0%}",
            "tolera_poco": "{:.0%}",
            "espera_mas_de_30": "{:.0%}",
        }
    )
)

,clientes,cartera_mediana,edad,test,usa_redes,sin_fondo,tolera_poco,espera_mas_de_30
direccion,,,,,,,,
sobreexpuesto,138,"13,328",46.3,3.92,38%,60%,27%,27%
coincide,313,"10,559",47.7,4.41,25%,51%,50%,15%
subexpuesto,177,"11,806",46.9,5.05,27%,57%,45%,17%


Comparados con los que coinciden, los sobreexpuestos se informan más por redes (38 % contra 25 %), esperan más de 30 % anual casi el doble de veces (27 % contra 15 %), tienen peor test (3,9 contra 4,4) y una cartera mediana más grande (USD 13.328 contra 10.559). Sólo el 27 % dice tolerar como mucho una pérdida del 5 %, contra el 50 % de los que coinciden: muchos se declararon conservadores aunque el resto de sus respuestas no lo son.

Los subexpuestos se parecen a los que coinciden, con mejor test (5,1).

In [69]:
# Pruebo cambiar los valores de los cortes

robustez = []
for corte_bajo in np.arange(2.0, 3.01, 0.1):
    for corte_alto in np.arange(3.0, 4.01, 0.1):
        if corte_alto <= corte_bajo:
            continue
        propuesto = (
            pd.cut(
                comparables["riesgo_efectivo"],
                bins=[0, corte_bajo, corte_alto, 5.01],
                labels=ETIQUETAS_PERFIL,
                right=False,
            )
            .astype(str)
            .map(ORDEN_PERFIL)
            .astype(int)
            .values
        )
        robustez.append(
            {
                "corte_bajo": round(corte_bajo, 1),
                "corte_alto": round(corte_alto, 1),
                "desajuste": (declarado != propuesto).mean(),
                "kappa_lineal": kappa_cohen(declarado, propuesto, pesos="lineal"),
                "sobreexpuestos": (propuesto > declarado).mean(),
                "subexpuestos": (propuesto < declarado).mean(),
            }
        )

robustez = pd.DataFrame(robustez)

es_adoptado = (robustez["corte_bajo"] == CORTE_CONSERVADOR_MODERADO) & (
    robustez["corte_alto"] == CORTE_MODERADO_AGRESIVO
)
detalle_hover = (
    "<b>Cortes %{customdata[0]:.1f} y %{customdata[1]:.1f}</b><br>"
    "Desajuste: %{x:.1%}<br>"
    "Kappa ponderado: %{y:.2f}<br>"
    "Sobreexpuestos %{customdata[2]:.1%} · subexpuestos %{customdata[3]:.1%}"
    "<extra></extra>"
)
columnas_hover = ["corte_bajo", "corte_alto", "sobreexpuestos", "subexpuestos"]

figura = go.Figure()
figura.add_hrect(y0=0.40, y1=0.62, fillcolor="#f0efec", line_width=0, layer="below")
figura.add_annotation(
    x=0.405,
    y=0.58,
    text="Zona de acuerdo moderado o mejor (kappa mayor a 0,40):<br>"
    "ninguna de las 120 combinaciones llega",
    showarrow=False,
    xanchor="left",
    align="left",
    font=dict(color=TINTA_SECUNDARIA),
)
figura.add_trace(
    go.Scatter(
        x=robustez.loc[~es_adoptado, "desajuste"],
        y=robustez.loc[~es_adoptado, "kappa_lineal"],
        mode="markers",
        marker=dict(size=10, color=aclarar(COLOR_PERFIL["Conservador"], 0.35)),
        customdata=robustez.loc[~es_adoptado, columnas_hover].values,
        hovertemplate=detalle_hover,
        name="Otros cortes posibles",
    )
)
figura.add_trace(
    go.Scatter(
        x=robustez.loc[es_adoptado, "desajuste"],
        y=robustez.loc[es_adoptado, "kappa_lineal"],
        mode="markers",
        marker=dict(size=16, color=TINTA_PRIMARIA, line=dict(color="white", width=2)),
        customdata=robustez.loc[es_adoptado, columnas_hover].values,
        hovertemplate=detalle_hover,
        name="Cortes adoptados",
    )
)
figura.update_xaxes(
    title="Desajuste entre perfil declarado y real",
    tickformat=".0%",
    range=[0.40, 0.60],
)
figura.update_yaxes(title="Kappa ponderado", range=[0, 0.62], dtick=0.1)
figura.add_annotation(
    x=robustez.loc[es_adoptado, "desajuste"].iloc[0],
    y=robustez.loc[es_adoptado, "kappa_lineal"].iloc[0],
    text="<b>Cortes adoptados (2,5 y 3,5)</b>",
    showarrow=True,
    arrowcolor=TINTA_SECUNDARIA,
    ax=0,
    ay=-70,
    font=dict(color=TINTA_PRIMARIA),
    bgcolor=SUPERFICIE,
)
figura.update_layout(showlegend=False)
presentar(
    figura,
    "Con cualquier par de cortes, el kappa entre perfil declarado y real no llega a "
    "0,30",
    "120 combinaciones de cortes · pasá el mouse para ver el detalle",
    alto=470,
)

print(f"Combinaciones de cortes evaluadas: {len(robustez)}")
print(
    f"El desajuste va de {robustez['desajuste'].min():.1%} a "
    f"{robustez['desajuste'].max():.1%} "
    f"(mediana {robustez['desajuste'].median():.1%})."
)
print(
    f"El kappa ponderado va de {robustez['kappa_lineal'].min():.3f} a "
    f"{robustez['kappa_lineal'].max():.3f}."
)
print(
    f"\n¿Alguna combinación alcanza acuerdo 'moderado' (kappa > 0,40)? "
    f"{'Sí' if robustez['kappa_lineal'].max() > 0.40 else 'NO, ninguna'}."
)
print(
    f"Sobreexpuestos: de {robustez['sobreexpuestos'].min():.1%} a "
    f"{robustez['sobreexpuestos'].max():.1%} | subexpuestos: de "
    f"{robustez['subexpuestos'].min():.1%} a {robustez['subexpuestos'].max():.1%}."
)

Combinaciones de cortes evaluadas: 120
El desajuste va de 45.4% a 57.5% (mediana 50.6%).
El kappa ponderado va de 0.185 a 0.273.

¿Alguna combinación alcanza acuerdo 'moderado' (kappa > 0,40)? NO, ninguna.
Sobreexpuestos: de 8.1% a 39.0% | subexpuestos: de 18.5% a 37.4%.


No depende de los cortes. Con cualquier combinación el desajuste está entre 45,4 % y 57,5 % y el kappa entre 0,185 y 0,273; ninguna llega a un acuerdo moderado (0,40). Con los cortes elegidos, el intervalo del 95 % del kappa va de 0,19 a 0,31. Lo que sí cambia es el reparto entre sobreexpuestos (8,1 % a 39,0 %) y subexpuestos (18,5 % a 37,4 %), así que esos números van siempre con sus cortes.

## ¿Las expectativas son posibles?

Comparo el retorno que el cliente dice esperar con lo que rinde su cartera, medido como el retorno histórico de cada instrumento ponderado por monto. La encuesta no dice en qué moneda piensa el cliente su retorno, así que lo comparo contra el retorno nominal de la cartera en pesos (el real en pesos más la inflación uruguaya).

In [70]:
# Retorno esperado declarado contra lo que efectivamente rinde la cartera

# Piso del rango declarado: lo mínimo que el cliente tendría que obtener para no fallar
PISO_DEL_RANGO = {
    "Menos de 5%": 0.00,
    "5% a 10%": 0.05,
    "10% a 20%": 0.10,
    "20% a 30%": 0.20,
    "Más de 30%": 0.30,
}
ORDEN_EXPECTATIVA = list(PISO_DEL_RANGO)

# El nominal de la cartera promedia instrumentos en pesos y en dólares, que no se
# comparan entre sí. Se pasa todo a un nominal en pesos: el retorno real en pesos
# más la inflación uruguaya
comparables["retorno_nominal_pesos_cartera"] = (
    1 + comparables["retorno_real_pesos_cartera"]
) * (1 + INFLACION_UY) - 1

expectativas = (
    comparables.groupby("retorno_esperado_anual", observed=True)
    .agg(
        clientes=("retorno_nominal_cartera", "size"),
        rinde_nominal=("retorno_nominal_pesos_cartera", "mean"),
        rinde_real=("retorno_real_cartera", "mean"),
        riesgo=("riesgo_efectivo", "mean"),
        volatilidad=("volatilidad_cartera", "mean"),
    )
    .reindex(ORDEN_EXPECTATIVA)
)
expectativas["piso_esperado"] = [PISO_DEL_RANGO[k] for k in ORDEN_EXPECTATIVA]
expectativas["brecha"] = expectativas["piso_esperado"] - expectativas["rinde_nominal"]
expectativas["retorno_por_volatilidad"] = (
    expectativas["rinde_nominal"] / expectativas["volatilidad"]
)

display(
    expectativas.style.format(
        {
            "rinde_nominal": "{:.2%}",
            "rinde_real": "{:.2%}",
            "riesgo": "{:.2f}",
            "volatilidad": "{:.2%}",
            "piso_esperado": "{:.0%}",
            "brecha": "{:+.2%}",
            "retorno_por_volatilidad": "{:.2f}",
        }
    )
)

,clientes,rinde_nominal,rinde_real,riesgo,volatilidad,piso_esperado,brecha,retorno_por_volatilidad
retorno_esperado_anual,,,,,,,,
Menos de 5%,144,8.93%,2.78%,2.02,4.20%,0%,-8.93%,2.13
5% a 10%,139,9.29%,3.10%,2.32,6.13%,5%,-4.29%,1.52
10% a 20%,140,9.09%,2.99%,2.33,5.94%,10%,+0.91%,1.53
20% a 30%,92,10.56%,4.11%,2.79,10.22%,20%,+9.44%,1.03
Más de 30%,113,11.29%,4.65%,3.06,12.57%,30%,+18.71%,0.90


In [71]:
# Los clientes que esperan los retornos más altos

mas_exigentes = comparables[comparables["retorno_esperado_anual"] == "Más de 30%"]
mejor_instrumento = catalogo_real.loc[catalogo_real["retorno_real"].idxmax()]

print(
    f"Clientes que esperan MÁS DE 30 % anual: {len(mas_exigentes)} "
    f"({len(mas_exigentes) / total:.1%} de los comparables)"
)
print(
    f"  su cartera rinde, nominal en pesos : "
    f"{mas_exigentes['retorno_nominal_pesos_cartera'].mean():.2%}"
)
print(f"  su cartera rinde, real: {mas_exigentes['retorno_real_cartera'].mean():.2%}")
piso_exigentes = PISO_DEL_RANGO["Más de 30%"]
nominal_exigentes = mas_exigentes["retorno_nominal_pesos_cartera"]
real_exigentes = mas_exigentes["retorno_real_cartera"]
print(
    "  brecha contra su propio piso declarado: "
    f"{piso_exigentes - nominal_exigentes.mean():.1%}"
)
print(
    "\n  cuántos tienen una cartera que rinda más de 30 % nominal: "
    f"{int((nominal_exigentes > piso_exigentes).sum())}"
)
print(f"  cuántos, en términos reales: {int((real_exigentes > piso_exigentes).sum())}")

# El instrumento más rentable, medido igual que las carteras: nominal en pesos
mejor_nominal_pesos = (1 + mejor_instrumento["retorno_real_pesos"]) * (
    1 + INFLACION_UY
) - 1
print(
    "\nY no es que armaron mal la cartera: el instrumento más rentable del catálogo es "
    f"{mejor_instrumento['nombre_instrumento']}, con "
    f"{mejor_nominal_pesos:.1%} nominal en pesos y "
    f"{mejor_instrumento['retorno_real']:.1%} real."
)
print(
    "Sólo una cartera casi toda en cripto llega a un 30 % nominal en pesos, y eso "
    "porque\nincluye la devaluación esperada. En términos reales ninguna llega."
)

Clientes que esperan MÁS DE 30 % anual: 113 (18.0% de los comparables)
  su cartera rinde, nominal en pesos : 11.29%
  su cartera rinde, real: 4.65%
  brecha contra su propio piso declarado: 18.7%

  cuántos tienen una cartera que rinda más de 30 % nominal: 5
  cuántos, en términos reales: 0

Y no es que armaron mal la cartera: el instrumento más rentable del catálogo es Criptomonedas, con 30.4% nominal en pesos y 21.5% real.
Sólo una cartera casi toda en cripto llega a un 30 % nominal en pesos, y eso porque
incluye la devaluación esperada. En términos reales ninguna llega.


En los tramos altos la expectativa no se cumple. Quien espera más de 30 % obtiene 11,3 % nominal en pesos y 4,7 % real. Sólo 5 de los 113 llegan a 30 % nominal, con carteras casi todas en cripto, y en términos reales ninguno. Los que esperan menos de 5 % obtienen 8,9 %: el problema está arriba.

Y actúan según esa expectativa: del tramo más bajo al más alto la volatilidad se triplica (de 4,2 % a 12,6 %) para ganar apenas un cuarto más de retorno (de 8,9 % a 11,3 %). El retorno por unidad de volatilidad cae de 2,13 a 0,90. Es una versión simplificada del ratio de Sharpe (1966): el original le resta al retorno la tasa libre de riesgo antes de dividir por la volatilidad; acá no la resto porque sólo comparo tramos entre sí.

In [72]:
# Los dólares inmovilizados, contra la inflación de Estados Unidos

con_efectivo = clientes[clientes["efectivo_usd"] > 0]
efectivo_total = con_efectivo["efectivo_usd"].sum()
excedente_total = con_efectivo["excedente"].sum()
reserva_total = con_efectivo["reserva"].sum()

plazo_fijo_usd = catalogo_real[
    catalogo_real["nombre_instrumento"] == "Plazo fijo en dólares"
].iloc[0]

print(
    f"Clientes con dólares en caja de seguridad: {len(con_efectivo)} "
    f"({len(con_efectivo) / len(clientes):.1%})"
)
print(f"Monto total: USD {efectivo_total:,.0f}\n")

print(
    f"La inflación de Estados Unidos es {INFLACION_US:.1%} anual y el efectivo rinde 0"
    f" %,"
)
print(f"así que su retorno real es {-INFLACION_US / (1 + INFLACION_US):.2%}.\n")

perdida = pd.DataFrame(
    [
        {"tramo": "Reserva de emergencia (justificada)", "monto_usd": reserva_total},
        {
            "tramo": "Excedente sobre la reserva (sin justificación)",
            "monto_usd": excedente_total,
        },
        {"tramo": "TOTAL inmovilizado", "monto_usd": efectivo_total},
    ]
).set_index("tramo")
perdida["pierde_por_anio"] = perdida["monto_usd"] * INFLACION_US / (1 + INFLACION_US)
perdida["pierde_en_5_anios"] = perdida["monto_usd"] * (1 - 1 / (1 + INFLACION_US) ** 5)
# Alternativa: plazo fijo en dólares, también riesgo 1. Se compara en términos reales,
# igual que la pérdida
perdida["ganaria_real_en_plazo_fijo"] = (
    perdida["monto_usd"] * plazo_fijo_usd["retorno_real"]
)
display(perdida.style.format("{:,.0f}"))
liquidez = catalogo.set_index("id_instrumento")["liquidez"]
print(
    f"Plazo fijo en dólares: {plazo_fijo_usd['retorno_real']:.2%} real, liquidez "
    f"{liquidez[2].lower()} · dólares en la caja: liquidez "
    f"{liquidez[ID_EFECTIVO].lower()}"
)

concentrados = con_efectivo[con_efectivo["peso_efectivo"] > 0.5]
print(
    f"Clientes con más de la mitad de todo lo que tienen en efectivo: "
    f"{len(concentrados)}"
)
print(f"  monto que acumulan: USD {concentrados['efectivo_usd'].sum():,.0f}")

Clientes con dólares en caja de seguridad: 171 (26.7%)
Monto total: USD 1,056,554

La inflación de Estados Unidos es 2.7% anual y el efectivo rinde 0 %,
así que su retorno real es -2.63%.



,monto_usd,pierde_por_anio,pierde_en_5_anios,ganaria_real_en_plazo_fijo
tramo,,,,
Reserva de emergencia (justificada),"631,936","16,614","78,814","3,692"
Excedente sobre la reserva (sin justificación),"424,618","11,163","52,958","2,481"
TOTAL inmovilizado,"1,056,554","27,777","131,772","6,173"


Plazo fijo en dólares: 0.58% real, liquidez baja · dólares en la caja: liquidez alta
Clientes con más de la mitad de todo lo que tienen en efectivo: 36
  monto que acumulan: USD 396,338


Con 2,7 % de inflación en EE. UU., los dólares en la caja pierden 2,6 % real por año: sobre USD 1.056.554 son unos USD 27.800 anuales y USD 131.800 en cinco años, sin que se vea en ningún estado de cuenta.

Los USD 631.936 de reserva de emergencia tienen un motivo. Los USD 424.618 que sobran pierden unos USD 11.200 por año; en un plazo fijo en dólares, también de riesgo 1 pero de liquidez baja, ganarían 0,58 % real (unos USD 2.500).

## Qué explica el comportamiento

In [73]:
# Regresión por mínimos cuadrados, corrección por comparaciones múltiples y
# estandarización


def regresion_ols(X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
    """Ajusta y = a + Xb y devuelve coeficientes, errores estándar, t y p."""
    diseno = np.column_stack([np.ones(len(X)), X.values])
    coeficientes, *_ = np.linalg.lstsq(diseno, y, rcond=None)
    residuos = y - diseno @ coeficientes
    n, k = diseno.shape
    inversa = np.linalg.inv(diseno.T @ diseno)
    centro = diseno.T @ (diseno * (residuos**2)[:, None])
    errores = np.sqrt(np.diag(inversa @ centro @ inversa) * n / (n - k))
    t = coeficientes / errores
    p = 2 * (1 - stats.t.cdf(np.abs(t), n - k))
    r2 = 1 - (residuos @ residuos) / ((y - y.mean()) @ (y - y.mean()))

    resultado = pd.DataFrame(
        {"coeficiente": coeficientes, "error_estandar": errores, "t": t, "p": p},
        index=["(constante)", *X.columns],
    )
    resultado.attrs["r2"] = r2
    resultado.attrs["n"] = n
    return resultado


def ajuste_holm(p: pd.Series) -> pd.Series:
    """Valores p corregidos por comparaciones múltiples.

    Con muchas pruebas a la vez, alguna sale significativa por azar. Holm ordena los p
    de menor a mayor, multiplica el i-ésimo por (m - i) y no deja que bajen.
    """
    orden = p.sort_values()
    m = len(orden)
    ajustados = (orden * (m - np.arange(m))).cummax().clip(upper=1)
    return ajustados.reindex(p.index)


def estandarizar(tabla: pd.DataFrame) -> pd.DataFrame:
    """Lleva cada columna a media 0 y desvío 1, para comparar coeficientes."""
    return (tabla - tabla.mean()) / tabla.std()


print("Funciones de regresión y de corrección definidas.")

Funciones de regresión y de corrección definidas.

In [74]:
# Variables numéricas asociadas al riesgo efectivo de la cartera

VARIABLES_NUMERICAS = [
    "edad",
    "ingreso_mensual_liquido",
    "patrimonio_invertible_usd",
    "anios_experiencia_invirtiendo",
    "puntaje_test_conocimiento",
    "conocimiento_autopercibido",
    "meses_gastos_cubiertos",
    "cantidad_dependientes",
    "cartera_usd",
    "concentracion",
]

correlaciones = []
for variable in VARIABLES_NUMERICAS:
    datos = comparables[[variable, "riesgo_efectivo"]].dropna()
    r, p = stats.spearmanr(datos[variable], datos["riesgo_efectivo"])
    correlaciones.append(
        {
            "variable": variable,
            "n": len(datos),
            "r": r,
            "p": p,
            "significativa": p < 0.05,
        }
    )

correlaciones = pd.DataFrame(correlaciones).set_index("variable")
correlaciones["p_holm"] = ajuste_holm(correlaciones["p"])
correlaciones["significativa"] = correlaciones["p_holm"] < 0.05
display(
    correlaciones.reindex(
        correlaciones["r"].abs().sort_values(ascending=False).index
    ).style.format({"r": "{:+.3f}", "p": "{:.4f}", "p_holm": "{:.4f}"})
)

,n,r,p,significativa,p_holm
variable,,,,,
meses_gastos_cubiertos,615,-0.242,0.0000,True,0.0000
puntaje_test_conocimiento,610,-0.216,0.0000,True,0.0000
edad,606,-0.089,0.0288,False,0.2301
ingreso_mensual_liquido,567,-0.067,0.1106,False,0.7741
cartera_usd,628,+0.056,0.1639,False,0.9835
conocimiento_autopercibido,628,-0.048,0.2324,False,1.0000
cantidad_dependientes,628,-0.045,0.2601,False,1.0000
anios_experiencia_invirtiendo,628,+0.031,0.4363,False,1.0000
concentracion,628,-0.031,0.4386,False,1.0000


In [75]:
# Variables categóricas: riesgo medio por grupo

VARIABLES_CATEGORICAS = [
    "sexo",
    "nivel_educativo",
    "ocupacion",
    "tiene_deudas",
    "tiene_fondo_emergencia",
    "fuente_informacion",
    "objetivo_principal",
    "retorno_esperado_anual",
]

for variable in VARIABLES_CATEGORICAS:
    tabla = (
        comparables.groupby(variable, observed=True)
        .agg(
            clientes=("riesgo_efectivo", "size"),
            riesgo_medio=("riesgo_efectivo", "mean"),
            volatilidad=("volatilidad_cartera", "mean"),
        )
        .sort_values("riesgo_medio", ascending=False)
    )
    rango = tabla["riesgo_medio"].max() - tabla["riesgo_medio"].min()
    print(f"\n[{variable}] — rango de riesgo entre grupos: {rango:.2f} puntos")
    display(tabla.style.format({"riesgo_medio": "{:.2f}", "volatilidad": "{:.2%}"}))


[sexo] — rango de riesgo entre grupos: 0.27 puntos


,clientes,riesgo_medio,volatilidad
sexo,,,
Masculino,310,2.59,8.33%
Femenino,318,2.32,6.50%



[nivel_educativo] — rango de riesgo entre grupos: 0.32 puntos


,clientes,riesgo_medio,volatilidad
nivel_educativo,,,
Secundaria completa,138,2.66,8.85%
Posgrado,99,2.49,7.99%
Terciaria incompleta,104,2.49,7.85%
Terciaria completa,287,2.34,6.34%



[ocupacion] — rango de riesgo entre grupos: 0.21 puntos


,clientes,riesgo_medio,volatilidad
ocupacion,,,
Profesional independiente,84,2.54,7.43%
Empleado en relación de dependencia,240,2.48,8.11%
Jubilado,97,2.48,7.12%
Empleado público,92,2.43,6.90%
Estudiante,40,2.38,7.51%
Empresario / dueño de negocio,75,2.33,6.04%


... (6 outputs left out)


[objetivo_principal] — rango de riesgo entre grupos: 0.33 puntos


,clientes,riesgo_medio,volatilidad
objetivo_principal,,,
Crecimiento del capital,166,2.62,9.78%
Compra de vivienda,81,2.46,7.57%
Complemento jubilatorio,107,2.45,7.44%
No contesta,25,2.45,6.24%
Educación de los hijos,50,2.40,7.37%
Preservar el capital,92,2.38,6.13%
Generar una renta mensual,107,2.30,4.94%



[retorno_esperado_anual] — rango de riesgo entre grupos: 1.04 puntos


,clientes,riesgo_medio,volatilidad
retorno_esperado_anual,,,
Más de 30%,113,3.06,12.57%
20% a 30%,92,2.79,10.22%
10% a 20%,140,2.33,5.94%
5% a 10%,139,2.32,6.13%
Menos de 5%,144,2.02,4.20%


Para las numéricas uso la correlación de Spearman, que mide la relación sobre el orden y no se deja arrastrar por los extremos. Como hago muchas pruebas a la vez, corrijo los valores p con el método de Holm, que los ajusta según cuántas pruebas se hicieron para que no aparezcan relaciones por azar.

El riesgo se mueve con los meses de gastos cubiertos (−0,24) y con el puntaje del test (−0,22). El ingreso y el patrimonio no se relacionan con el riesgo, y la edad tampoco una vez corregida por la cantidad de pruebas.

Entre las categóricas, el riesgo se mueve con la fuente de información (1,07 puntos entre la más y la menos riesgosa), el retorno esperado (1,04) y tener o no fondo de emergencia (0,54). Las demográficas no separan más de un tercio de punto: lo que el cliente dice sobre cómo se comporta explica más que quién es.

Para ver qué relaciones se sostienen uso una regresión múltiple, comparo el coeficiente de cada variable sola contra el mismo coeficiente con las demás variables adentro.

In [76]:
# Comparación de cada variable sola contra el modelo con todas las demás

modelo = comparables.assign(
    usa_redes=(comparables["fuente_informacion"] == "Redes sociales").astype(float),
    es_masculino=(comparables["sexo"] == "Masculino").astype(float),
    sin_fondo=(comparables["tiene_fondo_emergencia"] == "No").astype(float),
    se_dice_agresivo=(comparables["perfil_autodeclarado"] == "Agresivo").astype(float),
    retorno_esperado=comparables["retorno_esperado_anual"].map(PISO_DEL_RANGO),
)

REGRESORES = [
    "edad",
    "ingreso_mensual_liquido",
    "patrimonio_invertible_usd",
    "anios_experiencia_invirtiendo",
    "puntaje_test_conocimiento",
    "conocimiento_autopercibido",
    "meses_gastos_cubiertos",
    "cantidad_dependientes",
    "usa_redes",
    "es_masculino",
    "sin_fondo",
    "se_dice_agresivo",
    "retorno_esperado",
]

datos_modelo = modelo[REGRESORES + ["riesgo_efectivo"]].dropna().astype(float)
y = datos_modelo["riesgo_efectivo"].values
X = estandarizar(datos_modelo[REGRESORES])

completo = regresion_ols(X, y)
print(f"Modelo completo: n = {completo.attrs['n']}, R² = {completo.attrs['r2']:.3f}\n")

comparacion_regresion = pd.DataFrame(
    [
        {
            "variable": variable,
            "b_sola": regresion_ols(X[[variable]], y).loc[variable, "coeficiente"],
            "p_sola": regresion_ols(X[[variable]], y).loc[variable, "p"],
            "b_controlado": completo.loc[variable, "coeficiente"],
            "p_controlado": completo.loc[variable, "p"],
        }
        for variable in REGRESORES
    ]
).set_index("variable")
# Son 13 pruebas en cada columna: los p se corrigen por comparaciones múltiples
comparacion_regresion["p_sola_holm"] = ajuste_holm(comparacion_regresion["p_sola"])
comparacion_regresion["p_controlado_holm"] = ajuste_holm(
    comparacion_regresion["p_controlado"]
)
sola_sig = comparacion_regresion["p_sola_holm"] < 0.05
controlada_sig = comparacion_regresion["p_controlado_holm"] < 0.05
comparacion_regresion["veredicto"] = np.select(
    [sola_sig & ~controlada_sig, sola_sig & controlada_sig, ~sola_sig & controlada_sig],
    ["SE DESARMA", "sobrevive", "aparece al controlar"],
    default="nunca significativa",
)

display(
    comparacion_regresion.style.format(
        {
            "b_sola": "{:+.4f}",
            "p_sola": "{:.4f}",
            "b_controlado": "{:+.4f}",
            "p_controlado": "{:.4f}",
            "p_sola_holm": "{:.4f}",
            "p_controlado_holm": "{:.4f}",
        }
    )
)

Modelo completo: n = 518, R² = 0.286



,b_sola,p_sola,b_controlado,p_controlado,p_sola_holm,p_controlado_holm,veredicto
variable,,,,,,,
edad,-0.0676,0.1389,-0.0205,0.6425,0.4618,1.0000,nunca significativa
ingreso_mensual_liquido,-0.0882,0.0426,-0.0094,0.8180,0.2557,1.0000,nunca significativa
patrimonio_invertible_usd,+0.0013,0.9735,+0.0122,0.7602,0.9735,1.0000,nunca significativa
anios_experiencia_invirtiendo,+0.0493,0.2772,+0.1234,0.0085,0.5545,0.0767,nunca significativa
puntaje_test_conocimiento,-0.2194,0.0000,-0.0983,0.0395,0.0000,0.2762,SE DESARMA
conocimiento_autopercibido,-0.0800,0.0924,-0.0055,0.9121,0.4618,1.0000,nunca significativa
meses_gastos_cubiertos,-0.2244,0.0000,+0.0188,0.7527,0.0000,1.0000,SE DESARMA
cantidad_dependientes,-0.0730,0.1077,-0.0204,0.6024,0.4618,1.0000,nunca significativa
usa_redes,+0.3853,0.0000,+0.2051,0.0000,0.0000,0.0006,sobrevive


In [77]:
# La relación que se desarma, paso a paso, con intervalo para la parte mediada

CONTROLES_PASO_A_PASO = {
    "Sin controles": [],
    "+ fuente (redes)": ["usa_redes"],
    "+ retorno esperado": ["retorno_esperado"],
    "+ fondo de emergencia": ["sin_fondo"],
    "+ fuente y retorno esperado": ["usa_redes", "retorno_esperado"],
}
pasos = pd.DataFrame(
    {
        nombre: regresion_ols(X[["puntaje_test_conocimiento", *control]], y).loc[
            "puntaje_test_conocimiento", ["coeficiente", "error_estandar", "p"]
        ]
        for nombre, control in CONTROLES_PASO_A_PASO.items()
    }
).T
display(
    pasos.style.format(
        {"coeficiente": "{:+.3f}", "error_estandar": "{:.3f}", "p": "{:.4f}"}
    )
)

crudo_test = pasos.loc["Sin controles", "coeficiente"]
controlado_test = pasos.loc["+ fuente y retorno esperado", "coeficiente"]
print(
    f"El coeficiente pasa de {crudo_test:+.3f} a {controlado_test:+.3f}: pierde el "
    f"{1 - abs(controlado_test) / abs(crudo_test):.0%}."
)

# La parte del efecto que pasa por la fuente y la expectativa (efecto total menos
# efecto directo), con intervalo por remuestreo, en lugar de mirar sólo si el p cruza
# 0,05
columnas_mediacion = ["puntaje_test_conocimiento", "usa_redes", "retorno_esperado"]
generador = np.random.default_rng(SEMILLA)
mediado = []
for _ in range(REPETICIONES_BOOTSTRAP):
    muestra = generador.integers(0, len(y), len(y))
    x_muestra, y_muestra = X.iloc[muestra], y[muestra]
    efecto_total = np.linalg.lstsq(
        np.column_stack([np.ones(len(muestra)), x_muestra[columnas_mediacion[:1]]]),
        y_muestra,
        rcond=None,
    )[0][1]
    directo = np.linalg.lstsq(
        np.column_stack([np.ones(len(muestra)), x_muestra[columnas_mediacion]]),
        y_muestra,
        rcond=None,
    )[0][1]
    mediado.append(efecto_total - directo)
bajo_m, alto_m = np.percentile(mediado, [2.5, 97.5])
print(
    "Parte del efecto que pasa por fuente y expectativa: "
    f"{crudo_test - controlado_test:+.3f} (intervalo del 95 %: {bajo_m:+.3f} a "
    f"{alto_m:+.3f})"
)

,coeficiente,error_estandar,p
Sin controles,-0.219,0.043,0.0000
+ fuente (redes),-0.121,0.043,0.0050
+ retorno esperado,-0.097,0.043,0.0252
+ fondo de emergencia,-0.181,0.041,0.0000
+ fuente y retorno esperado,-0.063,0.043,0.1414


El coeficiente pasa de -0.219 a -0.063: pierde el 71%.


Parte del efecto que pasa por fuente y expectativa: -0.157 (intervalo del 95 %: -0.205 a -0.116)


In [78]:
# La cadena que explica el espejismo

eslabones = pd.DataFrame(
    [
        {
            "relación": "saber menos  →  usar redes sociales como fuente",
            "r": datos_modelo["puntaje_test_conocimiento"].corr(
                datos_modelo["usa_redes"]
            ),
        },
        {
            "relación": "usar redes  →  esperar retornos más altos",
            "r": datos_modelo["usa_redes"].corr(datos_modelo["retorno_esperado"]),
        },
        {
            "relación": "esperar más  →  tomar más riesgo",
            "r": datos_modelo["retorno_esperado"].corr(datos_modelo["riesgo_efectivo"]),
        },
        {
            "relación": "saber menos  →  tomar más riesgo (el espejismo)",
            "r": datos_modelo["puntaje_test_conocimiento"].corr(
                datos_modelo["riesgo_efectivo"]
            ),
        },
    ]
).set_index("relación")
display(eslabones.style.format({"r": "{:+.3f}"}))

print("Perfil de cada fuente de información:")
display(
    comparables.groupby("fuente_informacion", observed=True)
    .agg(
        clientes=("riesgo_efectivo", "size"),
        test_conocimiento=("puntaje_test_conocimiento", "mean"),
        riesgo=("riesgo_efectivo", "mean"),
        volatilidad=("volatilidad_cartera", "mean"),
    )
    .sort_values("test_conocimiento")
    .style.format(
        {"test_conocimiento": "{:.2f}", "riesgo": "{:.2f}", "volatilidad": "{:.2%}"}
    )
)

,r
relación,
saber menos → usar redes sociales como fuente,-0.279
usar redes → esperar retornos más altos,+0.465
esperar más → tomar más riesgo,+0.384
saber menos → tomar más riesgo (el espejismo),-0.212


Perfil de cada fuente de información:


,clientes,test_conocimiento,riesgo,volatilidad
fuente_informacion,,,,
Redes sociales,178,3.69,3.06,12.82%
Amigos y familia,116,4.07,2.44,6.07%
Prensa especializada,65,4.86,2.25,5.47%
Asesor financiero,75,5.06,1.99,4.16%
Investigación propia,92,5.08,2.12,4.90%
Banco o corredor de bolsa,102,5.11,2.20,5.35%


In [79]:
comparables_tabla = comparables.assign(
    nivel_test=pd.cut(
        comparables["puntaje_test_conocimiento"],
        [-0.1, 3, 6, 10],
        labels=["test bajo (0-3)", "test medio (4-6)", "test alto (7-10)"],
    ),
    fuente=np.where(
        comparables["fuente_informacion"] == "Redes sociales",
        "usa redes sociales",
        "otra fuente",
    ),
)

print("ANTES — riesgo efectivo medio sólo por nivel de conocimiento:")
antes_tabla = comparables_tabla.groupby("nivel_test", observed=True)[
    "riesgo_efectivo"
].mean()
display(antes_tabla.to_frame("riesgo_medio").style.format("{:.2f}"))

print("DESPUÉS — el mismo corte, separando por fuente de información:")
medias = comparables_tabla.pivot_table(
    index="nivel_test",
    columns="fuente",
    values="riesgo_efectivo",
    aggfunc="mean",
    observed=True,
)
conteos = comparables_tabla.pivot_table(
    index="nivel_test",
    columns="fuente",
    values="riesgo_efectivo",
    aggfunc="size",
    observed=True,
)
display(medias.style.format("{:.2f}"))
print("n por celda:")
display(conteos)

brecha_entre_fuentes = (medias["usa redes sociales"] - medias["otra fuente"]).mean()
brecha_entre_niveles = (medias.iloc[0] - medias.iloc[-1]).mean()
print(f"\nBrecha media entre fuentes (columnas) : {brecha_entre_fuentes:.2f} puntos")
print(f"Brecha media entre niveles de test (filas): {brecha_entre_niveles:.2f} puntos")
brecha_cruda = antes_tabla.iloc[0] - antes_tabla.iloc[-1]
print(f"Brecha entre test bajo y alto, sin separar: {brecha_cruda:.2f} puntos")
for fuente in medias.columns:
    print(
        f"  dentro de '{fuente}': "
        f"{medias[fuente].iloc[0] - medias[fuente].iloc[-1]:.2f}"
    )
print(
    f"Ojo: la celda 'test alto' con redes tiene {int(conteos.iloc[-1, -1])} clientes."
)

ANTES — riesgo efectivo medio sólo por nivel de conocimiento:


,riesgo_medio
nivel_test,
test bajo (0-3),2.69
test medio (4-6),2.44
test alto (7-10),2.01


DESPUÉS — el mismo corte, separando por fuente de información:


fuente,otra fuente,usa redes sociales
nivel_test,,
test bajo (0-3),2.32,3.17
test medio (4-6),2.27,3.01
test alto (7-10),1.90,2.71


n por celda:


fuente,otra fuente,usa redes sociales
nivel_test,,
test bajo (0-3),102,79
test medio (4-6),273,84
test alto (7-10),62,10



Brecha media entre fuentes (columnas) : 0.80 puntos
Brecha media entre niveles de test (filas): 0.44 puntos
Brecha entre test bajo y alto, sin separar: 0.68 puntos
  dentro de 'otra fuente': 0.42
  dentro de 'usa redes sociales': 0.45
Ojo: la celda 'test alto' con redes tiene 10 clientes.


La relación entre conocimiento financiero y riesgo existe, pero pasa por otro lado. Quien sabe menos se informa más por redes (correlación de −0,28), quien se informa por redes espera retornos más altos (+0,47) y quien espera más toma más riesgo (+0,38). Controlando por la fuente y la expectativa, el efecto del test baja de −0,22 a −0,06 (pierde el 71 %) y deja de ser significativo (p = 0,14).

En el modelo con las trece variables el test queda en −0,10 (p = 0,04 sin corregir, 0,28 con Holm). Separando sólo por fuente, del test bajo al alto el riesgo baja 0,68 puntos sin separar y 0,42 y 0,45 dentro de cada fuente.

Con la corrección, en el modelo completo se sostienen cuatro variables: usar redes, no tener fondo de emergencia, el retorno esperado y la autodeclaración (+0,18, p < 0,001), que algo de información tiene, aunque poca para clasificar. El ingreso no es significativo ni solo ni con las demás. Los años de experiencia suben al controlar (+0,12) pero no pasan la corrección (p = 0,08), y el sexo, significativo solo, deja de serlo con controles. Los meses de gastos cubiertos se desarman, repiten la información de tener o no fondo.

### Inconsistencias internas

In [80]:
# Reglas de inconsistencia, en tres familias

es_agresiva = (
    (comparables["perfil_real"].astype(str) == "Agresivo")
    .reindex(clientes.index)
    .fillna(False)
)

# 1. Respuestas del formulario que se contradicen entre sí
REGLAS_FORMULARIO = {
    "Horizonte menor a 1 año pero invierte para el complemento jubilatorio": (
        clientes["horizonte_inversion"] == "Menos de 1 año"
    )
    & (clientes["objetivo_principal"] == "Complemento jubilatorio"),
    "Dice no aceptar NINGUNA pérdida pero espera más de 30 % anual": (
        clientes["tolerancia_perdida_maxima"] == "No aceptaría ninguna pérdida"
    )
    & (clientes["retorno_esperado_anual"] == "Más de 30%"),
    "Dice no aceptar NINGUNA pérdida pero ante una caída compraría más": (
        clientes["tolerancia_perdida_maxima"] == "No aceptaría ninguna pérdida"
    )
    & (clientes["reaccion_caida_20"] == "Compraría más"),
    "Cero años de experiencia pero se autopercibe conocedor (4 o 5 de 5)": (
        clientes["anios_experiencia_invirtiendo"] == 0
    )
    & (clientes["conocimiento_autopercibido"] >= 4),
}

# 2. La autodeclaración contra otra respuesta. Se reportan, pero no entran a la
#    segmentación, que no puede usar la autodeclaración
REGLAS_AUTODECLARACION = {
    "Se declara Conservador pero espera más de 30 % anual": (
        clientes["perfil_autodeclarado"] == "Conservador"
    )
    & (clientes["retorno_esperado_anual"] == "Más de 30%"),
    "Se declara Agresivo pero vendería todo ante una caída de 20 %": (
        clientes["perfil_autodeclarado"] == "Agresivo"
    )
    & (clientes["reaccion_caida_20"] == "Vendería todo"),
}

# 3. Una respuesta contra la cartera real
REGLAS_CONTRA_CARTERA = {
    "Declara NO tener fondo de emergencia pero guarda dólares en caja": (
        clientes["tiene_fondo_emergencia"] == "No"
    )
    & (clientes["efectivo_usd"] > 0),
    "Su objetivo es preservar el capital pero su cartera es Agresiva": (
        clientes["objetivo_principal"] == "Preservar el capital"
    )
    & es_agresiva,
    "Tolera hasta un 5 % de pérdida pero su cartera tiene más de 15 % de volatilidad": (
        clientes["tolerancia_perdida_maxima"].isin(
            ["Hasta 5%", "No aceptaría ninguna pérdida"]
        )
    )
    & (clientes["volatilidad_cartera"] > 0.15),
    "Dice no aceptar ninguna pérdida pero su cartera es Agresiva": (
        clientes["tolerancia_perdida_maxima"] == "No aceptaría ninguna pérdida"
    )
    & es_agresiva,
}

# Quedó afuera "horizonte de más de 5 años y mira la cartera todos los días": es una
# tensión, no una contradicción (se puede invertir a largo plazo y mirar seguido)

FAMILIAS = {
    "formulario": REGLAS_FORMULARIO,
    "autodeclaración": REGLAS_AUTODECLARACION,
    "contra la cartera": REGLAS_CONTRA_CARTERA,
}
banderas_por_familia = {
    familia: pd.DataFrame(reglas).fillna(False) for familia, reglas in FAMILIAS.items()
}
banderas = pd.concat(banderas_por_familia.values(), axis=1)

resumen = pd.concat(
    [
        pd.DataFrame({"familia": familia, "clientes": tabla.sum()})
        for familia, tabla in banderas_por_familia.items()
    ]
)
resumen["pct_base"] = resumen["clientes"] / len(clientes)
display(resumen.style.format({"pct_base": "{:.1%}"}))

clientes["inconsistencias"] = banderas.sum(axis=1)
clientes["inconsistencias_formulario"] = banderas_por_familia["formulario"].sum(axis=1)
for familia, tabla in banderas_por_familia.items():
    alguna = int((tabla.sum(axis=1) > 0).sum())
    print(
        f"Con al menos una de la familia '{familia}': {alguna} "
        f"({alguna / len(clientes):.1%})"
    )
al_menos_una = int((clientes["inconsistencias"] > 0).sum())
print(
    f"\nClientes con AL MENOS UNA inconsistencia: {al_menos_una} de {len(clientes)} "
    f"({al_menos_una / len(clientes):.1%})"
)

,familia,clientes,pct_base
Horizonte menor a 1 año pero invierte para el complemento jubilatorio,formulario,9,1.4%
Dice no aceptar NINGUNA pérdida pero espera más de 30 % anual,formulario,6,0.9%
Dice no aceptar NINGUNA pérdida pero ante una caída compraría más,formulario,26,4.1%
Cero años de experiencia pero se autopercibe conocedor (4 o 5 de 5),formulario,3,0.5%
Se declara Conservador pero espera más de 30 % anual,autodeclaración,36,5.6%
Se declara Agresivo pero vendería todo ante una caída de 20 %,autodeclaración,34,5.3%
Declara NO tener fondo de emergencia pero guarda dólares en caja,contra la cartera,73.000000,11.4%
Su objetivo es preservar el capital pero su cartera es Agresiva,contra la cartera,14.000000,2.2%
Tolera hasta un 5 % de pérdida pero su cartera tiene más de 15 % de volatilidad,contra la cartera,7.000000,1.1%
Dice no aceptar ninguna pérdida pero su cartera es Agresiva,contra la cartera,4.000000,0.6%


Con al menos una de la familia 'formulario': 44 (6.9%)
Con al menos una de la familia 'autodeclaración': 70 (10.9%)
Con al menos una de la familia 'contra la cartera': 93 (14.5%)

Clientes con AL MENOS UNA inconsistencia: 192 de 640 (30.0%)


In [81]:
# La inconsistencia predice el desajuste?
comparables["inconsistencias_formulario"] = clientes[
    "inconsistencias_formulario"
].reindex(comparables.index)

por_direccion = (
    comparables.groupby(np.sign(comparables["distancia"]))
    .agg(
        clientes=("inconsistencias_formulario", "size"),
        inconsistencias_promedio=("inconsistencias_formulario", "mean"),
        pct_con_alguna=("inconsistencias_formulario", lambda s: (s > 0).mean()),
    )
    .rename(index={-1: "subexpuesto", 0: "coincide", 1: "sobreexpuesto"})
)
display(
    por_direccion.style.format(
        {"inconsistencias_promedio": "{:.2f}", "pct_con_alguna": "{:.1%}"}
    )
)

banderas.assign(total=banderas.sum(axis=1)).to_csv(
    SALIDAS / "3_inconsistencias_por_cliente.csv", encoding="utf-8-sig"
)
print(f"Detalle guardado en {SALIDAS / '3_inconsistencias_por_cliente.csv'}")

,clientes,inconsistencias_promedio,pct_con_alguna
distancia,,,
subexpuesto,177,0.12,12.4%
coincide,313,0.06,5.8%
sobreexpuesto,138,0.03,2.9%


Detalle guardado en C:\Users\prisc\Downloads\prog analitica II\proyecto-final-gerlach\salidas\3_inconsistencias_por_cliente.csv


El 30 % de los clientes tiene al menos una inconsistencia, sobre todo contra la cartera (14,5 %) o contra la autodeclaración (10,9 %). Las del formulario consigo mismo son pocas: 44 clientes, el 6,9 %.

No anticipan la sobreexposición: las tiene el 2,9 % de los sobreexpuestos, contra el 5,8 % de los que coinciden y el 12,4 % de los subexpuestos. Sirven como pregunta de control en el formulario, no como alerta de riesgo legal.

## Resumen de la etapa

- El 50,2 % de los clientes con cartera tiene un perfil real distinto del declarado. El kappa ponderado es 0,25 (débil; 0,18 sin ponderar, insignificante) y no llega a 0,40 con ningún par de cortes.
- Hay 138 sobreexpuestos (22 %); entre ellos, 22 conservadores con cartera agresiva y USD 435.579 en juego. Se informan más por redes y esperan más. Los 177 subexpuestos son un tema comercial.
- Los 113 que esperan más de 30 % obtienen 11,3 % nominal en pesos y 4,7 % real.
- Lo que predice el riesgo es de dónde se informa el cliente y qué espera, no cuánto sabe.
- El 30 % tiene al menos una respuesta que no cuadra con otra o con su cartera.

# Visualización

## Radar de los perfiles reales

Cada polígono es un perfil sobre siete dimensiones. Para poner montos, puntajes y porcentajes en el mismo dibujo, llevé cada eje a una escala de 0 a 1 entre el percentil 5 y el 95. Por eso el área no significa nada y los ejes no se comparan entre sí: lo que se lee es en qué dimensión un perfil se separa de otro.

Dejé afuera la volatilidad, que es casi igual al riesgo efectivo, y la concentración, que no distingue perfiles.

In [82]:
# Función reutilizable de radar: se vuelve a usar con los perfiles nuevos

# Dimensión visible -> columna de la tabla de clientes
DIMENSIONES_RADAR = {
    "Riesgo de la cartera": "riesgo_efectivo",
    "Retorno esperado": "retorno_esperado",
    "Usa redes sociales": "usa_redes",
    "Peso en cripto": "peso_cripto",
    "Cartera en la firma": "cartera_usd",
    "Conocimiento (test)": "puntaje_test_conocimiento",
    "Cobertura de emergencia": "meses_gastos_cubiertos",
}


def normalizar_por_percentiles(
    tabla: pd.DataFrame,
    columnas: list[str],
    inferior: float = 0.05,
    superior: float = 0.95,
) -> pd.DataFrame:
    """Lleva cada columna al rango 0-1 usando los percentiles 5 y 95 de la población.

    Se usan percentiles y no el mínimo y el máximo para que un solo caso extremo no
    aplaste al resto del eje.
    """
    normalizada = pd.DataFrame(index=tabla.index)
    for columna in columnas:
        bajo, alto = tabla[columna].quantile([inferior, superior])
        if alto == bajo:
            alto = tabla[columna].max() or 1.0
        normalizada[columna] = ((tabla[columna] - bajo) / (alto - bajo)).clip(0, 1)
    return normalizada


def datos_radar(
    tabla: pd.DataFrame, columna_grupo: str, orden_grupos: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Devuelve (valores normalizados por grupo, medias sin normalizar)."""
    columnas = list(DIMENSIONES_RADAR.values())
    normalizada = normalizar_por_percentiles(tabla, columnas)
    normalizada[columna_grupo] = tabla[columna_grupo].astype(str)

    promedios = (
        normalizada.groupby(columna_grupo)[columnas].mean().reindex(orden_grupos)
    )
    promedios.columns = list(DIMENSIONES_RADAR)
    crudos = (
        tabla.groupby(columna_grupo, observed=True)[columnas]
        .mean()
        .reindex(orden_grupos)
    )
    crudos.columns = list(DIMENSIONES_RADAR)
    return promedios, crudos


def ficha_perfil(grupo: str, crudos: pd.DataFrame, conteos: pd.Series) -> str:
    """Texto del cartelito: los valores reales del perfil, en sus unidades."""
    fila = crudos.loc[grupo]
    return (
        f"<b>{grupo}</b> · {int(conteos[grupo])} clientes<br>"
        f"Riesgo de la cartera: {formato_es(fila['Riesgo de la cartera'], 2)} (escala "
        f"1-5)<br>"
        f"Retorno esperado: {formato_es(fila['Retorno esperado'] * 100, 1)} % anual<br>"
        f"Usa redes sociales: {formato_es(fila['Usa redes sociales'] * 100)} % del "
        f"grupo<br>"
        f"Peso en cripto: {formato_es(fila['Peso en cripto'] * 100, 1)} % de la "
        f"cartera<br>"
        f"Cartera media en la firma: USD {formato_es(fila['Cartera en la firma'])}<br>"
        f"Conocimiento (test): {formato_es(fila['Conocimiento (test)'], 1)} de 10<br>"
        f"Cobertura de emergencia: {formato_es(fila['Cobertura de emergencia'], 1)} "
        f"meses"
    )


def radar_plotly(
    promedios: pd.DataFrame,
    colores: dict[str, str],
    titulo: str,
    subtitulo: str,
    crudos: pd.DataFrame,
    conteos: pd.Series,
) -> go.Figure:
    """Radar interactivo: al pasar el mouse por la red aparece la ficha del perfil.

    Plotly sólo muestra el cartel sobre los puntos, así que además de los siete
    vértices se agregan puntos invisibles a lo largo de cada lado del polígono: el
    cartel acompaña al mouse mientras recorre la red, no sólo en las puntas.
    """
    figura = go.Figure()
    ejes = list(promedios.columns)
    angulos = [i * 360 / len(ejes) for i in range(len(ejes))]
    pasos = 12  # puntos invisibles por lado

    for grupo in promedios.index:
        valores = promedios.loc[grupo].tolist()
        ficha = ficha_perfil(grupo, crudos, conteos)

        # Polígono visible con un punto por vértice
        figura.add_trace(
            go.Scatterpolar(
                r=valores + [valores[0]],
                theta=angulos + [angulos[0]],
                name=str(grupo),
                mode="lines+markers",
                line=dict(color=colores[grupo], width=2),
                marker=dict(size=8, color=colores[grupo]),
                fill="toself",
                fillcolor="rgba({}, {}, {}, 0.15)".format(
                    *[int(v * 255) for v in mcolors.to_rgb(colores[grupo])]
                ),
                text=[f"{ficha}<br><i>vértice: {eje}</i>" for eje in ejes + ejes[:1]],
                hovertemplate="%{text}<extra></extra>",
                legendgroup=str(grupo),
            )
        )

        # Puntos invisibles sobre los lados, interpolados en coordenadas cartesianas
        # para que caigan sobre la línea recta entre dos vértices
        x = np.array(valores) * np.cos(np.radians(angulos))
        y = np.array(valores) * np.sin(np.radians(angulos))
        r_lado, theta_lado = [], []
        for k in range(len(ejes)):
            sig = (k + 1) % len(ejes)
            for f in np.linspace(0, 1, pasos, endpoint=False)[1:]:
                xi = x[k] + f * (x[sig] - x[k])
                yi = y[k] + f * (y[sig] - y[k])
                r_lado.append(float(np.hypot(xi, yi)))
                theta_lado.append(float(np.degrees(np.arctan2(yi, xi)) % 360))
        figura.add_trace(
            go.Scatterpolar(
                r=r_lado,
                theta=theta_lado,
                mode="markers",
                marker=dict(size=14, color=colores[grupo], opacity=0),
                hovertemplate=ficha + "<extra></extra>",
                hoverlabel=dict(bgcolor="white", bordercolor=colores[grupo]),
                showlegend=False,
                legendgroup=str(grupo),
            )
        )

    figura.update_traces(
        hoverlabel=dict(
            bgcolor="white", font=dict(size=12, color=TINTA_PRIMARIA), align="left"
        )
    )
    figura.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><sup>{subtitulo}</sup>",
            x=0.02,
            font=dict(size=14, color=TINTA_PRIMARIA),
        ),
        polar=dict(
            bgcolor=SUPERFICIE,
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickvals=[0, 0.25, 0.5, 0.75, 1],
                tickfont=dict(size=8, color=TINTA_SECUNDARIA),
                gridcolor=GRIS_GRILLA,
                linecolor=GRIS_GRILLA,
            ),
            angularaxis=dict(
                tickmode="array",
                tickvals=angulos,
                ticktext=ejes,
                rotation=90,
                direction="counterclockwise",
                tickfont=dict(size=10, color=TINTA_SECUNDARIA),
                gridcolor=GRIS_GRILLA,
                linecolor=GRIS_GRILLA,
            ),
        ),
        hovermode="closest",
        hoverdistance=40,
        showlegend=True,
        legend=dict(
            orientation="h",
            y=-0.08,
            x=0.5,
            xanchor="center",
            font=dict(color=TINTA_SECUNDARIA),
        ),
        paper_bgcolor=SUPERFICIE,
        width=760,
        height=640,
        margin=dict(t=90, b=90, l=90, r=90),
    )
    return figura


print("Funciones de radar definidas.")

Funciones de radar definidas.


In [83]:
# Radar de los tres perfiles reales

base_radar = comparables.assign(
    retorno_esperado=comparables["retorno_esperado_anual"].map(PISO_DEL_RANGO),
    usa_redes=(comparables["fuente_informacion"] == "Redes sociales").astype(float),
    perfil=comparables["perfil_real"].astype(str),
)

promedios_radar, crudos_radar = datos_radar(base_radar, "perfil", ETIQUETAS_PERFIL)

figura_radar = radar_plotly(
    promedios_radar,
    COLOR_PERFIL,
    titulo="Los perfiles de cartera se separan por comportamiento, no por tamaño",
    subtitulo=f"{len(base_radar)} clientes con cartera · cada eje entre el "
    "percentil 5 y el 95 · pasá el mouse por la red para ver la ficha · datos al "
    "30/06/2026",
    crudos=crudos_radar,
    conteos=base_radar["perfil"].value_counts(),
)
figura_radar.show()

In [84]:
# La tabla que acompaña al radar: valores sin normalizar

presentacion = crudos_radar.T.copy()
presentacion.index.name = "dimensión (unidades originales)"
display(
    presentacion.style.format(
        {"Conservador": "{:,.3f}", "Moderado": "{:,.3f}", "Agresivo": "{:,.3f}"}
    )
)

print("Unidades: riesgo 1 a 5 · retorno esperado y peso en cripto en tanto por uno ·")
print("usa redes como proporción del grupo · cartera en la firma en USD ·")
print("test de 0 a 10 ·", end=" ")
print("cobertura de emergencia en meses de gastos.")
print()
print("Valores normalizados que dibuja el radar:")
display(promedios_radar.T.style.format("{:.3f}"))

perfil,Conservador,Moderado,Agresivo
dimensión (unidades originales),,,
Riesgo de la cartera,1.655,2.970,4.147
Retorno esperado,0.087,0.129,0.192
Usa redes sociales,0.152,0.351,0.594
Peso en cripto,0.001,0.009,0.164
Cartera en la firma,"18,014.553","22,664.799","18,146.145"
Conocimiento (test),4.812,4.176,3.949
Cobertura de emergencia,3.488,2.378,1.364


Unidades: riesgo 1 a 5 · retorno esperado y peso en cripto en tanto por uno ·
usa redes como proporción del grupo · cartera en la firma en USD ·
test de 0 a 10 · cobertura de emergencia en meses de gastos.

Valores normalizados que dibuja el radar:


perfil,Conservador,Moderado,Agresivo
Riesgo de la cartera,0.200,0.600,0.916
Retorno esperado,0.290,0.429,0.640
Usa redes sociales,0.152,0.351,0.594
Peso en cripto,0.004,0.038,0.305
Cartera en la firma,0.243,0.298,0.259
Conocimiento (test),0.543,0.457,0.426
Cobertura de emergencia,0.314,0.213,0.121


Los perfiles se separan por comportamiento (riesgo, cripto, redes y retorno esperado) y no por tamaño de cartera, que es parecido en los tres: USD 18.015, 22.665 y 18.146. El conocimiento cambia poco (4,8, 4,2 y 3,9). El agresivo es el que menos colchón tiene: 1,4 meses de gastos contra 3,5 del conservador.

## Declarado vs. real

Una barra por perfil declarado, partida según lo que hace su cartera. 

In [85]:
# Lo que dijo ser contra lo que su cartera hace

cruce = pd.crosstab(
    comparables["perfil_autodeclarado"], comparables["perfil_real"].astype(str)
).reindex(index=ETIQUETAS_PERFIL, columns=ETIQUETAS_PERFIL)
cruce_pct = cruce.div(cruce.sum(axis=1), axis=0)
monto_cruce = (
    comparables.assign(real=comparables["perfil_real"].astype(str))
    .pivot_table(
        index="perfil_autodeclarado",
        columns="real",
        values="cartera_usd",
        aggfunc="sum",
    )
    .reindex(index=ETIQUETAS_PERFIL, columns=ETIQUETAS_PERFIL)
    .fillna(0)
)
CARTERA_FEMENINO = {
    "Conservador": "conservadora",
    "Moderado": "moderada",
    "Agresivo": "agresiva",
}
PLURAL = {
    "Conservador": "conservadores",
    "Moderado": "moderados",
    "Agresivo": "agresivos",
}
filas_barra = [f"Se declaró<br>{p.lower()}" for p in ETIQUETAS_PERFIL]

figura = go.Figure()
for real in ETIQUETAS_PERFIL:
    coincide = [real == declarado for declarado in ETIQUETAS_PERFIL]
    textos, hovers = [], []
    for declarado in ETIQUETAS_PERFIL:
        pct = cruce_pct.loc[declarado, real]
        n = int(cruce.loc[declarado, real])
        textos.append(
            f"{formato_es(pct * 100)} %<br>{n} clientes"
            if pct >= 0.1
            else f"{formato_es(pct * 100)} %<br>{n}"
        )
        distancia = ORDEN_PERFIL[real] - ORDEN_PERFIL[declarado]
        lectura = (
            "Coincide con lo que declaró"
            if distancia == 0
            else (
                "Sobreexpuesto: riesgo legal"
                if distancia > 0
                else "Subexpuesto: oportunidad comercial"
            )
        )
        hovers.append(
            f"<b>Se declaró {declarado.lower()}, su cartera es "
            f"{CARTERA_FEMENINO[real]}</b><br>"
            f"{n} clientes ({formato_es(pct * 100)} % de los que se declararon "
            f"{PLURAL[declarado]})<br>"
            f"Cartera: USD {formato_es(monto_cruce.loc[declarado, real])}<br>"
            f"<i>{lectura}</i>"
        )
    figura.add_trace(
        go.Bar(
            name=real,
            y=filas_barra,
            x=cruce_pct[real].values,
            orientation="h",
            marker=dict(
                color=[
                    COLOR_PERFIL[real] if c else aclarar(COLOR_PERFIL[real])
                    for c in coincide
                ],
                line=dict(
                    color=[TINTA_PRIMARIA if c else SUPERFICIE for c in coincide],
                    width=[3 if c else 1.5 for c in coincide],
                ),
            ),
            text=textos,
            textposition="inside",
            insidetextanchor="middle",
            textangle=0,
            textfont=dict(color=TINTA_PRIMARIA, size=12.5),
            hovertext=hovers,
            hovertemplate="%{hovertext}<extra></extra>",
            showlegend=False,
        )
    )
    # Muestra de la leyenda con el color lleno, sin borde
    figura.add_trace(
        go.Bar(
            name=real,
            y=[None],
            x=[None],
            orientation="h",
            marker_color=COLOR_PERFIL[real],
            hoverinfo="skip",
        )
    )

for declarado, fila_barra in zip(ETIQUETAS_PERFIL, filas_barra):
    figura.add_annotation(
        x=1.01,
        y=fila_barra,
        xanchor="left",
        showarrow=False,
        align="left",
        text=(
            f"<b>{int(cruce.loc[declarado, declarado])} de "
            f"{int(cruce.loc[declarado].sum())}</b><br>coinciden"
        ),
    )
figura.update_layout(
    barmode="stack",
    bargap=0.3,
    legend=dict(
        title="Perfil real:",
        orientation="h",
        y=-0.2,
        x=0,
        yanchor="top",
        traceorder="normal",
    ),
    margin=dict(r=110),
)
figura.update_xaxes(
    tickformat=".0%",
    range=[0, 1],
    showgrid=False,
    title="% de los clientes de cada perfil declarado",
)
figura.update_yaxes(autorange="reversed", showgrid=False, title="Perfil declarado")
presentar(
    figura,
    "Sólo la mitad de los clientes invierte como dijo que invertiría",
    f"{int(cruce.values.sum())} clientes con cartera · el tramo con borde "
    "es el que coincide · pasá el mouse para ver clientes y monto",
    alto=470,
)

Sólo coincide el 49,8 %. En la barra de los conservadores, el último tramo son los 22 con cartera agresiva: el riesgo legal. La de los agresivos es la más repartida: de 137, sólo 47 tienen cartera agresiva. El gráfico no permite concluir que el cliente mienta: no sabemos si cada compra la decidió él o su asesor.

## Conocimiento y riesgo: antes y después de controlar

In [86]:
# Antes y después: el coeficiente del test de conocimiento en cada modelo

colores_paso = [
    COLOR_PERFIL["Conservador"] if p < 0.05 else GRIS_NEUTRO for p in pasos["p"]
]
etiquetas_paso = [
    f"{nombre} · {formato_es(v, 2)} "
    f"(p {'< 0,001' if p < 0.001 else '= ' + formato_es(p, 2)})"
    for nombre, v, p in zip(pasos.index, pasos["coeficiente"], pasos["p"])
]
figura = go.Figure(
    go.Bar(
        x=pasos["coeficiente"],
        y=etiquetas_paso,
        orientation="h",
        marker_color=colores_paso,
        error_x=dict(
            type="data",
            array=1.96 * pasos["error_estandar"],
            color=TINTA_SECUNDARIA,
            thickness=1.5,
        ),
    )
)
figura.add_vline(x=0, line_color=TINTA_SECUNDARIA, line_width=1)
figura.update_xaxes(
    title="Cambio en el riesgo por un desvío más de test (intervalo del 95 %)",
    range=[pasos["coeficiente"].min() - 0.15, 0.12],
)
figura.update_yaxes(autorange="reversed", showgrid=False)
presentar(
    figura,
    "El efecto del conocimiento sobre el riesgo se desarma al controlar por fuente y "
    "expectativa",
    f"Regresión sobre {int(X.shape[0])} clientes con todos los datos · en "
    "gris, no significativo",
    alto=430,
    interactivo=False,
)

Sin controles, un desvío más de puntaje en el test se asocia con 0,22 puntos menos de riesgo. Controlando por la fuente de información baja a 0,12; por el retorno esperado, a 0,10; con los dos juntos, a 0,06, que ya no se distingue de cero. Controlar por el fondo de emergencia casi no lo mueve (0,18): la relación pasa por la fuente y la expectativa, no por la situación financiera. No quiere decir que el conocimiento no importe, sino que actúa a través de dónde se informa el cliente y qué espera.

## Retorno esperado vs. obtenido

In [87]:
# Brecha entre el retorno esperado declarado y el que rinde la cartera

tramos = expectativas.dropna(subset=["rinde_nominal"])
etiquetas_tramo = [
    f"{tramo_esperado}<br><span style='font-size:11px'>{int(n)} clientes</span>"
    for tramo_esperado, n in zip(tramos.index, tramos["clientes"])
]
hovers = [
    f"<b>Esperan {tramo_esperado.lower()} · {int(fila.clientes)} clientes</b><br>"
    f"Esperan al menos: {formato_es(fila.piso_esperado * 100)} % anual<br>"
    f"Su cartera rinde (nominal en pesos): {formato_es(fila.rinde_nominal * 100, 1)} % "
    f"({formato_es(fila.rinde_real * 100, 1)} % descontando inflación)<br>"
    f"Brecha: {formato_es(fila.brecha * 100, 1)} puntos"
    for tramo_esperado, fila in tramos.iterrows()
]

figura = go.Figure()
# Una línea gris por tramo, entre lo que esperan y lo que obtienen
segmentos_x, segmentos_y = [], []
for etiqueta, piso, rinde in zip(
    etiquetas_tramo, tramos["piso_esperado"], tramos["rinde_nominal"]
):
    segmentos_x += [etiqueta, etiqueta, None]
    segmentos_y += [piso, rinde, None]
figura.add_trace(
    go.Scatter(
        x=segmentos_x,
        y=segmentos_y,
        mode="lines",
        line=dict(color=GRIS_GRILLA, width=4),
        hoverinfo="skip",
        showlegend=False,
    )
)
for columna, nombre, color in [
    ("piso_esperado", "Retorno esperado (piso del tramo)", COLOR_PERFIL["Moderado"]),
    (
        "rinde_nominal",
        "Retorno nominal de la cartera, en pesos",
        COLOR_PERFIL["Conservador"],
    ),
]:
    figura.add_trace(
        go.Scatter(
            x=etiquetas_tramo,
            y=tramos[columna],
            mode="markers",
            name=nombre,
            marker=dict(size=16, color=color, line=dict(color="white", width=2)),
            hovertext=hovers,
            hovertemplate="%{hovertext}<extra></extra>",
        )
    )
ultimo = tramos.iloc[-1]
figura.add_annotation(
    x=etiquetas_tramo[-1],
    y=(ultimo["piso_esperado"] + ultimo["rinde_nominal"]) / 2,
    text=f"<b>{formato_es(ultimo['brecha'] * 100)} puntos</b><br>de brecha",
    showarrow=False,
    xshift=-12,
    xanchor="right",
    align="right",
    font=dict(color=TINTA_PRIMARIA),
)
figura.update_yaxes(tickformat=".0%", title="Retorno anual", range=[-0.02, 0.33])
figura.update_xaxes(title="Retorno anual esperado declarado", showgrid=False)
figura.update_layout(legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
presentar(
    figura,
    f"Los que esperan más de 30 % obtienen "
    f"{formato_es(ultimo['rinde_nominal'] * 100, 1)} %: sólo "
    f"{int((nominal_exigentes > piso_exigentes).sum())} de {len(mas_exigentes)} "
    "llegan a su piso",
    "628 clientes con cartera · retorno ponderado por monto · "
    f"descontando inflación, el tramo más alto obtiene "
    f"{formato_es(ultimo['rinde_real'] * 100, 1)} %",
    interactivo=False,
    alto=470,
)

display(
    tramos[
        ["clientes", "piso_esperado", "rinde_nominal", "rinde_real", "brecha"]
    ].style.format(
        {
            "piso_esperado": "{:.0%}",
            "rinde_nominal": "{:.2%}",
            "rinde_real": "{:.2%}",
            "brecha": "{:+.2%}",
        }
    )
)

,clientes,piso_esperado,rinde_nominal,rinde_real,brecha
retorno_esperado_anual,,,,,
Menos de 5%,144,0%,8.93%,2.78%,-8.93%
5% a 10%,139,5%,9.29%,3.10%,-4.29%
10% a 20%,140,10%,9.09%,2.99%,+0.91%
20% a 30%,92,20%,10.56%,4.11%,+9.44%
Más de 30%,113,30%,11.29%,4.65%,+18.71%


La brecha se abre a medida que sube la expectativa. Quien espera menos de 5 % obtiene 8,9 % nominal en pesos; quien espera más de 30 % obtiene 11,3 % (4,7 % real). Sólo 5 de los 113 llegan a su piso, y en términos reales ninguno: ni la cripto, lo de mayor retorno del catálogo con 21,5 % real, llega a 30 %. No quiere decir que nunca vayan a ganar 30 % en un año (con 65 % de volatilidad la cripto puede), sino que no es lo esperable.

## Composición de la cartera por perfil

In [88]:
# Composición de la cartera por familia de activo y perfil

# Cada posición hereda el perfil real del cliente que la tiene
posiciones_con_perfil = posiciones.assign(
    perfil_real=posiciones["id_cliente"].map(comparables["perfil_real"].astype(str))
)
posiciones_con_perfil = posiciones_con_perfil[
    posiciones_con_perfil["perfil_real"].notna()
]

# Para el gráfico se agrupan los cinco niveles de riesgo en tres bandas: con cinco
# tonos del mismo color no se distinguen a simple vista. El detalle por familia de
# activo aparece al pasar el mouse, y los cinco niveles quedan en la tabla de abajo
ETIQUETA_RIESGO = {
    1: "1 · muy conservador",
    2: "2 · conservador",
    3: "3 · moderado",
    4: "4 · agresivo",
    5: "5 · muy agresivo",
}
BANDA_RIESGO = {
    1: "Riesgo bajo",
    2: "Riesgo bajo",
    3: "Riesgo medio",
    4: "Riesgo alto",
    5: "Riesgo alto",
}
COLOR_BANDA = {
    "Riesgo bajo": "#d6e6f8",
    "Riesgo medio": "#6ba6e3",
    "Riesgo alto": "#1b5ba3",
}
TEXTO_BANDA = {
    "Riesgo bajo": TINTA_PRIMARIA,
    "Riesgo medio": TINTA_PRIMARIA,
    "Riesgo alto": "white",
}
NIVELES_BANDA = {
    "Riesgo bajo": "niveles 1 y 2",
    "Riesgo medio": "nivel 3",
    "Riesgo alto": "niveles 4 y 5",
}

composicion_perfil = (
    posiciones_con_perfil.groupby(["perfil_real", "nivel_riesgo"], observed=True)[
        "monto_cartera"
    ]
    .sum()
    .unstack(fill_value=0)
)
composicion_perfil = composicion_perfil.div(composicion_perfil.sum(axis=1), axis=0)
composicion_perfil = composicion_perfil.reindex(ETIQUETAS_PERFIL).sort_index(axis=1)
composicion_perfil.columns = [ETIQUETA_RIESGO[n] for n in composicion_perfil.columns]

con_banda = posiciones_con_perfil.assign(
    banda=posiciones_con_perfil["nivel_riesgo"].map(BANDA_RIESGO)
)
total_perfil = con_banda.groupby("perfil_real")["monto_cartera"].sum()
por_banda = con_banda.groupby(["perfil_real", "banda"])["monto_cartera"].sum()
por_familia = con_banda.groupby(["perfil_real", "banda", "categoria"])[
    "monto_cartera"
].sum()


def detalle_familias_banda(perfil: str, banda: str) -> str:
    """Las familias de activo que componen una banda, para el cartel del mouse."""
    if (perfil, banda) not in por_banda.index:
        return ""
    familias = por_familia.loc[(perfil, banda)].sort_values(ascending=False)
    return "<br>".join(
        f"  {familia}: {formato_es(monto / total_perfil[perfil] * 100, 1)} %"
        for familia, monto in familias.items()
        if monto / total_perfil[perfil] >= 0.005
    )


es_muy_iliquido = con_banda["liquidez"] == "Muy baja"
iliquido = con_banda[es_muy_iliquido].groupby("perfil_real")["monto_cartera"].sum()
iliquido_pct = (iliquido / total_perfil).reindex(ETIQUETAS_PERFIL).fillna(0)
instrumentos_iliquidos = (
    con_banda[es_muy_iliquido]
    .groupby(["perfil_real", "nombre_instrumento"])["monto_cartera"]
    .sum()
)

figura = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    column_widths=[0.66, 0.34],
    horizontal_spacing=0.05,
    subplot_titles=[
        "<b>Composición por nivel de riesgo</b>",
        "<b>Liquidez muy baja</b> (% de la cartera)",
    ],
)
for banda in ["Riesgo bajo", "Riesgo medio", "Riesgo alto"]:
    valores = [
        por_banda.get((perfil, banda), 0) / total_perfil[perfil]
        for perfil in ETIQUETAS_PERFIL
    ]
    figura.add_trace(
        go.Bar(
            y=ETIQUETAS_PERFIL,
            x=valores,
            name=f"{banda} ({NIVELES_BANDA[banda]})",
            orientation="h",
            marker=dict(color=COLOR_BANDA[banda], line=dict(color=SUPERFICIE, width=2)),
            text=[f"{v:.0%}" if v >= 0.07 else "" for v in valores],
            textposition="inside",
            insidetextanchor="middle",
            textfont=dict(color=TEXTO_BANDA[banda], size=13),
            hovertext=[
                f"<b>{perfil} · {banda.lower()}</b><br>"
                f"{formato_es(v * 100, 1)} % de la cartera:<br>"
                + detalle_familias_banda(perfil, banda)
                for perfil, v in zip(ETIQUETAS_PERFIL, valores)
            ],
            hovertemplate="%{hovertext}<extra></extra>",
        ),
        row=1,
        col=1,
    )
figura.add_trace(
    go.Bar(
        y=ETIQUETAS_PERFIL,
        x=iliquido_pct.values,
        orientation="h",
        marker_color=[COLOR_PERFIL[p] for p in ETIQUETAS_PERFIL],
        text=[f"{formato_es(v * 100, 1)} %" for v in iliquido_pct],
        textposition="outside",
        hovertext=[
            f"<b>{perfil}</b><br>USD {formato_es(iliquido.get(perfil, 0))} en "
            "instrumentos de liquidez muy baja:<br>"
            + "<br>".join(
                f"  {nombre}: USD {formato_es(monto)}"
                for nombre, monto in instrumentos_iliquidos.loc[perfil]
                .sort_values(ascending=False)
                .items()
            )
            for perfil in ETIQUETAS_PERFIL
        ],
        hovertemplate="%{hovertext}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=2,
)
figura.update_layout(
    barmode="stack",
    bargap=0.3,
    legend=dict(orientation="h", y=-0.2, x=0, yanchor="top", traceorder="normal"),
)
figura.update_annotations(font=dict(size=13, color=TINTA_PRIMARIA))
figura.update_yaxes(autorange="reversed", showgrid=False)
figura.update_xaxes(tickformat=".0%", showgrid=False, title="% de la cartera")
figura.update_xaxes(range=[0, 1], col=1)
figura.update_xaxes(range=[0, iliquido_pct.max() * 1.35], col=2)
presentar(
    figura,
    "Cada perfil concentra su cartera en un nivel de riesgo, y el moderado es el menos "
    "líquido",
    "628 clientes con cartera · % del monto invertido · pasá el mouse "
    "para ver el detalle por instrumento",
    alto=470,
    ancho=1000,
)

display(composicion_perfil.style.format("{:.1%}"))

print("Detalle por familia de activo, que el gráfico resume en cinco niveles:")
detalle_familias = (
    posiciones_con_perfil.groupby(["perfil_real", "categoria"], observed=True)[
        "monto_cartera"
    ]
    .sum()
    .unstack(fill_value=0)
)
detalle_familias = detalle_familias.div(detalle_familias.sum(axis=1), axis=0).reindex(
    ETIQUETAS_PERFIL
)
orden_familias = (
    catalogo.groupby("categoria")["nivel_riesgo"].mean().sort_values().index.tolist()
)
detalle_familias = detalle_familias[
    [c for c in orden_familias if c in detalle_familias.columns]
]
display(detalle_familias.style.format("{:.1%}"))

# El nivel de riesgo mide variación de precio, no la facilidad para salir. Son dos
# riesgos distintos y conviene mirarlos juntos
print("Y la otra cara del riesgo, que la escala de la firma no mide: la liquidez.")
ORDEN_LIQUIDEZ = ["Alta", "Media", "Baja", "Muy baja"]
liquidez_perfil = (
    posiciones_con_perfil.groupby(["perfil_real", "liquidez"], observed=True)[
        "monto_cartera"
    ]
    .sum()
    .unstack(fill_value=0)
)
liquidez_perfil = liquidez_perfil.div(liquidez_perfil.sum(axis=1), axis=0).reindex(
    index=ETIQUETAS_PERFIL
)[[c for c in ORDEN_LIQUIDEZ if c in liquidez_perfil.columns]]
display(liquidez_perfil.style.format("{:.1%}"))

,1 · muy conservador,2 · conservador,3 · moderado,4 · agresivo,5 · muy agresivo
perfil_real,,,,,
Conservador,54.2%,29.1%,15.4%,0.9%,0.4%
Moderado,4.3%,9.8%,71.6%,12.6%,1.7%
Agresivo,1.9%,1.1%,15.1%,54.9%,27.0%


Detalle por familia de activo, que el gráfico resume en cinco niveles:


categoria,Efectivo,Renta fija local,Fondos,Inmobiliario,Renta fija corporativa,Previsional,Materias primas,Renta variable,Cripto
perfil_real,,,,,,,,,
Conservador,6.7%,59.9%,16.7%,1.5%,5.9%,8.0%,0.3%,0.7%,0.2%
Moderado,0.5%,10.5%,3.3%,7.0%,39.2%,25.4%,8.4%,4.7%,1.2%
Agresivo,0.1%,2.6%,0.4%,0.0%,7.7%,7.4%,33.1%,37.1%,11.6%


Y la otra cara del riesgo, que la escala de la firma no mide: la liquidez.


liquidez,Alta,Media,Baja,Muy baja
perfil_real,,,,
Conservador,24.3%,19.5%,43.7%,12.5%
Moderado,9.5%,15.1%,20.9%,54.4%
Agresivo,49.2%,35.3%,3.1%,12.4%


Cada perfil tiene un bloque dominante: el conservador tiene el 54 % en riesgo 1, el moderado el 72 % en riesgo 3 y el agresivo el 55 % en riesgo 4 y el 27 % en riesgo 5.

Lo inesperado es la liquidez. El moderado es el menos líquido, con el 54,4 % en instrumentos de liquidez muy baja (fideicomisos, AFAP e inmuebles), contra el 12,5 % del conservador y el 12,4 % del agresivo, que tiene casi la mitad en liquidez alta. La escala de 1 a 5 mide cuánto varía el precio y no qué tan fácil es salir, así que esto no lo ve; por eso en las carteras modelo la liquidez es una restricción aparte.

## Distribución de la cartera: promedio vs. cliente típico

In [89]:
# Distribución del patrimonio: el promedio no describe al cliente típico

patrimonios = comparables["cartera_usd"]
media, mediana = patrimonios.mean(), patrimonios.median()

decil_superior = patrimonios.nlargest(int(len(patrimonios) * 0.1)).sum()

figura = go.Figure(
    go.Histogram(
        x=patrimonios,
        xbins=dict(start=0, size=5000),
        marker=dict(color="#a9cbf0", line=dict(color=SUPERFICIE, width=1)),
        hovertemplate="<b>USD %{x}</b><br>%{y} clientes<extra></extra>",
    )
)
for valor, nombre, color, alto_texto, desplazamiento in [
    (mediana, "Mediana", COLOR_PERFIL["Conservador"], 0.92, 150),
    (media, "Media", COLOR_PERFIL["Moderado"], 0.62, 110),
]:
    figura.add_vline(x=valor, line=dict(color=color, width=3))
    figura.add_annotation(
        x=valor,
        y=alto_texto,
        yref="paper",
        text=f"<b>{nombre}</b>: USD {formato_es(valor)}",
        showarrow=True,
        arrowcolor=color,
        arrowwidth=1.5,
        ax=desplazamiento,
        ay=0,
        xanchor="left",
        bgcolor=SUPERFICIE,
        font=dict(color=TINTA_PRIMARIA),
    )
figura.add_annotation(
    x=0.98,
    y=0.6,
    xref="paper",
    yref="paper",
    xanchor="right",
    align="right",
    showarrow=False,
    text=(
        f"El decil superior concentra<br>el "
        f"<b>{decil_superior / patrimonios.sum():.0%}</b> del patrimonio"
    ),
    font=dict(color=TINTA_PRIMARIA, size=13),
)
figura.update_xaxes(
    title="Cartera por cliente (USD, costo de adquisición)", tickformat=",.0f"
)
figura.update_yaxes(title="Cantidad de clientes")
figura.update_layout(bargap=0.05, showlegend=False)
presentar(
    figura,
    f"La cartera mediana es USD {formato_es(mediana)}, {1 - mediana / media:.0%} por "
    f"debajo de la media (USD {formato_es(media)})",
    "628 clientes con cartera · cada barra es un tramo de USD 5.000",
    interactivo=False,
    alto=440,
)

decilicos = pd.DataFrame(
    {
        "patrimonio_usd": patrimonios.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]),
    }
)
decilicos.index = [f"percentil {int(p * 100)}" for p in decilicos.index]
display(decilicos.style.format("{:,.0f}"))

print(
    f"El 10 % de clientes más grandes concentra el "
    f"{decil_superior / patrimonios.sum():.0%} del patrimonio total."
)
print(f"Clientes por debajo del promedio: " f"{(patrimonios < media).mean():.0%}")

,patrimonio_usd
percentil 10,"2,138"
percentil 25,"5,266"
percentil 50,"11,664"
percentil 75,"23,430"
percentil 90,"42,518"
percentil 95,"63,092"
percentil 99,"126,823"


El 10 % de clientes más grandes concentra el 40% del patrimonio total.
Clientes por debajo del promedio: 69%


La media es USD 19.450 y la mediana USD 11.664: la media está 67 % por encima del cliente típico y el 69 % de los clientes tiene menos que el promedio. El 10 % de los clientes con más plata concentra el 40 % del patrimonio. Por eso las carteras modelo se dimensionan con la mediana y no con el promedio.

# Perfiles y carteras modelo

Lo que no sirve para perfilar: la autodeclaración (kappa 0,25), el test de conocimiento (casi no predice una vez que se conoce la fuente) y las demográficas (ninguna separa más de un tercio de punto). Lo que sí separa es el comportamiento y la situación financiera, así que armo los perfiles con eso.

## Segmentación

Agrupo a los clientes por cómo se comportan y por su situación financiera, no por patrimonio, edad ni lo que dicen ser, como pide la consigna.

Uso siete variables: riesgo efectivo, peso en cripto y concentración de la cartera; retorno esperado y si se informa por redes; si tiene fondo de emergencia; y el tamaño de la cartera, en logaritmo porque tiene cola larga.

Dejé afuera:

- La autodeclaración, también la indirecta: el conteo de inconsistencias internas incluye reglas que usan el perfil declarado.
- Las inconsistencias del formulario consigo mismo: sólo 44 clientes tienen alguna, y una variable que casi siempre vale cero tiende a armar un grupo propio con esos pocos clientes.
- Los meses de gastos cubiertos: valen 0 justo cuando no hay fondo, y con las dos variables esa información pesaba doble.
- La volatilidad, que repite casi lo mismo que el riesgo efectivo.


In [90]:
# Parámetro: inicializaciones de k-medias
ARRANQUES_KMEANS = 20  # se queda con la mejor

In [91]:
# Segmentación conductual con k-medias

# Las inconsistencias del formulario se calcularon sobre la tabla `clientes`
base_segmentacion = clientes[clientes["tiene_cartera"]].copy()
base_segmentacion["retorno_esperado"] = base_segmentacion["retorno_esperado_anual"].map(
    PISO_DEL_RANGO
)
base_segmentacion["usa_redes"] = (
    base_segmentacion["fuente_informacion"] == "Redes sociales"
).astype(float)
base_segmentacion["sin_fondo"] = (
    base_segmentacion["tiene_fondo_emergencia"] == "No"
).astype(float)
base_segmentacion["log_cartera"] = np.log10(
    base_segmentacion["cartera_usd"].clip(lower=1)
)

VARIABLES_SEGMENTACION = [
    "riesgo_efectivo",
    "peso_cripto",
    "concentracion",
    "retorno_esperado",
    "usa_redes",
    "sin_fondo",
    "log_cartera",
]

datos_segmentacion = base_segmentacion[VARIABLES_SEGMENTACION].dropna()
print(
    f"Clientes segmentables: {len(datos_segmentacion)} de {len(base_segmentacion)} con"
    f" cartera "
    f"({len(base_segmentacion) - len(datos_segmentacion)} quedan fuera por falta de "
    f"algún dato)"
)

escaladas = StandardScaler().fit_transform(datos_segmentacion)

comparacion_k = []
for k in range(2, 7):
    modelo_k = KMeans(n_clusters=k, n_init=ARRANQUES_KMEANS, random_state=SEMILLA).fit(
        escaladas
    )
    comparacion_k.append(
        {
            "grupos": k,
            "silueta": silhouette_score(escaladas, modelo_k.labels_),
            "tamanos": sorted(np.bincount(modelo_k.labels_).tolist(), reverse=True),
        }
    )
display(
    pd.DataFrame(comparacion_k).set_index("grupos").style.format({"silueta": "{:.3f}"})
)

Clientes segmentables: 628 de 628 con cartera (0 quedan fuera por falta de algún dato)


,silueta,tamanos
grupos,,
2,0.271,"[432, 196]"
3,0.259,"[412, 201, 15]"
4,0.234,"[226, 220, 167, 15]"
5,0.233,"[183, 172, 155, 103, 15]"
6,0.227,"[160, 152, 131, 92, 77, 16]"


In [92]:
inercias = pd.Series(
    {
        k: KMeans(n_clusters=k, n_init=ARRANQUES_KMEANS, random_state=SEMILLA)
        .fit(escaladas)
        .inertia_
        for k in range(1, 9)
    }
)

figura = go.Figure(
    go.Scatter(
        x=inercias.index,
        y=inercias.values,
        mode="lines+markers",
        line=dict(color=COLOR_PERFIL["Conservador"]),
    )
)
figura.update_layout(
    title="Método del codo",
    xaxis_title="Cantidad de grupos (k)",
    yaxis_title="Inercia",
    width=600,
    height=380,
)
figura.show()

caida = -inercias.diff() / inercias.loc[1] * 100
print("Cuánto baja la inercia con cada grupo extra (% de la inercia con un grupo):")
print(caida.dropna().round(1).to_string())

Cuánto baja la inercia con cada grupo extra (% de la inercia con un grupo):
2   21.80
3   11.30
4   10.50
5    6.90
6    3.90
7    3.60
8    2.70


Con el segundo, tercero y cuarto grupo la inercia baja 21,8, 11,3 y 10,5 puntos; con el quinto, 6,9. El codo es suave y la silueta es baja para todos los k (de 0,23 a 0,27, máxima con dos grupos): los clientes no forman grupos nítidos sino una nube continua. Elegir cuatro es entonces una decisión de negocio: es la menor cantidad de grupos que separa a los que tienen fondo de emergencia de los que no.

Más de la mitad en cripto, se informa por redes, no tiene fondo. Las variables de sí o no, estandarizadas, pesan mucho en la distancia. Lo veo como una ventaja: son señales que la firma puede observar.

Rehaciendo la segmentación sobre 250 remuestreos, el índice de Rand mediano es 0,97, aunque el percentil 10 es 0,36: con un grupo de sólo 15 clientes, a veces la partición se reacomoda. Contra una versión que incluía la autodeclaración y contaba dos veces el fondo de emergencia, el índice de Rand es 0,77.

In [93]:
# Parámetro: cantidad de perfiles, elegida con el codo y la silueta de arriba
CANTIDAD_PERFILES = 4

In [94]:
# Los cuatro segmentos y sus nombres

modelo_kmeans = KMeans(
    n_clusters=CANTIDAD_PERFILES, n_init=ARRANQUES_KMEANS, random_state=SEMILLA
).fit(escaladas)
base_segmentacion.loc[datos_segmentacion.index, "grupo"] = modelo_kmeans.labels_

# Los nombres se asignan según el rasgo que define a cada grupo, no según el número que
# le tocó al cluster. Se ordenan por riesgo para que el mapeo sea estable entre
# corridas
centroides_riesgo = (
    base_segmentacion.groupby("grupo")["riesgo_efectivo"].mean().sort_values()
)
tiene_fondo_grupo = base_segmentacion.groupby("grupo")["sin_fondo"].mean()

print("Grupos ordenados por riesgo efectivo medio:")
display(
    pd.DataFrame(
        {
            "riesgo_medio": centroides_riesgo,
            "sin_fondo_de_emergencia": tiene_fondo_grupo.reindex(
                centroides_riesgo.index
            ),
            "clientes": base_segmentacion.groupby("grupo")
            .size()
            .reindex(centroides_riesgo.index),
        }
    ).style.format({"riesgo_medio": "{:.2f}", "sin_fondo_de_emergencia": "{:.0%}"})
)

# Los dos grupos de menor riesgo se distinguen por el fondo de emergencia, no por el
# riesgo
ordenados = centroides_riesgo.index.tolist()
bajos = sorted(
    ordenados[:2], key=lambda g: tiene_fondo_grupo[g]
)  # primero el que SÍ tiene fondo

NOMBRES_PERFIL = {
    bajos[0]: "El Previsor",
    bajos[1]: "El Desprotegido",
    ordenados[2]: "El Optimista",
    ordenados[3]: "El Apostador",
}
ORDEN_PERFILES = ["El Previsor", "El Desprotegido", "El Optimista", "El Apostador"]

base_segmentacion["perfil_nuevo"] = base_segmentacion["grupo"].map(NOMBRES_PERFIL)
segmentados = base_segmentacion[base_segmentacion["perfil_nuevo"].notna()].copy()

# Control: ¿cuánto se parecen estos perfiles a los de la versión anterior, que usaba
# también la autodeclaración? Se reproduce esa versión y se compara con el índice de
# Rand ajustado (1 = misma partición, 0 = lo que daría el azar)
VARIABLES_VERSION_ANTERIOR = [
    "riesgo_efectivo",
    "volatilidad_cartera",
    "peso_cripto",
    "concentracion",
    "retorno_esperado",
    "usa_redes",
    "meses_gastos_cubiertos",
    "sin_fondo",
    "log_cartera",
    "inconsistencias",
]
anterior = base_segmentacion.assign(
    inconsistencias=clientes["inconsistencias"].reindex(base_segmentacion.index)
)[VARIABLES_VERSION_ANTERIOR].dropna()
comunes = anterior.index.intersection(datos_segmentacion.index)
etiquetas_anteriores = pd.Series(
    KMeans(n_clusters=CANTIDAD_PERFILES, n_init=ARRANQUES_KMEANS, random_state=SEMILLA)
    .fit(StandardScaler().fit_transform(anterior))
    .labels_,
    index=anterior.index,
)
parecido_anterior = adjusted_rand_score(
    etiquetas_anteriores[comunes], base_segmentacion.loc[comunes, "grupo"]
)
print(
    "\nRand ajustado contra la segmentación con autodeclaración: "
    f"{parecido_anterior:.2f}"
)

print("\nAsignación de nombres:")
for grupo, nombre in NOMBRES_PERFIL.items():
    print(f"  grupo {int(grupo)} -> {nombre}")

Grupos ordenados por riesgo efectivo medio:


,riesgo_medio,sin_fondo_de_emergencia,clientes
grupo,,,
2.000000,1.92,0%,220
1.000000,2.40,100%,226
0.000000,3.04,63%,167
3.000000,4.60,80%,15



Rand ajustado contra la segmentación con autodeclaración: 0.77

Asignación de nombres:
  grupo 2 -> El Previsor
  grupo 1 -> El Desprotegido
  grupo 0 -> El Optimista
  grupo 3 -> El Apostador


In [95]:
# ¿Qué tan simples y qué tan estables son los perfiles?

# 1. Una regla de tres señales: ¿reproduce la segmentación?
MAYORIA_EN_CRIPTO = 0.5  # más de la mitad de la cartera
regla_simple = np.select(
    [
        segmentados["peso_cripto"] > MAYORIA_EN_CRIPTO,
        segmentados["usa_redes"] == 1,
        segmentados["sin_fondo"] == 1,
    ],
    ["El Apostador", "El Optimista", "El Desprotegido"],
    default="El Previsor",
)
print(
    "Rand ajustado entre k-medias y la regla «cripto > 50 % · usa redes · sin fondo»: "
    f"{adjusted_rand_score(segmentados['perfil_nuevo'], regla_simple):.2f}"
)

# 2. Estabilidad: se vuelve a segmentar sobre muestras con reposición y se compara con
# la segmentación original en los clientes de cada muestra
generador = np.random.default_rng(SEMILLA)
estabilidad = []
# Un solo hilo: con datos tan chicos, repartir cada ajuste entre hilos cuesta más de lo
# que ahorra y el bucle tarda varias veces más. El resultado es el mismo
with threadpool_limits(1):
    for _ in range(REPETICIONES_BOOTSTRAP // 4):
        muestra = generador.integers(0, len(escaladas), len(escaladas))
        etiquetas = (
            KMeans(
                n_clusters=CANTIDAD_PERFILES,
                n_init=ARRANQUES_KMEANS,
                random_state=SEMILLA,
            )
            .fit(escaladas[muestra])
            .labels_
        )
        estabilidad.append(
            adjusted_rand_score(modelo_kmeans.labels_[muestra], etiquetas)
        )
print(
    f"Estabilidad en {len(estabilidad)} remuestreos: Rand ajustado mediano "
    f"{np.median(estabilidad):.2f}, percentil 10 {np.percentile(estabilidad, 10):.2f}"
)

Rand ajustado entre k-medias y la regla «cripto > 50 % · usa redes · sin fondo»: 0.95


Estabilidad en 250 remuestreos: Rand ajustado mediano 0.97, percentil 10 0.36


## Los perfiles

In [96]:
# Tamaño y patrimonio de cada perfil

tamanos = (
    segmentados.groupby("perfil_nuevo")
    .agg(
        clientes=("cartera_usd", "size"),
        patrimonio_usd=("cartera_usd", "sum"),
        cartera_mediana=("cartera_usd", "median"),
        cartera_p90=("cartera_usd", lambda s: s.quantile(0.90)),
    )
    .reindex(ORDEN_PERFILES)
)
tamanos["pct_clientes"] = tamanos["clientes"] / tamanos["clientes"].sum()
tamanos["pct_patrimonio"] = tamanos["patrimonio_usd"] / tamanos["patrimonio_usd"].sum()

display(
    tamanos[
        [
            "clientes",
            "pct_clientes",
            "patrimonio_usd",
            "pct_patrimonio",
            "cartera_mediana",
            "cartera_p90",
        ]
    ].style.format(
        {
            "pct_clientes": "{:.1%}",
            "pct_patrimonio": "{:.1%}",
            "patrimonio_usd": "{:,.0f}",
            "cartera_mediana": "{:,.0f}",
            "cartera_p90": "{:,.0f}",
        }
    )
)

,clientes,pct_clientes,patrimonio_usd,pct_patrimonio,cartera_mediana,cartera_p90
perfil_nuevo,,,,,,
El Previsor,220,35.0%,"4,564,623",37.4%,"11,183","43,804"
El Desprotegido,226,36.0%,"4,060,058",33.2%,"11,566","37,831"
El Optimista,167,26.6%,"3,404,436",27.9%,"12,749","46,240"
El Apostador,15,2.4%,"185,510",1.5%,"5,775","29,694"


In [97]:
# Qué caracteriza a cada perfil: las variables que entraron al agrupamiento

caracterizacion = (
    segmentados.groupby("perfil_nuevo")
    .agg(
        riesgo=("riesgo_efectivo", "mean"),
        volatilidad=("volatilidad_cartera", "mean"),
        peso_cripto=("peso_cripto", "mean"),
        concentracion=("concentracion", "mean"),
        retorno_esperado=("retorno_esperado", "mean"),
        usa_redes=("usa_redes", "mean"),
        meses_cubiertos=("meses_gastos_cubiertos", "mean"),
        sin_fondo=("sin_fondo", "mean"),
        inconsistencias=("inconsistencias", "mean"),
        retorno_real=("retorno_real_cartera", "mean"),
    )
    .reindex(ORDEN_PERFILES)
)
display(caracterizacion.T.style.format("{:.3f}"))

perfil_nuevo,El Previsor,El Desprotegido,El Optimista,El Apostador
riesgo,1.920,2.404,3.040,4.604
volatilidad,0.037,0.058,0.101,0.551
peso_cripto,0.002,0.006,0.028,0.826
concentracion,0.597,0.598,0.627,0.778
retorno_esperado,0.066,0.094,0.206,0.213
usa_redes,0.023,0.000,0.958,0.867
meses_cubiertos,6.383,0.000,2.212,1.800
sin_fondo,0.000,1.000,0.629,0.800
inconsistencias,0.177,0.425,0.413,0.400
retorno_real,0.026,0.029,0.039,0.184


In [98]:
# Cómo son los perfiles en variables que NO participaron del agrupamiento
#
# Kruskal-Wallis y no ANOVA: compara los grupos por orden, sin suponer normalidad
# (el ingreso tiene cola larga)

VARIABLES_EXTERNAS = [
    "edad",
    "puntaje_test_conocimiento",
    "ingreso_mensual_liquido",
    "anios_experiencia_invirtiendo",
    "cantidad_dependientes",
    "conocimiento_autopercibido",
]

externas = (
    segmentados.groupby("perfil_nuevo")[VARIABLES_EXTERNAS]
    .mean()
    .reindex(ORDEN_PERFILES)
)
display(externas.style.format("{:,.2f}"))
print("Ingreso mensual líquido mediano (UYU), por perfil:")
print(
    segmentados.groupby("perfil_nuevo")["ingreso_mensual_liquido"]
    .median()
    .reindex(ORDEN_PERFILES)
    .map("{:,.0f}".format)
    .to_string()
)

kruskal = pd.DataFrame(
    [
        dict(
            zip(
                ["variable", "H", "p"],
                [
                    variable,
                    *stats.kruskal(
                        *[
                            sub[variable].dropna()
                            for _, sub in segmentados.groupby("perfil_nuevo")
                        ]
                    ),
                ],
            )
        )
        for variable in VARIABLES_EXTERNAS
    ]
).set_index("variable")
kruskal["p_holm"] = ajuste_holm(kruskal["p"])
kruskal["difieren"] = np.where(kruskal["p_holm"] < 0.05, "sí", "no")
print("¿Los perfiles difieren en variables que no se usaron para construirlos?")
display(kruskal.style.format({"H": "{:.2f}", "p": "{:.5f}", "p_holm": "{:.5f}"}))

,edad,puntaje_test_conocimiento,ingreso_mensual_liquido,anios_experiencia_invirtiendo,cantidad_dependientes,conocimiento_autopercibido
perfil_nuevo,,,,,,
El Previsor,50.08,5.07,"102,865.35",6.01,0.97,2.63
El Desprotegido,45.04,4.55,"82,434.83",5.08,1.01,2.49
El Optimista,46.45,3.67,"86,115.89",5.11,0.74,2.62
El Apostador,44.27,3.93,"68,515.38",5.20,0.67,2.53


Ingreso mensual líquido mediano (UYU), por perfil:
perfil_nuevo
El Previsor        90,600
El Desprotegido    70,900
El Optimista       68,100
El Apostador       71,500


¿Los perfiles difieren en variables que no se usaron para construirlos?


,H,p,p_holm,difieren
variable,,,,
edad,23.67,0.00003,0.00015,sí
puntaje_test_conocimiento,48.60,0.00000,0.00000,sí
ingreso_mensual_liquido,13.85,0.00312,0.01246,sí
anios_experiencia_invirtiendo,1.92,0.58910,1.00000,no
cantidad_dependientes,9.00,0.02925,0.08776,no
conocimiento_autopercibido,2.04,0.56427,1.00000,no


In [99]:
# Control clave: los perfiles nuevos no son la autodeclaración disfrazada

cruce_declarado = pd.crosstab(
    segmentados["perfil_nuevo"], segmentados["perfil_autodeclarado"], normalize="index"
).reindex(index=ORDEN_PERFILES, columns=ETIQUETAS_PERFIL)
display(cruce_declarado.style.format("{:.1%}"))

print(
    "Si los perfiles nuevos fueran un reflejo de la autodeclaración, cada fila tendría"
)
print(
    "casi todo su peso en una sola columna. No pasa: los perfiles nuevos cortan la base"
)
print("por otro lado, que es justamente lo que pedía el gerente.")

perfil_autodeclarado,Conservador,Moderado,Agresivo
perfil_nuevo,,,
El Previsor,65.5%,24.5%,10.0%
El Desprotegido,50.0%,31.4%,18.6%
El Optimista,31.7%,31.1%,37.1%
El Apostador,13.3%,13.3%,73.3%


Si los perfiles nuevos fueran un reflejo de la autodeclaración, cada fila tendría
casi todo su peso en una sola columna. No pasa: los perfiles nuevos cortan la base
por otro lado, que es justamente lo que pedía el gerente.


El Previsor: 220 clientes (35 %) y USD 4,56 millones en la firma (37,4 %). Riesgo 1,92, volatilidad 3,7 % y casi nada en cripto. Todos declaran fondo de emergencia, de 6,4 meses en promedio. Es el de más edad (50 años), mejor test (5,1) y mayor ingreso mediano (UYU 90.600).

El Desprotegido: 226 clientes (36 %) y USD 4,06 millones (33,2 %). Algo más de riesgo que el Previsor (2,40, volatilidad 5,8 %) y casi sin cripto, pero ninguno declara fondo de emergencia, y su ingreso mediano es más bajo (UYU 70.900). Sumando a los Optimistas y Apostadores sin fondo, el 55 % de los clientes con cartera invierte sin red.

El Optimista: 167 clientes (26,6 %) y USD 3,40 millones (27,9 %). Riesgo 3,04 y volatilidad 10,1 %. El 96 % se informa por redes y en promedio espera más de 20 % anual. Tiene el peor test (3,7) y el 63 % no tiene fondo de emergencia.

El Apostador: 15 clientes (2,4 %) y USD 0,19 millones (1,5 %). Tiene el 83 % de la cartera en cripto y una volatilidad de 55 %. Es el que menos tiene en la firma (mediana de USD 5.775) y el 80 % no tiene fondo. Son pocos, pero es donde una caída del mercado se convierte en un reclamo.

Los perfiles también difieren en variables que no entraron al modelo: edad, test e ingreso. En experiencia, dependientes y conocimiento autopercibido no difieren. Eso los describe mejor, aunque no los valida del todo: el test está relacionado con la fuente de información, que sí entró al modelo. Y no son la autodeclaración, entre los Previsores un 10 % se declara agresivo, y entre los Optimistas los tres perfiles declarados aparecen en partes casi iguales.

 El Previsor y el Desprotegido invierten parecido y la escala de la firma deja a los dos en conservador. Los separa el fondo de emergencia, que es capacidad de aguantar una pérdida y no preferencia: juntarlos borraría justo lo que cambia la recomendación.

## Cómo se asigna el perfil a un cliente nuevo

Los perfiles salen de la cartera, pero un cliente nuevo llega sin cartera. cREO una regla que vaya de las respuestas del formulario al perfil.

Uso un árbol de decisión de dos niveles, porque da reglas que un asesor puede aplicar sin computadora. Sólo le doy preguntas del formulario, nunca datos de la cartera. Las preguntas con orden (tolerancia, frecuencia de monitoreo, horizonte) pasan a número respetando ese orden; la reacción ante una caída, sin un orden claro de riesgo, pasa a columnas de 0 y 1.

In [100]:
# Parámetros del árbol de asignación
PROFUNDIDAD_ARBOL = 2  # dos preguntas
PROPORCION_PRUEBA = 0.25  # clientes apartados para medir el acierto
MINIMO_CLIENTES_POR_HOJA = 5  # mínimo de clientes detrás de cada regla

In [101]:
# Regla de asignación: del formulario al perfil

ORDEN_TOLERANCIA = {
    "No aceptaría ninguna pérdida": 0,
    "Hasta 5%": 1,
    "Hasta 10%": 2,
    "Hasta 20%": 3,
    "Más de 20%": 4,
}
ORDEN_FRECUENCIA = {
    "Anual o menos": 0,
    "Trimestral": 1,
    "Mensual": 2,
    "Semanal": 3,
    "Diaria": 4,
}
ORDEN_HORIZONTE = {
    "Menos de 1 año": 0,
    "1 a 3 años": 1,
    "3 a 5 años": 2,
    "Más de 5 años": 3,
}
assert set(segmentados["tolerancia_perdida_maxima"].dropna()) <= set(
    ORDEN_TOLERANCIA
), "Hay respuestas de tolerancia sin orden asignado"

respuestas = pd.concat(
    [
        pd.DataFrame(
            {
                "tolerancia_perdida": segmentados["tolerancia_perdida_maxima"].map(
                    ORDEN_TOLERANCIA
                ),
                "frecuencia_monitoreo": segmentados["frecuencia_monitoreo"].map(
                    ORDEN_FRECUENCIA
                ),
                "horizonte": segmentados["horizonte_inversion"].map(ORDEN_HORIZONTE),
                "retorno_esperado": segmentados["retorno_esperado"],
                "usa_redes": segmentados["usa_redes"],
                "meses_gastos_cubiertos": segmentados["meses_gastos_cubiertos"],
            }
        ),
        pd.get_dummies(segmentados["reaccion_caida_20"], prefix="reaccion", dtype=int),
    ],
    axis=1,
).dropna()
perfil_objetivo = segmentados.loc[respuestas.index, "perfil_nuevo"]

x_entrenamiento, x_prueba, y_entrenamiento, y_prueba = train_test_split(
    respuestas,
    perfil_objetivo,
    test_size=PROPORCION_PRUEBA,
    random_state=SEMILLA,
    stratify=perfil_objetivo,
)
arbol = DecisionTreeClassifier(
    max_depth=PROFUNDIDAD_ARBOL,
    min_samples_leaf=MINIMO_CLIENTES_POR_HOJA,
    random_state=SEMILLA,
)
arbol.fit(x_entrenamiento, y_entrenamiento)
prediccion = arbol.predict(x_prueba)

referencia = y_prueba.value_counts(normalize=True).max()
print(
    f"Entrenamiento: {len(x_entrenamiento)} clientes · prueba: {len(x_prueba)} clientes"
)
print(f"Acierto del árbol sobre la prueba: {accuracy_score(y_prueba, prediccion):.1%}")
print(f"Acierto de asignar a todos el perfil más frecuente: {referencia:.1%}")
print(
    "Acierto balanceado (promedio de lo que reconoce de cada perfil): "
    f"{balanced_accuracy_score(y_prueba, prediccion):.1%}"
)

print("\nDe cada perfil, qué parte reconoce el árbol (sensibilidad):")
sensibilidad_perfil = pd.Series(
    recall_score(y_prueba, prediccion, labels=ORDEN_PERFILES, average=None),
    index=ORDEN_PERFILES,
)
print(sensibilidad_perfil.map("{:.0%}".format).to_string())

print("\nMatriz de confusión (filas: perfil real · columnas: asignado):")
display(
    pd.crosstab(
        pd.Series(y_prueba.values, name="real"),
        pd.Series(prediccion, name="asignado"),
    ).reindex(index=ORDEN_PERFILES, columns=ORDEN_PERFILES, fill_value=0)
)

print("Las reglas, tal como las aplicaría un asesor:")
print(export_text(arbol, feature_names=list(respuestas.columns), decimals=2))

Entrenamiento: 461 clientes · prueba: 154 clientes
Acierto del árbol sobre la prueba: 94.8%
Acierto de asignar a todos el perfil más frecuente: 37.0%
Acierto balanceado (promedio de lo que reconoce de cada perfil): 72.7%

De cada perfil, qué parte reconoce el árbol (sensibilidad):
El Previsor         98%
El Desprotegido    100%
El Optimista        93%
El Apostador         0%

Matriz de confusión (filas: perfil real · columnas: asignado):


asignado,El Previsor,El Desprotegido,El Optimista,El Apostador
real,,,,
El Previsor,51,0,1,0
El Desprotegido,0,57,0,0
El Optimista,1,2,38,0
El Apostador,0,0,4,0


Las reglas, tal como las aplicaría un asesor:
|--- usa_redes <= 0.50
|   |--- meses_gastos_cubiertos <= 0.50
|   |   |--- class: El Desprotegido
|   |--- meses_gastos_cubiertos >  0.50
|   |   |--- class: El Previsor
|--- usa_redes >  0.50
|   |--- tolerancia_perdida <= 3.50
|   |   |--- class: El Optimista
|   |--- tolerancia_perdida >  3.50
|   |   |--- class: El Optimista



Con dos preguntas alcanza para casi todos: si no se informa por redes, el fondo de emergencia separa al Previsor del Desprotegido; si se informa por redes, es Optimista. El corte por tolerancia termina en Optimista de los dos lados, así que no cambia nada. La regla acierta el 94,8 % de los clientes de prueba, contra el 37 % de darle a todos el perfil más común.

Ese acierto es alto en parte porque la fuente y el fondo son variables con las que se armaron los perfiles. Y es sobre el total: como el Apostador es tan chico, no reconocer a ninguno casi no lo mueve. Promediando lo que reconoce de cada perfil, el acierto es 72,7 %.

Al Apostador ninguna respuesta lo detecta, porque lo define tener la cartera en cripto. Lo trato como una alerta del control anual de la cartera contra el perfil, no como un perfil de entrada.

## Mínimos de inversión

Antes de armar las carteras comparo los mínimos de inversión, en dólares, con el tamaño real de las carteras.

Considero accesible un instrumento si su mínimo no exige más del 25 % de la cartera típica; es un criterio mío de diversificación. En las carteras modelo el tope por posición es 30 %, porque con montos invertibles de USD 1.800 a 11.200 un tope de 25 % deja afuera instrumentos necesarios. Como referencia, los fondos europeos (Directiva 2009/65/CE, artículo 52) no pueden poner más del 5 % en un mismo emisor, o hasta 10 % con un tope conjunto de 40 % para esas posiciones; para una cartera de USD 11.700 eso es demasiado estricto.

In [102]:
# Mínimos de inversión en dólares y qué porcentaje de cartera exigen

catalogo_modelo = catalogo_real.copy()
catalogo_modelo["minimo_usd"] = catalogo_modelo["inversion_minima"] * catalogo_modelo[
    "moneda"
].map(FACTOR_A_USD)
catalogo_modelo["retorno_por_volatilidad"] = catalogo_modelo[
    "retorno_real"
] / catalogo_modelo["volatilidad_anual"].replace(0, np.nan)

cartera_tipica = segmentados["cartera_usd"].median()
catalogo_modelo["peso_minimo_exigido"] = catalogo_modelo["minimo_usd"] / cartera_tipica

display(
    catalogo_modelo[
        [
            "nombre_instrumento",
            "moneda",
            "minimo_usd",
            "peso_minimo_exigido",
            "nivel_riesgo",
            "liquidez",
            "retorno_real",
            "volatilidad_anual",
            "retorno_por_volatilidad",
        ]
    ]
    .sort_values("minimo_usd")
    .style.format(
        {
            "minimo_usd": "{:,.0f}",
            "peso_minimo_exigido": "{:.1%}",
            "retorno_real": "{:.2%}",
            "volatilidad_anual": "{:.1%}",
            "retorno_por_volatilidad": "{:.2f}",
        }
    )
)

print(f"Cartera mediana de la base: USD {cartera_tipica:,.0f}\n")
inaccesibles = catalogo_modelo[catalogo_modelo["peso_minimo_exigido"] > 0.25]
print(
    "Instrumentos que exigirían más del 25 % de la cartera mediana, o sea que obligan "
    "a "
    "concentrar:"
)
for fila in inaccesibles.sort_values("peso_minimo_exigido").itertuples():
    print(
        f"  {fila.nombre_instrumento:<42s} riesgo {fila.nivel_riesgo} · "
        f"exige el {fila.peso_minimo_exigido:>6.0%} de la cartera"
    )

,nombre_instrumento,moneda,minimo_usd,peso_minimo_exigido,nivel_riesgo,liquidez,retorno_real,volatilidad_anual,retorno_por_volatilidad
0,Plazo fijo en pesos (BROU),UYU,23,0.2%,1,Baja,2.94%,0.5%,5.88
7,Ahorro voluntario en AFAP,UYU,47,0.4%,3,Muy baja,-0.47%,6.1%,-0.08
12,Criptomonedas,USD,100,0.9%,5,Alta,21.52%,65.3%,0.33
13,Dólares en caja de seguridad,USD,100,0.9%,1,Alta,-2.63%,0.0%,nan
4,Fondo de inversión local (AFISA),UYU,467,4.0%,2,Alta,1.61%,4.1%,0.39
9,Acciones individuales del exterior,USD,500,4.3%,5,Alta,7.21%,24.3%,0.30
1,Plazo fijo en dólares,USD,500,4.3%,1,Baja,0.58%,0.4%,1.46
8,ETF de índice global (S&P 500 / MSCI World),USD,"1,000",8.6%,4,Alta,6.52%,15.2%,0.43
2,Letras de Regulación Monetaria (BCU),UYU,"1,168",10.0%,1,Media,3.61%,1.3%,2.77
11,Oro,USD,"1,500",12.9%,4,Media,3.31%,14.1%,0.23


Cartera mediana de la base: USD 11,664

Instrumentos que exigirían más del 25 % de la cartera mediana, o sea que obligan a concentrar:
  Notas del Tesoro en Unidades Indexadas     riesgo 2 · exige el    39% de la cartera
  Obligaciones negociables (BVM)             riesgo 3 · exige el    43% de la cartera
  Fideicomiso financiero                     riesgo 3 · exige el    52% de la cartera
  Inmueble para renta                        riesgo 3 · exige el   686% de la cartera


In [103]:
# Parámetro: umbral para considerar accesible un instrumento
LIMITE_CONCENTRACION = 0.25  # su mínimo no puede exigir más de esta parte

`LIMITE_CONCENTRACION` (25 %): un instrumento es accesible si su mínimo no exige más de un cuarto de la cartera típica. Si exige más, entrar obliga a concentrar. Es un criterio mío de diversificación.

In [104]:
# El hueco: qué niveles de riesgo quedan accesibles con una cartera típica

catalogo_modelo["accesible"] = (
    catalogo_modelo["peso_minimo_exigido"] <= LIMITE_CONCENTRACION
)

cobertura = (
    catalogo_modelo.groupby("nivel_riesgo")
    .agg(instrumentos=("accesible", "size"), accesibles=("accesible", "sum"))
    .assign(
        cuales=lambda d: [
            ", ".join(
                catalogo_modelo.loc[
                    (catalogo_modelo["nivel_riesgo"] == n)
                    & catalogo_modelo["accesible"],
                    "nombre_instrumento",
                ].str[:28]
            )
            for n in d.index
        ]
    )
)
display(cobertura)

riesgo_tres = catalogo_modelo[
    (catalogo_modelo["nivel_riesgo"] == 3) & catalogo_modelo["accesible"]
]
print("Único instrumento de riesgo 3 accesible con una cartera típica:")
display(
    riesgo_tres[
        ["nombre_instrumento", "minimo_usd", "retorno_real", "liquidez"]
    ].style.format({"minimo_usd": "{:,.0f}", "retorno_real": "{:.2%}"})
)

,instrumentos,accesibles,cuales
nivel_riesgo,,,
1,4,4,"Plazo fijo en pesos (BROU), Plazo fijo en dóla..."
2,2,1,Fondo de inversión local (AF
3,4,1,Ahorro voluntario en AFAP
4,2,2,"ETF de índice global (S&P 50, Oro"
5,2,2,"Acciones individuales del ex, Criptomonedas"


Único instrumento de riesgo 3 accesible con una cartera típica:


,nombre_instrumento,minimo_usd,retorno_real,liquidez
7,Ahorro voluntario en AFAP,47,-0.47%,Muy baja


Con una cartera típica de unos USD 11.660, cuatro instrumentos obligan a poner más de un cuarto de la plata en una sola posición: Notas del Tesoro en UI (riesgo 2) piden el 39 %, obligaciones negociables (riesgo 3) el 43 %, el fideicomiso (riesgo 3) el 52 % y el inmueble (riesgo 3) el 686 %. El resto es accesible: plazo fijo en pesos desde USD 23, fondo local desde USD 467, Letras desde USD 1.168, cripto desde USD 100, acciones desde USD 500 y ETF desde USD 1.000.

Eso deja un solo instrumento de riesgo 3 contratable, el ahorro en AFAP, que pierde poder de compra (−0,47 % real) y es muy poco líquido. Por eso armo cada cartera en dos versiones: una que la cartera típica puede contratar y otra ampliada para patrimonios más grandes.


In [105]:
# ¿Los clientes respetaron los mínimos del catálogo en su primera compra?

primeras_compras = (
    movimientos_utiles.sort_values("fecha_operacion")
    .groupby(["id_cliente", "id_instrumento"], as_index=False)
    .first()
    .merge(
        catalogo[
            ["id_instrumento", "nombre_instrumento", "nivel_riesgo", "inversion_minima"]
        ],
        on="id_instrumento",
    )
)
primeras_compras["bajo_el_minimo"] = (
    primeras_compras["monto"] < primeras_compras["inversion_minima"]
)
entradas_bajo_minimo = (
    primeras_compras.groupby(["nombre_instrumento", "nivel_riesgo"])
    .agg(
        primeras_compras=("bajo_el_minimo", "size"),
        bajo_el_minimo=("bajo_el_minimo", "mean"),
    )
    .sort_values("bajo_el_minimo", ascending=False)
)
display(entradas_bajo_minimo.style.format({"bajo_el_minimo": "{:.0%}"}))

,,primeras_compras,bajo_el_minimo
nombre_instrumento,nivel_riesgo,,
Fideicomiso financiero,3,172,84%
Notas del Tesoro en Unidades Indexadas,2,160,82%
Inmueble para renta,3,3,67%
Obligaciones negociables (BVM),3,69,51%
Oro,4,87,29%
ETF de índice global (S&P 500 / MSCI World),4,87,29%
Letras de Regulación Monetaria (BCU),1,63,27%
Acciones individuales del exterior,5,45,18%
Plazo fijo en dólares,1,134,13%


No los respetaron: el 84 % de las primeras compras de fideicomisos, el 82 % de las de Notas en UI y el 51 % de las de obligaciones negociables están por debajo del mínimo del catálogo. O la firma no controla sus mínimos, o el catálogo no refleja los que se aplicaron. No cambia el análisis, porque las carteras modelo los respetan igual.

## Carteras modelo

Cada cartera cumple cinco condiciones: los pesos suman 100 %, ninguna posición pasa de 30 %, cada posición supera el mínimo del instrumento para el monto invertible mediano del perfil, la liquidez es acorde y el riesgo respeta la capacidad de pérdida.

- Liquidez: sin fondo de emergencia, al menos 60 % en liquidez alta o media y como mucho 10 % en liquidez muy baja. Para el Previsor, que tiene fondo, el techo de liquidez muy baja es 30 %.
- Capacidad: cuando lo que el cliente quiere y lo que puede no coinciden, manda lo más bajo. Los perfiles sin fondo no pasan de 10 % de volatilidad, el techo de un moderado en el SRRI.
- Idoneidad: a quien consintió el perfil conservador no se le da una cartera de más de 5 % de volatilidad sin re-perfilarlo.

Son criterios míos, no salen de una norma.

La reserva va antes que la cartera. A cada cliente sin fondo de emergencia le separo en efectivo los tres meses de ingreso que todavía no tiene apartados, y la cartera modelo se aplica a lo que queda. Lo hago por cliente y no por perfil, porque hay Optimistas y Apostadores con fondo. El monto invertible mediano queda en USD 11.183 para el Previsor, 6.765 para el Desprotegido, 11.023 para el Optimista y 1.801 para el Apostador. A 70 clientes (45 Desprotegidos, 20 Optimistas y 5 Apostadores) la reserva se les lleva toda la cartera.

Hago dos versiones por perfil: una base que el monto invertible mediano puede contratar y una ampliada con los instrumentos de mínimo alto, con el patrimonio a partir del cual es posible.

In [106]:
# Parámetros de las carteras modelo. Clave de los diccionarios: ¿tiene fondo?
MAXIMO_POR_POSICION = 0.30  # tope de cada posición
PISO_LIQUIDO = {True: 0.40, False: 0.60}  # mínimo en liquidez alta o media
TECHO_ILIQUIDO = {True: 0.30, False: 0.10}  # máximo en liquidez muy baja
VOLATILIDAD_MAXIMA_CONSERVADOR = CORTES_VOLATILIDAD_PERFIL[1]  # 5 %
VOLATILIDAD_MAXIMA_SIN_FONDO = CORTES_VOLATILIDAD_PERFIL[2]  # 10 %

Son criterios míos, no salen de una norma:

- `MAXIMO_POR_POSICION` (30 %): es algo más amplio que el 25 % del límite de concentración, porque con montos invertibles de USD 1.800 a 11.200 un tope de 25 % deja afuera instrumentos necesarios.
- `PISO_LIQUIDO` (60 % sin fondo, 40 % con fondo) y `TECHO_ILIQUIDO` (10 % sin fondo, 30 % con fondo): quien no tiene fondo de emergencia tiene que poder vender rápido si le surge un gasto, así que necesita más liquidez y casi nada inmovilizado.
- `VOLATILIDAD_MAXIMA_CONSERVADOR` (5 %): es el techo del perfil conservador en el SRRI agrupado. Una cartera más volátil no se le puede dar a quien firmó ese perfil sin re-perfilarlo.
- `VOLATILIDAD_MAXIMA_SIN_FONDO` (10 %): es la capacidad de pérdida. Sin fondo de emergencia la cartera no pasa del techo de un moderado, aunque el cliente quiera más riesgo.

In [107]:
# Definición de las carteras modelo

# Nombres cortos para no repetir el nombre completo del catálogo en cada diccionario
PLAZO_PESOS = "Plazo fijo en pesos (BROU)"
PLAZO_DOLARES = "Plazo fijo en dólares"
LETRAS = "Letras de Regulación Monetaria (BCU)"
NOTAS_UI = "Notas del Tesoro en Unidades Indexadas"
FONDO_LOCAL = "Fondo de inversión local (AFISA)"
OBLIGACIONES = "Obligaciones negociables (BVM)"
FIDEICOMISO = "Fideicomiso financiero"
ETF_GLOBAL = "ETF de índice global (S&P 500 / MSCI World)"
ACCIONES = "Acciones individuales del exterior"
ORO = "Oro"
CRIPTO = "Criptomonedas"
DOLARES_LIQUIDOS = "Dólares en caja de seguridad"

CARTERAS_BASE = {
    # Tiene colchón y más ingreso, pero dos de cada tres consintieron el perfil
    # conservador: la cartera se queda dentro de la volatilidad de un conservador
    "El Previsor": {
        LETRAS: 0.25,
        FONDO_LOCAL: 0.20,
        PLAZO_PESOS: 0.15,
        PLAZO_DOLARES: 0.20,
        ETF_GLOBAL: 0.20,
    },
    # Sin colchón: la reserva se aparta antes y la cartera es líquida y conservadora
    "El Desprotegido": {
        PLAZO_DOLARES: 0.20,
        LETRAS: 0.25,
        FONDO_LOCAL: 0.25,
        PLAZO_PESOS: 0.10,
        ETF_GLOBAL: 0.20,
    },
    # Quiere riesgo pero la mayoría no tiene fondo: manda la capacidad. Renta variable
    # global diversificada y la cripto en el nivel que ya tiene hoy (3 %)
    "El Optimista": {
        LETRAS: 0.20,
        FONDO_LOCAL: 0.25,
        PLAZO_DOLARES: 0.15,
        PLAZO_PESOS: 0.07,
        ETF_GLOBAL: 0.30,
        CRIPTO: 0.03,
    },
    # Después de apartar la reserva le queda poco: sólo entran instrumentos de mínimo
    # bajo. Más de la mitad en dólares, como hoy, y la cripto acotada al 10 %
    # Sin fondo de emergencia necesita 60 % líquido: los dólares líquidos completan ese
    # piso, porque las Letras y el ETF no llegan a su mínimo con este monto
    "El Apostador": {
        FONDO_LOCAL: 0.30,
        DOLARES_LIQUIDOS: 0.20,
        CRIPTO: 0.10,
        PLAZO_DOLARES: 0.30,
        PLAZO_PESOS: 0.10,
    },
}

CARTERAS_AMPLIADAS = {
    "El Previsor": {
        LETRAS: 0.10,
        FONDO_LOCAL: 0.10,
        NOTAS_UI: 0.20,
        OBLIGACIONES: 0.15,
        FIDEICOMISO: 0.10,
        ETF_GLOBAL: 0.20,
        ORO: 0.10,
        PLAZO_PESOS: 0.05,
    },
    "El Desprotegido": {
        PLAZO_DOLARES: 0.15,
        LETRAS: 0.15,
        FONDO_LOCAL: 0.15,
        NOTAS_UI: 0.20,
        OBLIGACIONES: 0.15,
        ETF_GLOBAL: 0.15,
        PLAZO_PESOS: 0.05,
    },
    "El Optimista": {
        LETRAS: 0.15,
        FONDO_LOCAL: 0.10,
        NOTAS_UI: 0.15,
        OBLIGACIONES: 0.15,
        ETF_GLOBAL: 0.20,
        ORO: 0.10,
        CRIPTO: 0.05,
        PLAZO_DOLARES: 0.10,
    },
    "El Apostador": {
        LETRAS: 0.20,
        FONDO_LOCAL: 0.15,
        NOTAS_UI: 0.20,
        ETF_GLOBAL: 0.25,
        PLAZO_DOLARES: 0.15,
        CRIPTO: 0.05,
    },
}

# Techos de iliquidez según si el perfil tiene fondo de emergencia constituido
TIENE_COLCHON = {
    "El Previsor": True,
    "El Desprotegido": False,
    "El Optimista": False,
    "El Apostador": False,
}

# Lo que cada cliente puede invertir: a los perfiles sin fondo se les descuenta la
# reserva de MESES_DE_RESERVA meses de ingreso que todavía no tienen apartada (la que ya
# está en dólares en caja se separó antes). Sin ingreso declarado, se usa la mediana
# del perfil
ingreso_para_reserva = segmentados["ingreso_mensual_liquido"].fillna(
    segmentados.groupby("perfil_nuevo")["ingreso_mensual_liquido"].transform("median")
)
reserva_faltante = (
    MESES_DE_RESERVA * ingreso_para_reserva / TIPO_CAMBIO - segmentados["reserva"]
).clip(lower=0)
# La reserva se descuenta según lo que declaró cada cliente, no según su perfil: hay
# Optimistas y Apostadores con fondo de emergencia
sin_fondo_cliente = segmentados["tiene_fondo_emergencia"] == "No"
segmentados["invertible_usd"] = (
    segmentados["cartera_usd"] - reserva_faltante.where(sin_fondo_cliente, 0)
).clip(lower=0)
sin_nada = segmentados["invertible_usd"] == 0
print(
    f"Clientes a los que la reserva les absorbe toda la cartera: {int(sin_nada.sum())}"
)
print(
    sin_nada.groupby(segmentados["perfil_nuevo"])
    .sum()
    .reindex(ORDEN_PERFILES)
    .to_string()
)

invertible_mediano = (
    segmentados.groupby("perfil_nuevo")["invertible_usd"]
    .median()
    .reindex(ORDEN_PERFILES)
)
print("Cartera e invertible medianos por perfil (USD):")
display(
    pd.DataFrame(
        {
            "cartera_mediana": tamanos["cartera_mediana"],
            "invertible_mediano": invertible_mediano,
        }
    ).style.format("{:,.0f}")
)

Clientes a los que la reserva les absorbe toda la cartera: 70
perfil_nuevo
El Previsor         0
El Desprotegido    45
El Optimista       20
El Apostador        5
Cartera e invertible medianos por perfil (USD):


,cartera_mediana,invertible_mediano
perfil_nuevo,,
El Previsor,"11,183","11,183"
El Desprotegido,"11,566","6,765"
El Optimista,"12,749","11,023"
El Apostador,"5,775","1,801"


In [108]:
# Evaluación y validación de una cartera modelo

datos_instrumento = catalogo_modelo.set_index("nombre_instrumento")


def evaluar_cartera(
    pesos: dict[str, float], cartera_referencia: float, tiene_colchon: bool
) -> dict:
    """Calcula métricas de una cartera modelo y verifica las tres restricciones.

    La volatilidad es el promedio ponderado de las volatilidades individuales, cada
    una en la moneda de su instrumento. Supone correlación perfecta entre activos, así
    que en moneda propia es una cota superior, pero no incluye el riesgo cambiario de
    medir todo en una sola moneda. Calcular la volatilidad verdadera exigiría las
    covarianzas entre instrumentos y con el tipo de cambio, que el catálogo no trae.
    """
    peso = pd.Series(pesos)
    instrumentos = datos_instrumento.loc[peso.index]

    monto = peso * cartera_referencia
    cumple_minimo = monto >= instrumentos["minimo_usd"]
    patrimonio_minimo = float((instrumentos["minimo_usd"] / peso).max())

    volatilidad = float((peso * instrumentos["volatilidad_anual"]).sum())
    muy_baja = float(peso[(instrumentos["liquidez"] == "Muy baja").to_numpy()].sum())
    liquida = float(
        peso[instrumentos["liquidez"].isin(["Alta", "Media"]).to_numpy()].sum()
    )

    return {
        "instrumentos": len(peso),
        "suma_pesos": float(peso.sum()),
        "riesgo": float((peso * instrumentos["nivel_riesgo"]).sum()),
        "volatilidad": volatilidad,
        "retorno_nominal": float(
            (peso * instrumentos["retorno_anual_historico"]).sum()
        ),
        "retorno_real": float((peso * instrumentos["retorno_real"]).sum()),
        "retorno_real_pesos": float((peso * instrumentos["retorno_real_pesos"]).sum()),
        "peso_muy_baja_liquidez": muy_baja,
        "peso_liquido": liquida,
        "patrimonio_minimo_usd": patrimonio_minimo,
        # La tolerancia evita que una cartera que da exactamente en el umbral falle por
        # el error de redondeo de sumar pesos en punto flotante
        "ok_pesos": bool(np.isclose(peso.sum(), 1.0)),
        "ok_minimos": bool(cumple_minimo.all()),
        "ok_iliquidez": bool(muy_baja <= TECHO_ILIQUIDO[tiene_colchon] + 1e-9),
        "ok_liquidez": bool(liquida >= PISO_LIQUIDO[tiene_colchon] - 1e-9),
        "ok_posicion": bool(peso.max() <= MAXIMO_POR_POSICION + 1e-9),
        "ok_capacidad": bool(
            tiene_colchon or volatilidad <= VOLATILIDAD_MAXIMA_SIN_FONDO + 1e-9
        ),
    }


def tabla_carteras(
    definiciones: dict[str, dict], referencias: dict[str, float]
) -> pd.DataFrame:
    """Evalúa un conjunto de carteras y devuelve una fila por perfil."""
    filas = {
        nombre: evaluar_cartera(pesos, referencias[nombre], TIENE_COLCHON[nombre])
        for nombre, pesos in definiciones.items()
    }
    return pd.DataFrame(filas).T.reindex(ORDEN_PERFILES)


medianas_perfil = invertible_mediano.to_dict()
evaluacion_base = tabla_carteras(CARTERAS_BASE, medianas_perfil)

COLUMNAS_METRICAS = [
    "instrumentos",
    "riesgo",
    "volatilidad",
    "retorno_nominal",
    "retorno_real",
    "retorno_real_pesos",
    "peso_muy_baja_liquidez",
    "peso_liquido",
    "patrimonio_minimo_usd",
]
display(
    evaluacion_base[COLUMNAS_METRICAS].style.format(
        {
            "instrumentos": "{:.0f}",
            "riesgo": "{:.2f}",
            "volatilidad": "{:.2%}",
            "retorno_nominal": "{:.2%}",
            "retorno_real": "{:.2%}",
            "retorno_real_pesos": "{:.2%}",
            "peso_muy_baja_liquidez": "{:.0%}",
            "peso_liquido": "{:.0%}",
            "patrimonio_minimo_usd": "{:,.0f}",
        }
    )
)

print("Controles de cada cartera base:")
controles = evaluacion_base[
    [
        "ok_pesos",
        "ok_minimos",
        "ok_posicion",
        "ok_iliquidez",
        "ok_liquidez",
        "ok_capacidad",
    ]
]
display(controles)
assert controles.all().all(), "Alguna cartera base incumple una restricción"
print("Las cuatro carteras base cumplen las cinco restricciones.")

# Idoneidad: ¿cuántos clientes de cada perfil consintieron el perfil conservador y
# recibirían una cartera de más volatilidad que la de un conservador?
volatil_para_conservador = (
    evaluacion_base["volatilidad"] > VOLATILIDAD_MAXIMA_CONSERVADOR + 1e-9
)
conservadores_por_perfil = (
    (segmentados["perfil_autodeclarado"] == "Conservador")
    .groupby(segmentados["perfil_nuevo"])
    .sum()
    .reindex(ORDEN_PERFILES)
)
print(
    "Clientes que consintieron el perfil conservador y necesitarían re-perfilarse "
    "para recibir su cartera modelo:"
)
print(conservadores_por_perfil.where(volatil_para_conservador, 0).to_string())

,instrumentos,riesgo,volatilidad,retorno_nominal,retorno_real,retorno_real_pesos,peso_muy_baja_liquidez,peso_liquido,patrimonio_minimo_usd
El Previsor,5,1.80,4.34%,7.54%,3.09%,3.83%,0%,65%,"5,000"
El Desprotegido,5,1.85,4.52%,7.47%,3.02%,3.76%,0%,70%,"5,000"
El Optimista,6,2.27,7.90%,8.27%,4.02%,4.93%,0%,78%,"5,841"
El Apostador,5,1.70,7.93%,6.45%,2.58%,3.69%,0%,60%,"1,667"


Controles de cada cartera base:


,ok_pesos,ok_minimos,ok_posicion,ok_iliquidez,ok_liquidez,ok_capacidad
El Previsor,True,True,True,True,True,True
El Desprotegido,True,True,True,True,True,True
El Optimista,True,True,True,True,True,True
El Apostador,True,True,True,True,True,True


Las cuatro carteras base cumplen las cinco restricciones.
Clientes que consintieron el perfil conservador y necesitarían re-perfilarse para recibir su cartera modelo:
perfil_nuevo
El Previsor         0
El Desprotegido     0
El Optimista       53
El Apostador        2


In [109]:
# Las carteras ampliadas y el patrimonio desde el que son viables

# Se evalúan sin tope de patrimonio para que el control de mínimos no las rechace: lo
# que interesa de estas carteras es justamente a partir de qué patrimonio se vuelven
# posibles
evaluacion_ampliada = tabla_carteras(
    CARTERAS_AMPLIADAS, {nombre: np.inf for nombre in CARTERAS_AMPLIADAS}
)
display(
    evaluacion_ampliada[COLUMNAS_METRICAS].style.format(
        {
            "instrumentos": "{:.0f}",
            "riesgo": "{:.2f}",
            "volatilidad": "{:.2%}",
            "retorno_nominal": "{:.2%}",
            "retorno_real": "{:.2%}",
            "retorno_real_pesos": "{:.2%}",
            "peso_muy_baja_liquidez": "{:.0%}",
            "peso_liquido": "{:.0%}",
            "patrimonio_minimo_usd": "{:,.0f}",
        }
    )
)

print("Comparación de las dos versiones:")
comparacion_versiones = pd.DataFrame(
    {
        "patrimonio mínimo base": evaluacion_base["patrimonio_minimo_usd"],
        "patrimonio mínimo ampliada": evaluacion_ampliada["patrimonio_minimo_usd"],
        "invertible mediano del perfil": pd.Series(medianas_perfil),
        "retorno real base": evaluacion_base["retorno_real"],
        "retorno real ampliada": evaluacion_ampliada["retorno_real"],
    }
).reindex(ORDEN_PERFILES)
display(
    comparacion_versiones.style.format(
        {
            "patrimonio mínimo base": "{:,.0f}",
            "patrimonio mínimo ampliada": "{:,.0f}",
            "invertible mediano del perfil": "{:,.0f}",
            "retorno real base": "{:.2%}",
            "retorno real ampliada": "{:.2%}",
        }
    )
)

for nombre in ORDEN_PERFILES:
    cartera_perfil = segmentados.loc[
        segmentados["perfil_nuevo"] == nombre, "invertible_usd"
    ]
    umbral_base = evaluacion_base.loc[nombre, "patrimonio_minimo_usd"]
    print(
        f"{nombre:<18s} la versión base necesita USD {umbral_base:>7,.0f} · "
        f"la alcanza el {(cartera_perfil >= umbral_base).mean():>4.0%} de ese perfil"
    )
print()
for nombre in ORDEN_PERFILES:
    umbral = evaluacion_ampliada.loc[nombre, "patrimonio_minimo_usd"]
    alcanzan = (
        segmentados.loc[segmentados["perfil_nuevo"] == nombre, "invertible_usd"]
        >= umbral
    ).mean()
    print(
        f"{nombre:<18s} la variante ampliada necesita USD {umbral:>7,.0f} · "
        f"la alcanza el {alcanzan:>4.0%} de ese perfil"
    )

,instrumentos,riesgo,volatilidad,retorno_nominal,retorno_real,retorno_real_pesos,peso_muy_baja_liquidez,peso_liquido,patrimonio_minimo_usd
El Previsor,8,2.70,7.13%,6.93%,4.27%,5.11%,10%,70%,"61,215"
El Desprotegido,7,2.10,4.56%,6.37%,3.17%,4.01%,0%,65%,"33,333"
El Optimista,8,2.65,9.62%,7.59%,4.49%,5.63%,0%,75%,"33,333"
El Apostador,6,2.30,8.52%,7.61%,4.38%,5.23%,0%,85%,"22,956"


Comparación de las dos versiones:


,patrimonio mínimo base,patrimonio mínimo ampliada,invertible mediano del perfil,retorno real base,retorno real ampliada
El Previsor,"5,000","61,215","11,183",3.09%,4.27%
El Desprotegido,"5,000","33,333","6,765",3.02%,3.17%
El Optimista,"5,841","33,333","11,023",4.02%,4.49%
El Apostador,"1,667","22,956","1,801",2.58%,4.38%


El Previsor        la versión base necesita USD   5,000 · la alcanza el  75% de ese perfil
El Desprotegido    la versión base necesita USD   5,000 · la alcanza el  58% de ese perfil
El Optimista       la versión base necesita USD   5,841 · la alcanza el  64% de ese perfil
El Apostador       la versión base necesita USD   1,667 · la alcanza el  53% de ese perfil

El Previsor        la variante ampliada necesita USD  61,215 · la alcanza el   7% de ese perfil
El Desprotegido    la variante ampliada necesita USD  33,333 · la alcanza el  10% de ese perfil
El Optimista       la variante ampliada necesita USD  33,333 · la alcanza el  15% de ese perfil
El Apostador       la variante ampliada necesita USD  22,956 · la alcanza el  13% de ese perfil


In [110]:
# La asignación instrumento por instrumento de cada cartera base

for nombre in ORDEN_PERFILES:
    pesos = pd.Series(CARTERAS_BASE[nombre], name="peso")
    detalle = datos_instrumento.loc[
        pesos.index,
        [
            "moneda",
            "nivel_riesgo",
            "liquidez",
            "minimo_usd",
            "retorno_real",
            "volatilidad_anual",
        ],
    ].join(pesos)
    detalle["monto_usd"] = detalle["peso"] * medianas_perfil[nombre]
    detalle["cumple_minimo"] = detalle["monto_usd"] >= detalle["minimo_usd"]
    detalle = detalle.sort_values("peso", ascending=False)

    print(
        f"\n{'=' * 78}\n{nombre}  ·  invertible de referencia USD "
        f"{medianas_perfil[nombre]:,.0f}\n{'=' * 78}"
    )
    display(
        detalle[
            [
                "peso",
                "monto_usd",
                "minimo_usd",
                "cumple_minimo",
                "moneda",
                "nivel_riesgo",
                "liquidez",
                "retorno_real",
                "volatilidad_anual",
            ]
        ].style.format(
            {
                "peso": "{:.0%}",
                "monto_usd": "{:,.0f}",
                "minimo_usd": "{:,.0f}",
                "retorno_real": "{:.2%}",
                "volatilidad_anual": "{:.1%}",
            }
        )
    )


El Previsor  ·  invertible de referencia USD 11,183


,peso,monto_usd,minimo_usd,cumple_minimo,moneda,nivel_riesgo,liquidez,retorno_real,volatilidad_anual
Letras de Regulación Monetaria (BCU),25%,"2,796","1,168",True,UYU,1,Media,3.61%,1.3%
Fondo de inversión local (AFISA),20%,"2,237",467,True,UYU,2,Alta,1.61%,4.1%
Plazo fijo en dólares,20%,"2,237",500,True,USD,1,Baja,0.58%,0.4%
ETF de índice global (S&P 500 / MSCI World),20%,"2,237","1,000",True,USD,4,Alta,6.52%,15.2%
Plazo fijo en pesos (BROU),15%,"1,677",23,True,UYU,1,Baja,2.94%,0.5%



El Desprotegido  ·  invertible de referencia USD 6,765


,peso,monto_usd,minimo_usd,cumple_minimo,moneda,nivel_riesgo,liquidez,retorno_real,volatilidad_anual
Letras de Regulación Monetaria (BCU),25%,"1,691","1,168",True,UYU,1,Media,3.61%,1.3%
Fondo de inversión local (AFISA),25%,"1,691",467,True,UYU,2,Alta,1.61%,4.1%
Plazo fijo en dólares,20%,"1,353",500,True,USD,1,Baja,0.58%,0.4%
ETF de índice global (S&P 500 / MSCI World),20%,"1,353","1,000",True,USD,4,Alta,6.52%,15.2%
Plazo fijo en pesos (BROU),10%,677,23,True,UYU,1,Baja,2.94%,0.5%



El Optimista  ·  invertible de referencia USD 11,023


,peso,monto_usd,minimo_usd,cumple_minimo,moneda,nivel_riesgo,liquidez,retorno_real,volatilidad_anual
ETF de índice global (S&P 500 / MSCI World),30%,"3,307","1,000",True,USD,4,Alta,6.52%,15.2%
Fondo de inversión local (AFISA),25%,"2,756",467,True,UYU,2,Alta,1.61%,4.1%
Letras de Regulación Monetaria (BCU),20%,"2,205","1,168",True,UYU,1,Media,3.61%,1.3%
Plazo fijo en dólares,15%,"1,653",500,True,USD,1,Baja,0.58%,0.4%
Plazo fijo en pesos (BROU),7%,772,23,True,UYU,1,Baja,2.94%,0.5%
Criptomonedas,3%,331,100,True,USD,5,Alta,21.52%,65.3%



El Apostador  ·  invertible de referencia USD 1,801


,peso,monto_usd,minimo_usd,cumple_minimo,moneda,nivel_riesgo,liquidez,retorno_real,volatilidad_anual
Fondo de inversión local (AFISA),30%,540,467,True,UYU,2,Alta,1.61%,4.1%
Plazo fijo en dólares,30%,540,500,True,USD,1,Baja,0.58%,0.4%
Dólares en caja de seguridad,20%,360,100,True,USD,1,Alta,-2.63%,0.0%
Criptomonedas,10%,180,100,True,USD,5,Alta,21.52%,65.3%
Plazo fijo en pesos (BROU),10%,180,23,True,UYU,1,Baja,2.94%,0.5%


In [111]:
# Qué cambia para cada perfil: situación actual contra cartera modelo

actual = caracterizacion[["riesgo", "volatilidad", "retorno_real"]]
antes_despues = pd.DataFrame(
    {
        "riesgo actual": actual["riesgo"],
        "riesgo modelo": evaluacion_base["riesgo"],
        "volatilidad actual": actual["volatilidad"],
        "volatilidad modelo": evaluacion_base["volatilidad"],
        "retorno real actual": actual["retorno_real"],
        "retorno real modelo": evaluacion_base["retorno_real"],
    }
).reindex(ORDEN_PERFILES)
# Con mucha volatilidad, lo que la cartera crece año tras año es menor que su retorno
# promedio. Suponiendo retornos lognormales:
# crecimiento = exp(ln(1 + r) - ln(1 + σ² / (1 + r)²) / 2) - 1


def crecimiento_compuesto(retorno: pd.Series, volatilidad: pd.Series) -> pd.Series:
    """Crecimiento anual compuesto esperado de una cartera con retornos lognormales."""
    retorno = retorno.astype(float)
    volatilidad = volatilidad.astype(float)
    return (
        np.exp(np.log1p(retorno) - np.log1p(volatilidad**2 / (1 + retorno) ** 2) / 2)
        - 1
    )


antes_despues["retorno real compuesto actual"] = crecimiento_compuesto(
    antes_despues["retorno real actual"], antes_despues["volatilidad actual"]
)
antes_despues["retorno real compuesto modelo"] = crecimiento_compuesto(
    antes_despues["retorno real modelo"], antes_despues["volatilidad modelo"]
)

# Las dos medidas de riesgo de la cartera modelo: la escala de la firma con los cortes
# elegidos y la volatilidad con la partición SRRI
antes_despues["perfil modelo por escala"] = pd.cut(
    antes_despues["riesgo modelo"], CORTES_PERFIL, labels=ETIQUETAS_PERFIL, right=False
)
antes_despues["perfil modelo por volatilidad"] = pd.cut(
    antes_despues["volatilidad modelo"],
    CORTES_VOLATILIDAD_PERFIL,
    labels=ETIQUETAS_PERFIL,
    right=False,
)

antes_despues["retorno por volatilidad actual"] = (
    antes_despues["retorno real actual"] / antes_despues["volatilidad actual"]
)
antes_despues["retorno por volatilidad modelo"] = (
    antes_despues["retorno real modelo"] / antes_despues["volatilidad modelo"]
)

display(
    antes_despues.style.format(
        {
            "riesgo actual": "{:.2f}",
            "riesgo modelo": "{:.2f}",
            "volatilidad actual": "{:.1%}",
            "volatilidad modelo": "{:.1%}",
            "retorno real actual": "{:.2%}",
            "retorno real modelo": "{:.2%}",
            "retorno por volatilidad actual": "{:.2f}",
            "retorno por volatilidad modelo": "{:.2f}",
            "retorno real compuesto actual": "{:.2%}",
            "retorno real compuesto modelo": "{:.2%}",
        }
    )
)

# Escenario de estrés: cuánto pierde cada perfil si la cripto cae 60 %, hoy y con la
# cartera modelo
CAIDA_CRIPTO = 0.60
peso_cripto_modelo = pd.Series(
    {nombre: pesos.get(CRIPTO, 0.0) for nombre, pesos in CARTERAS_BASE.items()}
)
estres = pd.DataFrame(
    {
        "pierde hoy": caracterizacion["peso_cripto"] * CAIDA_CRIPTO,
        "pierde con la cartera modelo": peso_cripto_modelo * CAIDA_CRIPTO,
    }
).reindex(ORDEN_PERFILES)
print(f"Si la cripto cae {CAIDA_CRIPTO:.0%}, cada perfil pierde:")
display(estres.style.format("{:.1%}"))

,riesgo actual,riesgo modelo,volatilidad actual,volatilidad modelo,retorno real actual,retorno real modelo,retorno real compuesto actual,retorno real compuesto modelo,perfil modelo por escala,perfil modelo por volatilidad,retorno por volatilidad actual,retorno por volatilidad modelo
El Previsor,1.92,1.80,3.7%,4.3%,2.57%,3.09%,2.50%,3.00%,Conservador,Conservador,0.69,0.71
El Desprotegido,2.40,1.85,5.8%,4.5%,2.92%,3.02%,2.76%,2.92%,Conservador,Conservador,0.50,0.67
El Optimista,3.04,2.27,10.1%,7.9%,3.90%,4.02%,3.42%,3.72%,Conservador,Moderado,0.39,0.51
El Apostador,4.60,1.70,55.1%,7.9%,18.40%,2.58%,7.36%,2.27%,Conservador,Moderado,0.33,0.33


Si la cripto cae 60%, cada perfil pierde:


,pierde hoy,pierde con la cartera modelo
El Previsor,0.1%,0.0%
El Desprotegido,0.3%,0.0%
El Optimista,1.7%,1.8%
El Apostador,49.6%,6.0%


Qué propone cada cartera base:

- Previsor: riesgo parecido al actual (de 1,92 a 1,80), con 20 % en renta variable global. Dos de cada tres consintieron el perfil conservador, así que la volatilidad queda en 4,3 %. El retorno real pasa de 2,57 % a 3,09 % (3,83 % en pesos) y el retorno por volatilidad apenas mejora (de 0,69 a 0,71). Mantiene el 60 % en pesos porque gasta en Uruguay.
- Desprotegido: primero el colchón. Después, una cartera más conservadora que la actual: riesgo de 2,40 a 1,85 y volatilidad de 5,8 % a 4,5 %, con casi el mismo retorno real (de 2,92 % a 3,02 %).
- Optimista: el 63 % no tiene fondo, así que manda la capacidad. El riesgo baja de 3,04 a 2,27 y la volatilidad de 10,1 % a 7,9 %, con 30 % en renta variable global y 3 % en cripto, lo mismo que tiene hoy. Aun así el retorno real sube de 3,90 % a 4,02 % (compuesto, de 3,42 % a 3,72 %), porque hoy toma riesgo que no le rinde. 53 Optimistas consintieron el perfil conservador y necesitan re-perfilarse antes.
- Apostador: con USD 1.801 invertibles sólo entran instrumentos de mínimo bajo, y sin fondo necesita 60 % líquido: 30 % fondo local, 30 % plazo fijo en dólares, 20 % dólares líquidos, 10 % cripto y 10 % plazo fijo en pesos. Los dólares líquidos pierden contra la inflación, pero las Letras y el ETF no llegan a su mínimo con este monto. La volatilidad baja de 55 % a 7,9 % y el retorno real también baja (de 18,4 % a 2,6 % en promedio, de 7,4 % a 2,3 % compuesto), pero si la cripto cae 60 % pierde el 6 % de la cartera y no la mitad. Dejo un 10 % en cripto a propósito: si le saco todo, no va a seguir la recomendación. Dos Apostadores consintieron el perfil conservador.

Las dos medidas de riesgo no siempre coinciden: por la escala de la firma las cuatro carteras son conservadoras, pero por volatilidad el Optimista y el Apostador quedan como moderados. La escala promedia niveles, y un 30 % en un ETF con 15 % de volatilidad, o un 10 % en cripto, sube mucho la volatilidad pero poco el promedio. Para la capacidad de pérdida uso la volatilidad, que es la más exigente.

La versión base la puede contratar entre el 53 % y el 75 % de cada perfil; la ampliada, entre el 7 % y el 15 %.

## Radar de los perfiles nuevos

In [112]:
# Radar de los perfiles nuevos, reutilizando las funciones del primer radar

COLOR_PERFIL_NUEVO = {
    "El Previsor": "#2a78d6",  # azul
    "El Desprotegido": "#eb6834",  # naranja
    "El Optimista": "#1baf7a",  # aqua
    "El Apostador": "#eda100",  # amarillo
}

promedios_nuevos, crudos_nuevos = datos_radar(
    segmentados, "perfil_nuevo", ORDEN_PERFILES
)

figura_perfiles = radar_plotly(
    promedios_nuevos,
    COLOR_PERFIL_NUEVO,
    titulo="Previsor y Desprotegido invierten parecido: los separa el fondo de "
    "emergencia",
    subtitulo=f"{len(segmentados)} clientes segmentados · pasá el mouse "
    "por "
    "la red para ver la ficha · datos al 30/06/2026",
    crudos=crudos_nuevos,
    conteos=segmentados["perfil_nuevo"].astype(str).value_counts(),
)
figura_perfiles.show()

print("Valores sin normalizar:")
display(crudos_nuevos.T.style.format("{:,.3f}"))

Valores sin normalizar:


perfil_nuevo,El Previsor,El Desprotegido,El Optimista,El Apostador
Riesgo de la cartera,1.920,2.404,3.040,4.604
Retorno esperado,0.066,0.094,0.206,0.213
Usa redes sociales,0.023,0.000,0.958,0.867
Peso en cripto,0.002,0.006,0.028,0.826
Cartera en la firma,"20,748.286","17,964.857","20,385.845","12,367.357"
Conocimiento (test),5.067,4.547,3.667,3.933
Cobertura de emergencia,6.383,0.000,2.212,1.800


Los cuatro perfiles se separan en riesgo, cripto, redes y cobertura de emergencia, y se superponen en lo que tienen invertido en la firma (el Apostador incluso tiene menos). El eje clave es la cobertura de emergencia: el Previsor y el Desprotegido invierten parecido, pero uno cubre 6,4 meses de gastos y el otro ninguno. Como en el primer radar, los ejes no se comparan entre sí porque cada uno tiene su propia escala.

## Rediseño del formulario

Para cada pregunta mido cuánto cambia el riesgo efectivo entre sus respuestas: si gente que responde distinto invierte igual, la pregunta no sirve para perfilar. Para saber si la diferencia puede ser azar uso un ANOVA de un factor, que compara el riesgo medio entre los grupos de respuesta, con corrección de Holm. Sólo cuento respuestas con al menos 15 clientes.

Restricción: la normativa del BCU obliga a considerar la edad, el horizonte, los ingresos, la tolerancia al riesgo, el conocimiento, la rentabilidad buscada y la experiencia. Esas preguntas no se pueden sacar aunque no predigan; sí se puede dejar de usarlas para decidir la cartera.

In [113]:
# Parámetro: mínimo de clientes para comparar una respuesta
MINIMO_CLIENTES_POR_GRUPO = 15

`MINIMO_CLIENTES_POR_GRUPO` (15): con menos clientes, el riesgo medio de una respuesta depende de uno o dos casos. Sin este filtro el departamento parecía separar 1,63 puntos de riesgo; con él separa 0,18.

In [114]:
# Poder de discriminación de cada pregunta del formulario actual


def poder_discriminante(columna: str, tabla: pd.DataFrame) -> dict:
    """Mide cuánto separa una pregunta el riesgo efectivo entre sus respuestas.

    Devuelve el rango entre la respuesta de mayor y la de menor riesgo medio, cuáles
    son, y el valor p de un ANOVA de un factor. El rango dice cuánto importa; el valor
    p,
    si es distinguible del azar. Las respuestas con menos de MINIMO_CLIENTES_POR_GRUPO
    clientes quedan afuera de las dos cosas: su media depende de muy pocas personas.
    """
    datos = tabla[[columna, "riesgo_efectivo"]].dropna()
    resumen = datos.groupby(columna)["riesgo_efectivo"].agg(["mean", "size"])
    resumen = resumen[resumen["size"] >= MINIMO_CLIENTES_POR_GRUPO]
    if len(resumen) < 2:
        return {}
    grupos = [
        datos.loc[datos[columna] == respuesta, "riesgo_efectivo"].values
        for respuesta in resumen.index
    ]
    f, p = stats.f_oneway(*grupos)
    return {
        "pregunta": columna,
        "respuestas": len(resumen),
        "rango_riesgo": float(resumen["mean"].max() - resumen["mean"].min()),
        "mas_riesgosa": str(resumen["mean"].idxmax()),
        "menos_riesgosa": str(resumen["mean"].idxmin()),
        "F": float(f),
        "p": float(p),
    }


PREGUNTAS_FORMULARIO = [
    "fuente_informacion",
    "retorno_esperado_anual",
    "tiene_fondo_emergencia",
    "reaccion_caida_20",
    "tolerancia_perdida_maxima",
    "horizonte_inversion",
    "frecuencia_monitoreo",
    "objetivo_principal",
    "perfil_autodeclarado",
    "nivel_educativo",
    "ocupacion",
    "sexo",
    "tiene_deudas",
    "departamento",
]

evaluacion_preguntas = (
    pd.DataFrame(
        [r for c in PREGUNTAS_FORMULARIO if (r := poder_discriminante(c, segmentados))]
    )
    .set_index("pregunta")
    .sort_values("rango_riesgo", ascending=False)
)
evaluacion_preguntas["p_holm"] = ajuste_holm(evaluacion_preguntas["p"])
evaluacion_preguntas["significativa"] = evaluacion_preguntas["p_holm"] < 0.05

display(
    evaluacion_preguntas[
        ["respuestas", "rango_riesgo", "mas_riesgosa", "menos_riesgosa", "p", "p_holm"]
    ].style.format({"rango_riesgo": "{:.2f}", "p": "{:.4f}", "p_holm": "{:.4f}"})
)

orden = evaluacion_preguntas.sort_values("rango_riesgo")


def lectura_p(p: float) -> str:
    """El valor p en palabras."""
    if p < 0.001:
        return "Significativa (ANOVA, p < 0,001)"
    if p < 0.05:
        return f"Significativa (ANOVA, p = {formato_es(p, 3)})"
    return f"No significativa (ANOVA, p = {formato_es(p, 2)})"


figura = go.Figure(
    go.Bar(
        x=orden["rango_riesgo"],
        y=[nombre_legible(p) for p in orden.index],
        orientation="h",
        marker_color=np.where(
            orden["significativa"], COLOR_PERFIL["Conservador"], GRIS_NEUTRO
        ),
        text=[formato_es(v, 2) for v in orden["rango_riesgo"]],
        textposition="outside",
        hovertext=[
            f"<b>{nombre_legible(p)}</b><br>"
            f"Entre la respuesta más y la menos riesgosa hay "
            f"{formato_es(fila.rango_riesgo, 2)} puntos de riesgo<br>"
            f"Más riesgosa: {fila.mas_riesgosa}<br>"
            f"Menos riesgosa: {fila.menos_riesgosa}<br>"
            f"<i>{lectura_p(fila.p_holm)}, corregido por comparaciones múltiples</i>"
            for p, fila in orden.iterrows()
        ],
        hovertemplate="%{hovertext}<extra></extra>",
    )
)
figura.update_xaxes(
    title="Rango de riesgo efectivo entre respuestas (escala 1 a 5)",
    range=[0, orden["rango_riesgo"].max() * 1.12],
)
figura.update_yaxes(showgrid=False)
presentar(
    figura,
    "Las preguntas de comportamiento discriminan más el riesgo que las demográficas",
    f"{len(segmentados)} clientes segmentados · respuestas con al menos "
    f"{MINIMO_CLIENTES_POR_GRUPO} clientes · en gris, no significativas (ANOVA con "
    f"corrección de Holm) "
    "· pasá el mouse para ver qué respuesta es la más riesgosa",
    alto=540,
    ancho=960,
)

,respuestas,rango_riesgo,mas_riesgosa,menos_riesgosa,p,p_holm
pregunta,,,,,,
tolerancia_perdida_maxima,5,1.90,Más de 20%,No aceptaría ninguna pérdida,0.0000,0.0000
frecuencia_monitoreo,5,1.61,Diaria,Anual o menos,0.0000,0.0000
reaccion_caida_20,5,1.36,Vendería todo,Compraría más,0.0000,0.0000
fuente_informacion,6,1.07,Redes sociales,Asesor financiero,0.0000,0.0000
retorno_esperado_anual,5,1.04,Más de 30%,Menos de 5%,0.0000,0.0000
perfil_autodeclarado,3,0.89,Agresivo,Conservador,0.0000,0.0000
tiene_fondo_emergencia,2,0.54,No,Sí,0.0000,0.0000
objetivo_principal,7,0.33,Crecimiento del capital,Generar una renta mensual,0.2586,1.0000
nivel_educativo,4,0.32,Secundaria completa,Terciaria completa,0.0268,0.1611


Las cinco preguntas que más separan el riesgo son de comportamiento y expectativas: tolerancia a la pérdida máxima (1,90 puntos), frecuencia de monitoreo (1,61), reacción ante una caída (1,36), fuente de información (1,07) y retorno esperado (1,04). Entre las demográficas sólo el sexo sigue siendo significativo después de la corrección, y separa 0,27 puntos; ninguna pasa de 0,32. El horizonte (0,28) no es significativo.

La reacción ante una caída funciona al revés de lo esperable: los que venderían todo tienen las carteras más riesgosas y los que comprarían más, las menos riesgosas. No mide cuánto riesgo quiere el cliente; delata a quien tiene más riesgo del que aguantaría. Con la frecuencia de monitoreo pasa algo parecido: quien tiene una cartera volátil la mira más seguido. Las dos pueden ser consecuencia del riesgo que ya tiene, así que van como preguntas de control y no para decidir la cartera.

In [115]:
# Las numéricas, por correlación de Spearman con el riesgo efectivo (con Holm)

NUMERICAS_FORMULARIO = [
    "meses_gastos_cubiertos",
    "puntaje_test_conocimiento",
    "conocimiento_autopercibido",
    "anios_experiencia_invirtiendo",
    "edad",
    "ingreso_mensual_liquido",
    "patrimonio_invertible_usd",
    "cantidad_dependientes",
]

filas = []
for variable in NUMERICAS_FORMULARIO:
    datos = segmentados[[variable, "riesgo_efectivo"]].dropna()
    r, p = stats.spearmanr(datos[variable], datos["riesgo_efectivo"])
    filas.append(
        {
            "pregunta": variable,
            "n": len(datos),
            "r": r,
            "p": p,
            "significativa": p < 0.05,
        }
    )
numericas = pd.DataFrame(filas).set_index("pregunta")
numericas["p_holm"] = ajuste_holm(numericas["p"])
numericas["significativa"] = numericas["p_holm"] < 0.05
display(
    numericas.reindex(
        numericas["r"].abs().sort_values(ascending=False).index
    ).style.format({"r": "{:+.3f}", "p": "{:.5f}", "p_holm": "{:.5f}"})
)

print(
    "Recordatorio: el puntaje del test pierde la mayor parte de su efecto y deja de"
)
print(
    "ser significativo al controlar por fuente de información y expectativa de "
    "retorno. Su"
)
print("correlación cruda existe, pero no es suya.")

,n,r,p,significativa,p_holm
pregunta,,,,,
meses_gastos_cubiertos,615,-0.242,0.00000,True,0.00000
puntaje_test_conocimiento,610,-0.216,0.00000,True,0.00000
edad,606,-0.089,0.02877,False,0.17261
ingreso_mensual_liquido,567,-0.067,0.11059,False,0.55296
conocimiento_autopercibido,628,-0.048,0.23236,False,0.92945
cantidad_dependientes,628,-0.045,0.26008,False,0.92945
anios_experiencia_invirtiendo,628,+0.031,0.43632,False,0.92945
patrimonio_invertible_usd,628,+0.004,0.92778,False,0.92945


Recordatorio: el puntaje del test pierde la mayor parte de su efecto y deja de
ser significativo al controlar por fuente de información y expectativa de retorno. Su
correlación cruda existe, pero no es suya.


In [116]:
# Las siete prioridades declaradas: ¿predicen lo que el cliente efectivamente compra?
# Spearman, porque la prioridad es un orden (1 a 7) y no una cantidad

# Cada columna de prioridad se compara contra el peso real del instrumento
# correspondiente en la cartera del cliente. Si la preferencia declarada valiera, el
# instrumento preferido (prioridad 1) debería pesar más que el menos preferido
# (prioridad 7)
PRIORIDAD_A_CATEGORIA = {
    "prioridad_plazo_fijo": ["Renta fija local"],
    "prioridad_letras_bcu": ["Renta fija local"],
    "prioridad_fondos_locales": ["Fondos"],
    "prioridad_obligaciones": ["Renta fija corporativa"],
    "prioridad_etf_exterior": ["Renta variable"],
    "prioridad_inmuebles": ["Inmobiliario"],
    "prioridad_criptomonedas": ["Cripto"],
}

peso_por_categoria = (
    posiciones_con_perfil.groupby(["id_cliente", "categoria"])["monto_cartera"]
    .sum()
    .unstack(fill_value=0)
)
peso_por_categoria = peso_por_categoria.div(peso_por_categoria.sum(axis=1), axis=0)

filas = []
for columna, categorias in PRIORIDAD_A_CATEGORIA.items():
    presentes = [c for c in categorias if c in peso_por_categoria.columns]
    if not presentes:
        continue
    peso_real = peso_por_categoria[presentes].sum(axis=1)
    datos = pd.DataFrame(
        {
            "prioridad": segmentados[columna],
            "peso_real": peso_real.reindex(segmentados.index),
        }
    ).dropna()
    # Se espera correlación NEGATIVA: prioridad 1 (preferido) con peso alto
    r, p = stats.spearmanr(datos["prioridad"], datos["peso_real"])
    filas.append(
        {
            "pregunta": columna,
            "n": len(datos),
            "r_con_peso_real": r,
            "p": p,
            "predice": "sí" if (p < 0.05 and r < 0) else "no",
        }
    )

prioridades = pd.DataFrame(filas).set_index("pregunta")
prioridades["p_holm"] = ajuste_holm(prioridades["p"])
prioridades["predice"] = np.where(
    (prioridades["p_holm"] < 0.05) & (prioridades["r_con_peso_real"] < 0), "sí", "no"
)
display(
    prioridades.style.format(
        {"r_con_peso_real": "{:+.3f}", "p": "{:.5f}", "p_holm": "{:.5f}"}
    )
)

print(
    "Se esperaría una correlación negativa: si el cliente puso un instrumento como su"
)
print("preferido (prioridad 1), ese instrumento debería pesar más en su cartera.")

,n,r_con_peso_real,p,predice,p_holm
pregunta,,,,,
prioridad_plazo_fijo,628,-0.105,0.00849,no,0.05941
prioridad_letras_bcu,628,-0.078,0.05091,no,0.30548
prioridad_fondos_locales,628,+0.044,0.26675,no,0.59499
prioridad_obligaciones,628,-0.041,0.30910,no,0.59499
prioridad_etf_exterior,628,-0.062,0.11900,no,0.59499
prioridad_inmuebles,628,+0.059,0.13689,no,0.59499
prioridad_criptomonedas,628,-0.053,0.18435,no,0.59499


Se esperaría una correlación negativa: si el cliente puso un instrumento como su
preferido (prioridad 1), ese instrumento debería pesar más en su cartera.


### Propuesta

El formulario se ordena en tres bloques, como pide la práctica internacional: la autoridad británica pide separar cuánto riesgo quiere asumir el cliente de cuánto puede perder (FSA, guía FG11/05, de 2011); la europea pide no confiar en lo que el cliente dice de sí mismo y verificar que sus respuestas sean coherentes (ESMA, directrices de idoneidad de MiFID II, ESMA35-43-3172, directriz 4); y en EE. UU. la regla 2111 de FINRA exige que cada recomendación sea adecuada al perfil del cliente.

1. Disposición al riesgo (cuánto quiere arriesgar): la decide la tolerancia a la pérdida máxima, la que más separa el riesgo real (1,90 puntos). Es una pregunta de escenario, como las de la escala de Grable y Lytton (1999); conviene sumarle una o dos del mismo tipo.
2. Capacidad de pérdida (cuánto puede perder sin que le cambie la vida): fondo de emergencia, meses que cubre e ingreso. El fondo es lo único que separa al Previsor del Desprotegido y, junto con la fuente, asigna el perfil a un cliente nuevo.
3. Expectativas e información: retorno esperado y fuente de información (1,04 y 1,07 puntos). La fuente es una de las dos preguntas de la regla de asignación, y el retorno esperado detecta a los 113 que esperan algo que el catálogo no da.

Cuando la disposición supera a la capacidad, manda la capacidad, como en las carteras modelo.

Preguntas de control, que no deciden la cartera pero marcan incoherencias, como pide ESMA: la reacción ante una caída del 20 % (separa 1,36 puntos, pero al revés), la frecuencia de monitoreo (1,61) y preguntar la misma idea dos veces en partes distintas del formulario, por ejemplo la pérdida máxima y el retorno esperado: el 6,9 % de los clientes tiene respuestas que se contradicen entre sí.

Se mantienen por norma, pero dejan de decidir la cartera: la autodeclaración, porque el cliente tiene que consentir un perfil, aunque ESMA pide no apoyarse en la autoevaluación y acá tiene un kappa de 0,25; el horizonte, que separa sólo 0,28 puntos y no es significativo; el test de diez preguntas, que reduciría a dos o tres porque casi no predice una vez que se conoce la fuente; y las demográficas, que no separan más de 0,32 puntos.

Se sacan las siete preguntas de preferencia entre instrumentos: corregido por la cantidad de pruebas, ninguna anticipa lo que el cliente compra.

Cambio de proceso: comparar una vez por año la cartera real con el perfil consentido, la actualización periódica que piden las directrices de ESMA. Ese control detectaría a los 22 críticos y a los Apostadores, que ninguna pregunta identifica.

Fuentes: [ESMA35-43-3172](https://www.esma.europa.eu/press-news/esma-news/esma-publishes-final-guidelines-mifid-ii-suitability-requirements), [consulta previa a la guía FG11/05 (GC11/01)](https://fca.org.uk/publication/guidance-consultation/gc11_01.pdf), [FINRA, regla 2111](https://www.finra.org/rules-guidance/rulebooks/finra-rules/2111) y [Grable y Lytton (1999)](https://openjournals.libs.uga.edu/fsr/article/view/3240).

# Conclusiones

- El 50,2 % de los clientes con cartera tiene un perfil real distinto del declarado. El kappa ponderado es 0,25 (intervalo del 95 %: 0,19 a 0,31); en 120 combinaciones de cortes el desajuste nunca baja del 45 % y el kappa nunca llega a 0,40.
- 138 clientes (22 %) están sobreexpuestos; 22 se declararon conservadores y tienen cartera agresiva, con USD 435.579 en juego (listado en la carpeta de salidas). Los sobreexpuestos se informan más por redes y esperan más retorno.
- Los que no declaran patrimonio tienen carteras 76 % más grandes en mediana: el faltante no es aleatorio.
- Los 113 que esperan más de 30 % anual obtienen 11,3 % nominal en pesos y 4,7 % real.
- Los USD 1.056.554 en dólares en caja pierden unos USD 27.800 por año contra la inflación de EE. UU.
- Lo que predice el riesgo es la fuente de información y el retorno esperado; el efecto del conocimiento pierde el 71 % al controlar por ellos. Ninguna demográfica separa más de 0,32 puntos.
- Cuatro perfiles: Previsor (220), Desprotegido (226), Optimista (167) y Apostador (15). El 55 % de los clientes con cartera invierte sin fondo de emergencia.
- La fuente de información y los meses de gastos cubiertos asignan el perfil a un cliente nuevo con 94,8 % de acierto (72,7 % balanceado); al Apostador no lo reconocen.
- Las carteras modelo bajan la volatilidad del Desprotegido, el Optimista y el Apostador y cumplen mínimos, liquidez y capacidad; 53 Optimistas y 2 Apostadores necesitan re-perfilarse.
- Con una cartera típica de USD 11.660, el único riesgo 3 contratable es el ahorro en AFAP, que pierde poder de compra. La mayoría de las primeras compras de Notas en UI y fideicomisos entraron por debajo del mínimo del catálogo.

Los supuestos que pueden mover estos números están al principio del notebook.